# M8 — Integration and Deployment
## Road Surface Damage Detection

**Contributor:** Ghassan Saad AlGhaidani — 225495

## Objective
Build and evaluate a reproducible pipeline combining the trained
M4 road-damage detector and M5 pothole segmentation model.

The service will return damage classes, bounding boxes, pothole
area estimates, and an annotated image.

## Workflow
1. Configure local paths and inspect the Python environment.
2. Verify model files, fingerprints, and class mappings.
3. Run each model independently on development examples.
4. Integrate detection and pothole segmentation.
5. Implement an HTTP inference service.
6. Export to ONNX and check agreement with the original models.
7. Benchmark latency, throughput, and memory.
8. Document failures, acceptance results, and execution instructions.

## Evaluation Protocol
- Use validation data for development and threshold selection.
- Define acceptance criteria before final benchmarking.
- Keep final test data separate from development decisions.
- Record model versions, settings, and benchmark conditions.

## Scope and Limitations
M5 estimates pothole mask area, not physical depth.
Area measurements must state whether they refer to a crop or
the full image. Crack severity is not estimated by M5.

Integration quality and deployment performance remain unverified
until the corresponding tests are completed.

## Execution and Evidence Scope

This notebook records the development experiment
`development_20261107T084317_589129Z`.

Reported numerical results and historical discussions describe that
recorded experiment. Rerunning inference or benchmarks may produce
different measurements and new timestamped artifact directories.
Existing narrative results must be reviewed before being attributed
to a new run.

The final schema 1.2 service was verified in a fresh Python process
and through HTTP after extraction from its delivery archive.
Use SUBMISSION_INDEX.md to locate the selected package and evidence.

A complete Restart Kernel and Run All execution has not been verified.
Setup cells install dependencies, and service cells manage separate
processes; execution therefore requires staged handling.

For the demonstrated service, follow the packaged README.
Clean-environment dependency installation remains unverified.

------------------
## 1. Configure Project Paths

Define the local project root and explicitly select the M4 and M5
checkpoints. Update PROJECT_ROOT when running on another computer.

Check that the required files exist before installing dependencies
or loading models. This cell does not modify any files.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\PC\Downloads\project\project\phase2"
)

ASSETS_DIR = PROJECT_ROOT / "Drive_assests"
M8_DIR = PROJECT_ROOT / "M8"
NOTEBOOK_PATH = M8_DIR / "M8.ipynb"

M4_DIR = ASSETS_DIR / "M4_Combined"
M4_CHECKPOINT = M4_DIR / "runs" / "yolo11n_baseline" / "weights" / "best.pt"

M5_DIR = (
    ASSETS_DIR
    / "M5_Segmentation-20261106T135443Z-1-001"
    / "M5_Segmentation"
)
M5_EXPORT_DIR = M5_DIR / "exports" / "m5_yolo11n_seg_v1_with_lab"
M5_CHECKPOINT = M5_EXPORT_DIR / "best.pt"
M5_CONFIG = M5_EXPORT_DIR / "inference_config.json"

VAL_IMAGES = M4_DIR / "data" / "yolo_dataset" / "images" / "val"
VAL_LABELS = M4_DIR / "data" / "yolo_dataset" / "labels" / "val"

required_files = {
    "M8 notebook": NOTEBOOK_PATH,
    "M4 checkpoint": M4_CHECKPOINT,
    "M5 checkpoint": M5_CHECKPOINT,
    "M5 inference configuration": M5_CONFIG,
}

required_directories = {
    "Validation images": VAL_IMAGES,
    "Validation labels": VAL_LABELS,
}

missing = []

for name, path in required_files.items():
    exists = path.is_file()
    print(f"{'FOUND' if exists else 'MISSING'}: {name}")
    print(f"  {path}")
    if not exists:
        missing.append(name)

for name, path in required_directories.items():
    exists = path.is_dir()
    print(f"{'FOUND' if exists else 'MISSING'}: {name}")
    print(f"  {path}")
    if not exists:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "Required paths are missing: " + ", ".join(missing)
    )

print("\nPASS: Project paths verified.")
print("M4 uses the merged M4_Combined directory.")
print("No models loaded. No files modified.")

## 2. Inspect the Python Environment

Record the Python interpreter, operating system, and installed package
versions. Missing packages are reported without installing anything.

Package availability does not yet confirm successful imports or
compatibility with the saved models.

In [ ]:
import sys
import platform
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Operating system:", platform.platform())
print("Working directory:", Path.cwd())

packages = [
    "ipykernel",
    "numpy",
    "Pillow",
    "opencv-python",
    "torch",
    "torchvision",
    "ultralytics",
    "fastapi",
    "uvicorn",
    "python-multipart",
    "onnx",
    "onnxruntime",
    "onnxruntime-gpu",
    "psutil",
]

environment_versions = {}

print("\nPackage versions:")
for package in packages:
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        installed_version = None

    environment_versions[package] = installed_version
    print(f"{package}: {installed_version or 'NOT INSTALLED'}")

print("\nEnvironment inventory complete. No packages were installed.")
print("ONNX Runtime CPU and GPU packages are alternatives.")

## 3. Check Dependency Installation Compatibility

Ask pip to resolve the required packages without installing them.
Use the Ultralytics version recorded in the M5 experiment.

This checks dependency resolution only; model loading and ONNX
export will be tested separately.

In [ ]:
import subprocess

required_packages = [
    "ultralytics==8.4.197",
    "fastapi",
    "uvicorn",
    "python-multipart",
    "onnx",
    "onnxruntime",
    "psutil",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--dry-run",
    *required_packages,
]

check = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print(check.stdout)

if check.returncode != 0:
    print("CHECK FAILED: Review the resolver output before installation.")
else:
    print("PASS: pip resolved the requested packages.")
    print("No packages were installed. Runtime compatibility is not yet verified.")

## 4. Install Runtime Dependencies

Install the direct dependencies using the versions resolved in the
compatibility check and the active notebook's Python interpreter.

Run this setup cell once. Successful installation does not yet
confirm model loading or ONNX export compatibility.

In [ ]:
import subprocess
import sys

runtime_packages = [
    "ultralytics==8.4.197",
    "opencv-python==5.0.0.93",
    "fastapi==0.162.0",
    "uvicorn==0.54.0",
    "python-multipart==0.0.33",
    "onnx==1.23.2",
    "onnxruntime==1.31.1",
    "psutil==7.2.2",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        *runtime_packages,
    ],
    check=True,
)

print("PASS: Dependency installation completed.")
print("Save the notebook, then restart the kernel.")

## 5. Verify Imports and Available Hardware

Import the runtime dependencies and report the available PyTorch
device and ONNX Runtime providers.

This checks library imports only. Model loading and inference
will be verified in subsequent steps.

In [ ]:
import sys
import cv2
import numpy as np
import torch
import torchvision
import ultralytics
import fastapi
import uvicorn
import python_multipart
import onnx
import onnxruntime as ort
import psutil

from ultralytics import YOLO

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)
print("ONNX:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)

print("\nPyTorch device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("ONNX Runtime providers:", ort.get_available_providers())
print(
    "System RAM:",
    round(psutil.virtual_memory().total / (1024 ** 3), 1),
    "GB",
)

print("\nPASS: All imports succeeded.")
print("Model loading and inference have not been tested yet.")

## 6. Verify Checkpoint Fingerprints

Calculate SHA-256 fingerprints for both checkpoints before loading.

Compare M5 with the fingerprint documented in its handoff.
Compare M4 with the fingerprint recorded when its downloaded file
was first inspected. This confirms file consistency, not independent
verification of its reported training results.

In [ ]:
import hashlib

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

expected_hashes = {
    "M4": "e034ca9b5610017d20b22e9014d99439ea3f98117d783faf09113c82a3d9098a",
    "M5": "5fb5f6925489060792d160f4a04bc3620c0b59a5de5aba013f2e6f5523254d44",
}

checkpoint_paths = {
    "M4": M4_CHECKPOINT,
    "M5": M5_CHECKPOINT,
}

checkpoint_hashes = {}

for name, path in checkpoint_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} checkpoint missing: {path}")

    fingerprint = file_sha256(path)
    checkpoint_hashes[name] = fingerprint

    print(f"{name}: {path.name}")
    print(f"Size: {path.stat().st_size / (1024 ** 2):.2f} MB")
    print(f"SHA-256: {fingerprint}")

    if fingerprint != expected_hashes[name]:
        raise ValueError(f"{name} fingerprint mismatch. Check the selected file.")

    print(f"PASS: {name} matches its recorded fingerprint.\n")

print("Checkpoint fingerprint checks completed. No models loaded yet.")

## 7. Load Models and Verify Class Mappings

Load the verified checkpoints on the selected device.

M4 must detect four damage classes: D00, D10, D20, and D40.
M5 must perform segmentation with one pothole class.

The integration maps M4 class 3 (D40) to M5 class 0.
This step loads the models without training or image inference.

In [ ]:
m4_model = YOLO(str(M4_CHECKPOINT))
m5_model = YOLO(str(M5_CHECKPOINT))

m4_model.to(DEVICE)
m5_model.to(DEVICE)

def normalize_names(names):
    if isinstance(names, dict):
        return {int(key): str(value) for key, value in names.items()}
    return dict(enumerate(map(str, names)))

m4_names = normalize_names(m4_model.names)
m5_names = normalize_names(m5_model.names)

print("M4 task:", m4_model.task)
print("M4 classes:", m4_names)
print("\nM5 task:", m5_model.task)
print("M5 classes:", m5_names)
print("\nDevice:", DEVICE)

expected_m4_names = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}

if m4_model.task != "detect":
    raise ValueError("M4 must be an object detection model.")

if m4_names != expected_m4_names:
    raise ValueError(f"Unexpected M4 class mapping: {m4_names}")

if m5_model.task != "segment":
    raise ValueError("M5 must be a segmentation model.")

if set(m5_names) != {0}:
    raise ValueError(f"M5 must contain exactly class 0: {m5_names}")

if m5_names[0].strip().lower() != "pothole":
    raise ValueError(
        f"Unexpected M5 class name: {m5_names[0]!r}. "
        "Review the mapping before proceeding."
    )

POTHOLE_M4_CLASS_ID = 3
POTHOLE_M5_CLASS_ID = 0

print("\nPASS: Both checkpoints loaded.")
print("PASS: Tasks and class mappings verified.")
print("Mapping: M4 D40 (3) -> M5 pothole (0).")
print("No training or image inference performed.")

## 8. Configure Initial Inference

Load M5's documented inference settings and verify that its
configuration references the loaded checkpoint.

Use explicit provisional M4 settings for initial smoke tests.
M4 operating thresholds must be justified on validation data
before final evaluation.

Smoke-test outputs are not accuracy measurements or benchmarks.

In [ ]:
import json

with M5_CONFIG.open("r", encoding="utf-8") as handle:
    m5_config = json.load(handle)

if m5_config.get("checkpoint_sha256") != checkpoint_hashes["M5"]:
    raise ValueError("M5 configuration references a different checkpoint.")

if (
    m5_config.get("m5_class_id") != POTHOLE_M5_CLASS_ID
    or m5_config.get("m4_class_id") != POTHOLE_M4_CLASS_ID
    or m5_config.get("damage_code") != "D40"
):
    raise ValueError("M5 configuration has an unexpected class mapping.")

M4_SETTINGS = {
    "imgsz": 640,
    "conf": 0.25,
    "iou": 0.70,
    "max_det": 300,
}

M5_SETTINGS = {
    "imgsz": int(m5_config["imgsz"]),
    "conf": float(m5_config["threshold"]),
    "iou": float(m5_config["nms_iou"]),
    "max_det": int(m5_config["max_det"]),
}

for model_name, settings in [
    ("M4", M4_SETTINGS),
    ("M5", M5_SETTINGS),
]:
    if not 0 < settings["conf"] < 1:
        raise ValueError(f"{model_name}: invalid confidence threshold.")
    if not 0 < settings["iou"] <= 1:
        raise ValueError(f"{model_name}: invalid NMS IoU threshold.")
    if settings["imgsz"] <= 0 or settings["max_det"] <= 0:
        raise ValueError(f"{model_name}: invalid size or detection limit.")

print("M4 provisional smoke-test settings:", M4_SETTINGS)
print("M5 documented settings:", M5_SETTINGS)
print("M5 preprocessing:", m5_config["resize"])

print("\nPASS: Initial inference settings prepared.")
print("PENDING: Validation-based selection of M4 operating thresholds.")

## 9. Load the Fixed Validation Smoke-Test Image

Load China_MotorBike_China_MotorBike_000357.jpg from the merged M4
validation dataset. This is the fixed example used for the recorded
smoke-test results and discussion.

Verify that the image is readable and its reference label contains D40.
Reference labels are used only to validate the example, not as model inputs.

The image is fixed explicitly because scanning the merged dataset for
the first D40 example now selects a different image.
This single example is not a performance evaluation.

In [ ]:
from pathlib import Path
import math

from IPython.display import display
from PIL import Image

VAL_IMAGES = Path(M4_DIR) / "data" / "yolo_dataset" / "images" / "val"
VAL_LABELS = Path(M4_DIR) / "data" / "yolo_dataset" / "labels" / "val"

SMOKE_IMAGE = (
    VAL_IMAGES / "China_MotorBike_China_MotorBike_000357.jpg"
)
smoke_label_path = VAL_LABELS / f"{SMOKE_IMAGE.stem}.txt"

for path in (SMOKE_IMAGE, smoke_label_path):
    if not path.is_file():
        raise FileNotFoundError(f"Required smoke-test file missing: {path}")

smoke_label_rows = []

for line_number, line in enumerate(
    smoke_label_path.read_text(encoding="utf-8").splitlines(),
    start=1,
):
    if not line.strip():
        continue

    fields = line.split()

    if len(fields) != 5:
        raise ValueError(
            f"Invalid label structure at line {line_number}."
        )

    class_value = float(fields[0])
    coordinates = [float(value) for value in fields[1:]]

    if (
        not math.isfinite(class_value)
        or not class_value.is_integer()
        or int(class_value) not in {0, 1, 2, 3}
    ):
        raise ValueError(f"Invalid class at line {line_number}.")

    if not all(math.isfinite(value) for value in coordinates):
        raise ValueError(
            f"Nonfinite coordinates at line {line_number}."
        )

    center_x, center_y, box_width, box_height = coordinates

    if not (
        0 <= center_x <= 1
        and 0 <= center_y <= 1
        and 0 < box_width <= 1
        and 0 < box_height <= 1
    ):
        raise ValueError(
            f"Invalid normalized coordinates at line {line_number}."
        )

    smoke_label_rows.append({
        "class_id": int(class_value),
        "coordinates": coordinates,
    })

d40_reference_count = sum(
    row["class_id"] == 3
    for row in smoke_label_rows
)

if d40_reference_count == 0:
    raise ValueError("The fixed smoke-test label does not contain D40.")

with Image.open(SMOKE_IMAGE) as opened:
    opened.load()
    smoke_original_size = opened.size
    preview = opened.convert("RGB")

if smoke_original_size != (512, 512):
    raise ValueError(
        f"Unexpected image dimensions: {smoke_original_size}. "
        "The recorded example was 512 × 512."
    )

print("Fixed smoke-test image:", SMOKE_IMAGE)
print("Original size (width, height):", smoke_original_size)
print("Reference D40 boxes:", d40_reference_count)

preview.thumbnail((900, 650))
display(preview)

print("PASS: Fixed validation smoke-test image verified.")
print("No model inference performed. No files modified.")

## 10. M4 Detection Smoke Test

Run the detector on the selected original validation image using
the provisional settings. Display predicted boxes, classes, and scores.

Ground-truth annotations are not passed to the model.
Zero detections are a valid runtime outcome, not evidence that the
image contains no damage. This is not an accuracy or latency benchmark.

In [ ]:
smoke_bgr = cv2.imread(str(SMOKE_IMAGE))

if smoke_bgr is None:
    raise ValueError(f"Could not decode image: {SMOKE_IMAGE}")

m4_smoke_result = m4_model.predict(
    source=smoke_bgr,
    device=DEVICE,
    **M4_SETTINGS,
    augment=False,
    save=False,
    verbose=False,
)[0]

if tuple(m4_smoke_result.orig_shape) != smoke_bgr.shape[:2]:
    raise ValueError("Prediction dimensions differ from the original image.")

m4_smoke_detections = []

for box in m4_smoke_result.boxes:
    class_id = int(box.cls.item())
    confidence = float(box.conf.item())
    coordinates = box.xyxy[0].cpu().tolist()

    if class_id not in m4_names:
        raise ValueError(f"Unknown detection class: {class_id}")

    m4_smoke_detections.append({
        "class_id": class_id,
        "damage_code": m4_names[class_id],
        "confidence": confidence,
        "bbox_xyxy": coordinates,
    })

print("Image:", SMOKE_IMAGE.name)
print("Settings:", M4_SETTINGS)
print("Detections:", len(m4_smoke_detections))

for detection in m4_smoke_detections:
    print(
        detection["damage_code"],
        f"confidence={detection['confidence']:.3f}",
        "box=", [round(value, 1) for value in detection["bbox_xyxy"]],
    )

if not m4_smoke_detections:
    print("No detections at the provisional thresholds.")

annotated_bgr = m4_smoke_result.plot()
display(Image.fromarray(cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)))

print("\nPASS: M4 inference completed.")
print("Detection quality has not been evaluated by this smoke test.")

## 11. Independent M5 Segmentation Smoke Test

Run M5 independently on the full validation image using its documented
square-resize preprocessing and inference settings.

Resize predicted masks back to the original image dimensions.
Calculate the union mask area as a percentage of the full image.

This checks inference and mask handling on a road image.
It does not validate segmentation accuracy or M4-to-M5 integration.

In [ ]:
height, width = smoke_bgr.shape[:2]
size = M5_SETTINGS["imgsz"]

m5_input = cv2.resize(
    smoke_bgr,
    (size, size),
    interpolation=cv2.INTER_AREA,
)

m5_smoke_result = m5_model.predict(
    source=m5_input,
    device=DEVICE,
    **M5_SETTINGS,
    retina_masks=True,
    augment=False,
    save=False,
    verbose=False,
)[0]

m5_union_mask = np.zeros((height, width), dtype=bool)
m5_smoke_instances = []

if m5_smoke_result.masks is not None:
    masks = m5_smoke_result.masks.data.cpu().numpy()
    scores = m5_smoke_result.boxes.conf.cpu().numpy()
    classes = m5_smoke_result.boxes.cls.cpu().numpy()

    if not (len(masks) == len(scores) == len(classes)):
        raise ValueError("Mask, score, and class counts differ.")

    for mask, score, class_id in zip(masks, scores, classes):
        if int(class_id) != POTHOLE_M5_CLASS_ID:
            raise ValueError(f"Unexpected M5 class: {class_id}")

        original_mask = cv2.resize(
            (mask > 0.5).astype(np.uint8),
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        ).astype(bool)

        m5_union_mask |= original_mask
        m5_smoke_instances.append({
            "confidence": float(score),
            "area_pixels": int(original_mask.sum()),
        })

m5_area_percent = 100.0 * float(m5_union_mask.mean())

print("Image:", SMOKE_IMAGE.name)
print("Settings:", M5_SETTINGS)
print("Predicted instances:", len(m5_smoke_instances))
print("Union mask pixels:", int(m5_union_mask.sum()))
print(f"Predicted area / full image: {m5_area_percent:.3f}%")

overlay = smoke_bgr.copy()
if m5_union_mask.any():
    color = np.array([0, 255, 0], dtype=np.float32)
    overlay[m5_union_mask] = (
        0.6 * smoke_bgr[m5_union_mask].astype(np.float32)
        + 0.4 * color
    ).astype(np.uint8)

display(Image.fromarray(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)))

print("\nPASS: Independent M5 inference completed.")
print("Green indicates predicted pothole pixels, not verified ground truth.")

## 12. Initial Smoke-Test Findings

Both models loaded and completed inference on the selected validation
image, China_MotorBike_China_MotorBike_000357.jpg.

### M4
- Returned one D10 detection with confidence approximately 0.273.
- Returned no D40 detection at the provisional confidence threshold 0.25.
- The image was selected because its reference labels contain D40.

### M5
- Returned five predicted instances.
- The union mask covered 18,850 pixels, or approximately 7.191%
  of the full image.
- Visual inspection shows predicted pothole pixels on parts of the
  vehicle and manhole cover, indicating false-positive regions.
- No reference segmentation mask was used to quantify accuracy here.

### Integration Implications
- Successful execution does not establish prediction quality.
- A pipeline that segments only M4-detected potholes would skip M5
  for this image because M4 returned no D40 detection.
- Restricting segmentation to detected pothole crops is a candidate
  design that requires validation; it is not a demonstrated fix.
- Retain this example for failure analysis. Do not tune thresholds
  solely to improve this single image.

## 13. Select a Fixed Development Subset

Use the approved M4 validation population from China Drone,
China MotorBike, and Czech.

Select up to 30 eligible images per source using seed 42.
Exclude the fixed smoke-test image:
China_MotorBike_China_MotorBike_000357.jpg.

Missing or empty YOLO labels are accepted only when the original XML
annotation verifies that the image contains no annotated objects.
Unverified missing or empty labels must not be treated as backgrounds.

Sampling from the eligible population does not depend on model
predictions or target-class counts. Eligibility depends on the inclusion
manifest and annotation checks.

Save a manifest containing relative paths, image and label fingerprints,
and background-verification evidence.

Running this cell creates a new timestamped development-run directory.
Subsequent generated artifacts belong to that run.

This subset supports development and threshold exploration.
It is not an independent final test or a full-validation evaluation.

In [ ]:
import csv
import hashlib
import json
import random
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path

DEV_SEED = 42
DEV_IMAGES_PER_SOURCE = 30
DEV_SOURCES = ["China_Drone", "China_MotorBike", "Czech"]

# Updated merged dataset paths
M4_DIR = ASSETS_DIR / "M4_Combined"
M4_CHECKPOINT = M4_DIR / "runs/yolo11n_baseline/weights/best.pt"
VAL_IMAGES = M4_DIR / "data/yolo_dataset/images/val"
VAL_LABELS = M4_DIR / "data/yolo_dataset/labels/val"
SMOKE_IMAGE = VAL_IMAGES / "China_MotorBike_China_MotorBike_000357.jpg"
INCLUSION_MANIFEST = M4_DIR / "m4_inclusion_manifest.csv"

# Original country ZIP archives from Phase 1
RAW_ARCHIVE_ROOT = PROJECT_ROOT.parents[1]


def file_sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


with INCLUSION_MANIFEST.open(
    encoding="utf-8-sig", newline=""
) as stream:
    inclusion_rows = list(csv.DictReader(stream))

eligible = [
    row for row in inclusion_rows
    if row["split"] == "val"
    and row["include_m4"].lower() == "true"
]

assert len(eligible) == 6687
assert len({row["image_id"] for row in eligible}) == len(eligible)
assert {row["country"] for row in eligible} == set(DEV_SOURCES)

image_lookup = {}

for path in sorted(VAL_IMAGES.iterdir()):
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        assert path.stem not in image_lookup, (
            f"Duplicate image ID: {path.stem}"
        )
        image_lookup[path.stem] = path

missing_images = [
    row["image_id"] for row in eligible
    if row["image_id"] not in image_lookup
]

assert not missing_images, f"Missing images: {missing_images[:5]}"

# Verify absent/empty labels against original XML annotations.
# Missing labels alone are NOT evidence of background.
background_evidence = {}

for source in DEV_SOURCES:
    needs_xml = []

    for row in eligible:
        if row["country"] != source:
            continue

        label_path = VAL_LABELS / f"{row['image_id']}.txt"

        if (
            not label_path.is_file()
            or not label_path.read_text(encoding="utf-8").strip()
        ):
            needs_xml.append(row)

    if not needs_xml:
        continue

    archives = sorted(RAW_ARCHIVE_ROOT.rglob(source + ".zip"))

    assert len(archives) == 1, (
        f"Expected one original {source}.zip; found {len(archives)}"
    )

    archive = archives[0]

    with zipfile.ZipFile(archive) as zip_file:
        xml_lookup = {}

        for member in zip_file.namelist():
            if member.lower().endswith(".xml"):
                xml_lookup.setdefault(
                    Path(member).stem, []
                ).append(member)

        for row in needs_xml:
            image_id = row["image_id"]
            xml_stem = image_id[len(source) + 1:]
            matches = xml_lookup.get(xml_stem, [])

            assert len(matches) == 1, (
                f"Missing or ambiguous XML: {image_id}"
            )

            raw_xml = zip_file.read(matches[0])
            objects = ET.fromstring(raw_xml).findall(".//object")

            assert not objects, (
                f"Label missing/empty but XML contains objects: {image_id}"
            )

            background_evidence[image_id] = {
                "archive": archive.relative_to(
                    RAW_ARCHIVE_ROOT
                ).as_posix(),
                "xml_member": matches[0],
                "xml_sha256": hashlib.sha256(raw_xml).hexdigest(),
                "object_count": 0,
            }

# Sample from the approved validation population.
rng = random.Random(DEV_SEED)
dev_records = []

for source in DEV_SOURCES:
    candidates = sorted(
        row["image_id"] for row in eligible
        if row["country"] == source
        and row["image_id"] != SMOKE_IMAGE.stem
    )

    selected = sorted(
        rng.sample(
            candidates,
            min(DEV_IMAGES_PER_SOURCE, len(candidates))
        )
    )

    for image_id in selected:
        image_path = image_lookup[image_id]
        label_path = VAL_LABELS / f"{image_id}.txt"
        label_exists = label_path.is_file()

        label_text = (
            label_path.read_text(encoding="utf-8").strip()
            if label_exists else ""
        )

        if not label_text:
            assert image_id in background_evidence

        dev_records.append({
            "source": source,
            "image": image_path.relative_to(M4_DIR).as_posix(),
            "label": (
                label_path.relative_to(M4_DIR).as_posix()
                if label_exists else None
            ),
            "label_exists": label_exists,
            "image_sha256": file_sha256(image_path),
            "label_sha256": (
                file_sha256(label_path) if label_exists else None
            ),
            "empty_label": not bool(label_text),
            "annotation_status": (
                "xml_verified_background"
                if not label_text else "annotated"
            ),
            "background_evidence": background_evidence.get(image_id),
        })

assert len({row["image"] for row in dev_records}) == len(dev_records)

RUN_DIR = M8_DIR / "results" / (
    "development_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
RUN_DIR.mkdir(parents=True, exist_ok=False)

dev_manifest = {
    "purpose": "development_only",
    "seed": DEV_SEED,
    "requested_images_per_source": DEV_IMAGES_PER_SOURCE,
    "excluded_smoke_image": SMOKE_IMAGE.relative_to(M4_DIR).as_posix(),
    "inclusion_manifest_sha256": file_sha256(INCLUSION_MANIFEST),
    "eligible_images": len(eligible),
    "verified_backgrounds": background_evidence,
    "records": dev_records,
}

manifest_path = RUN_DIR / "development_manifest.json"
manifest_path.write_text(
    json.dumps(dev_manifest, indent=2),
    encoding="utf-8",
)

for source in DEV_SOURCES:
    rows = [row for row in dev_records if row["source"] == source]
    backgrounds = sum(row["empty_label"] for row in rows)
    print(
        f"{source}: {len(rows)} images, "
        f"{backgrounds} XML-verified backgrounds"
    )

print("\nEligible validation images:", len(eligible))
print("XML-verified backgrounds:", len(background_evidence))
print("Total development images:", len(dev_records))
print("Manifest saved to:", manifest_path)
print("PASS: Development subset selected and fingerprinted.")
print("No inference performed. Norway test data was not selected.")

## 14. Validate Development Labels

Validate image readability and YOLO annotation structure, class IDs,
finite coordinates, and bounding-box dimensions.

Missing or empty labels are accepted only for XML-verified backgrounds.
Record class counts and prepare ground-truth boxes in original-image
pixel coordinates. No model inference or test-set evaluation is performed.

In [ ]:
import math
from collections import Counter
from PIL import Image

CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}

dev_ground_truth = {}
class_box_counts = Counter()
class_image_counts = Counter()
background_count = 0

for record in dev_records:
    image_path = M4_DIR / record["image"]

    # Ensure the selected image has not changed.
    assert file_sha256(image_path) == record["image_sha256"]

    with Image.open(image_path) as image:
        image.load()
        width, height = image.size

    assert width > 0 and height > 0

    label_path = (
        M4_DIR / record["label"]
        if record["label"] is not None else None
    )

    if label_path is not None:
        assert label_path.is_file(), f"Missing label: {label_path}"
        assert file_sha256(label_path) == record["label_sha256"]
        lines = [
            line.strip()
            for line in label_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    else:
        lines = []

    boxes = []

    for line_number, line in enumerate(lines, start=1):
        context = f"{image_path.name}, label line {line_number}"
        fields = line.split()

        assert len(fields) == 5, f"Invalid YOLO row: {context}"

        values = list(map(float, fields))
        assert all(math.isfinite(value) for value in values), context

        class_value, xc, yc, bw, bh = values
        assert class_value.is_integer(), f"Non-integer class: {context}"

        class_id = int(class_value)
        assert class_id in CLASS_NAMES, f"Unknown class: {context}"
        assert 0 <= xc <= 1 and 0 <= yc <= 1, context
        assert 0 < bw <= 1 and 0 < bh <= 1, context

        x1, y1 = xc - bw / 2, yc - bh / 2
        x2, y2 = xc + bw / 2, yc + bh / 2

        # Allow only tiny rounding errors from saved normalized values.
        tolerance = 1e-6
        assert (
            x1 >= -tolerance and y1 >= -tolerance
            and x2 <= 1 + tolerance and y2 <= 1 + tolerance
        ), f"Box extends beyond image: {context}"

        boxes.append({
            "class_id": class_id,
            "damage_code": CLASS_NAMES[class_id],
            "bbox_xyxy": [
                max(0.0, x1) * width,
                max(0.0, y1) * height,
                min(1.0, x2) * width,
                min(1.0, y2) * height,
            ],
        })

        class_box_counts[class_id] += 1

    if not boxes:
        assert record["annotation_status"] == "xml_verified_background"
        evidence = record["background_evidence"]
        assert evidence is not None and evidence["object_count"] == 0
        background_count += 1

    for class_id in {box["class_id"] for box in boxes}:
        class_image_counts[class_id] += 1

    dev_ground_truth[record["image"]] = {
        "source": record["source"],
        "width": width,
        "height": height,
        "annotation_status": record["annotation_status"],
        "boxes": boxes,
    }

audit = {
    "purpose": "development_only",
    "images": len(dev_ground_truth),
    "background_images": background_count,
    "annotated_images": len(dev_ground_truth) - background_count,
    "total_boxes": sum(class_box_counts.values()),
    "boxes_per_class": {
        name: class_box_counts[class_id]
        for class_id, name in CLASS_NAMES.items()
    },
    "images_per_class": {
        name: class_image_counts[class_id]
        for class_id, name in CLASS_NAMES.items()
    },
}

(RUN_DIR / "development_label_audit.json").write_text(
    json.dumps(audit, indent=2), encoding="utf-8"
)
(RUN_DIR / "development_ground_truth.json").write_text(
    json.dumps(dev_ground_truth, indent=2), encoding="utf-8"
)

print(json.dumps(audit, indent=2))
print("\nPASS: Development images and labels validated.")
print("Ground-truth boxes use original-image pixel coordinates.")
print("No model inference performed.")

## 15. M4 Development Baseline Predictions

Run M4 on the fixed development subset using the provisional settings.
Save predictions in original-image pixel coordinates, including images
with no detections.

This run establishes an initial baseline, not a final operating threshold.
D40 has only five reference boxes across two images in this subset.
Prediction counts do not measure accuracy, and this run is not a latency
benchmark.

In [ ]:
import math
from collections import Counter

# Keep this baseline explicit and separate from later threshold selection.
M4_BASELINE_SETTINGS = {
    "imgsz": 640,
    "conf": 0.25,
    "iou": 0.70,
    "max_det": 300,
}

assert m4_model.task == "detect"
assert {
    int(key): value for key, value in m4_model.names.items()
} == CLASS_NAMES

m4_dev_predictions = {}
prediction_counts = Counter()
images_without_detections = 0

for index, record in enumerate(dev_records, start=1):
    image_key = record["image"]
    image_path = M4_DIR / image_key
    reference = dev_ground_truth[image_key]

    assert file_sha256(image_path) == record["image_sha256"], (
        f"Image changed since selection: {image_path.name}"
    )

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        raise ValueError(f"Cannot decode image: {image_path}")

    assert image_bgr.shape[:2] == (
        reference["height"], reference["width"]
    )

    result = m4_model.predict(
        source=image_bgr,
        device=DEVICE,
        **M4_BASELINE_SETTINGS,
        augment=False,
        save=False,
        verbose=False,
    )[0]

    assert tuple(result.orig_shape) == image_bgr.shape[:2]

    detections = []

    if result.boxes is not None:
        coordinates = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        for box, confidence, class_value in zip(
            coordinates, confidences, classes
        ):
            assert float(class_value).is_integer()
            class_id = int(class_value)
            confidence = float(confidence)
            box = [float(value) for value in box]

            assert class_id in CLASS_NAMES
            assert math.isfinite(confidence) and 0 <= confidence <= 1
            assert all(math.isfinite(value) for value in box)

            x1, y1, x2, y2 = box
            assert 0 <= x1 < x2 <= reference["width"]
            assert 0 <= y1 < y2 <= reference["height"]

            detections.append({
                "class_id": class_id,
                "damage_code": CLASS_NAMES[class_id],
                "confidence": confidence,
                "bbox_xyxy": box,
            })

            prediction_counts[class_id] += 1

    detections.sort(
        key=lambda detection: detection["confidence"],
        reverse=True,
    )

    if not detections:
        images_without_detections += 1

    m4_dev_predictions[image_key] = detections

    if index % 10 == 0 or index == len(dev_records):
        print(f"Processed {index}/{len(dev_records)} images")

assert set(m4_dev_predictions) == set(dev_ground_truth)

prediction_report = {
    "purpose": "development_baseline",
    "settings_status": "provisional",
    "settings": M4_BASELINE_SETTINGS,
    "device": DEVICE,
    "checkpoint_sha256": file_sha256(M4_CHECKPOINT),
    "development_manifest_sha256": file_sha256(manifest_path),
    "images": len(m4_dev_predictions),
    "images_without_detections": images_without_detections,
    "total_detections": sum(prediction_counts.values()),
    "detections_per_class": {
        name: prediction_counts[class_id]
        for class_id, name in CLASS_NAMES.items()
    },
    "predictions": m4_dev_predictions,
}

prediction_path = RUN_DIR / "m4_baseline_predictions.json"
prediction_path.write_text(
    json.dumps(prediction_report, indent=2),
    encoding="utf-8",
)

print("\nDetections per class:")
for class_id, name in CLASS_NAMES.items():
    print(f"{name}: {prediction_counts[class_id]}")

print("Images without detections:", images_without_detections)
print("Predictions saved to:", prediction_path)
print("PASS: M4 development baseline inference completed.")
print("Accuracy has not yet been calculated.")

## 16. Evaluate the M4 Development Baseline

Match predictions to reference boxes of the same class at IoU >= 0.50,
processing predictions in descending confidence order. Each reference
box may be matched once.

Report TP, FP, FN, precision, recall, and F1 at the provisional operating
point. Matching IoU (0.50) is separate from inference NMS IoU (0.70).

These are development-subset results, not AP/mAP or final test results.
D40 estimates are based on only five reference boxes across two images.
Undefined precision is reported as null.

In [ ]:
MATCH_IOU = 0.50


def box_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def calculate_metrics(tp, fp, fn):
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "reference_boxes": tp + fn,
        "predicted_boxes": tp + fp,
        "precision": tp / (tp + fp) if tp + fp else None,
        "recall": tp / (tp + fn) if tp + fn else None,
        "f1": (
            2 * tp / (2 * tp + fp + fn)
            if 2 * tp + fp + fn else None
        ),
    }


assert set(m4_dev_predictions) == set(dev_ground_truth)

counts = {
    class_id: {"TP": 0, "FP": 0, "FN": 0}
    for class_id in CLASS_NAMES
}
matching_details = {}

for image_key, reference in dev_ground_truth.items():
    ground_truth = reference["boxes"]
    predictions = sorted(
        m4_dev_predictions[image_key],
        key=lambda item: item["confidence"],
        reverse=True,
    )

    matched_gt = set()
    prediction_matches = []

    for prediction in predictions:
        class_id = prediction["class_id"]

        candidates = [
            (
                gt_index,
                box_iou(prediction["bbox_xyxy"], gt["bbox_xyxy"]),
            )
            for gt_index, gt in enumerate(ground_truth)
            if gt_index not in matched_gt
            and gt["class_id"] == class_id
        ]

        best_index, best_iou = max(
            candidates,
            key=lambda item: item[1],
            default=(None, 0.0),
        )

        is_match = best_index is not None and best_iou >= MATCH_IOU

        if is_match:
            matched_gt.add(best_index)
            counts[class_id]["TP"] += 1
        else:
            counts[class_id]["FP"] += 1

        prediction_matches.append({
            **prediction,
            "outcome": "TP" if is_match else "FP",
            "matched_gt_index": best_index if is_match else None,
            "best_available_same_class_iou": best_iou,
        })

    missed_gt_indices = [
        index for index in range(len(ground_truth))
        if index not in matched_gt
    ]

    for index in missed_gt_indices:
        counts[ground_truth[index]["class_id"]]["FN"] += 1

    matching_details[image_key] = {
        "source": reference["source"],
        "predictions": prediction_matches,
        "missed_gt_indices": missed_gt_indices,
    }

per_class_metrics = {
    CLASS_NAMES[class_id]: calculate_metrics(
    class_counts["TP"], class_counts["FP"], class_counts["FN"]
)
    for class_id, class_counts in counts.items()
}

# Verify that matching accounts for every prediction and reference box.
for class_id, name in CLASS_NAMES.items():
    metrics = per_class_metrics[name]
    assert metrics["reference_boxes"] == class_box_counts[class_id]
    assert metrics["predicted_boxes"] == prediction_counts[class_id]

totals = {
    key: sum(class_counts[key] for class_counts in counts.values())
    for key in ("TP", "FP", "FN")
}
micro_metrics = calculate_metrics(
    totals["TP"], totals["FP"], totals["FN"]
)
evaluation_report = {
    "purpose": "development_baseline",
    "inference_settings": M4_BASELINE_SETTINGS,
    "matching_iou": MATCH_IOU,
    "matching_method": "confidence_ordered_same_class_one_to_one",
    "prediction_file_sha256": file_sha256(prediction_path),
    "ground_truth_file_sha256": file_sha256(
        RUN_DIR / "development_ground_truth.json"
    ),
    "per_class": per_class_metrics,
    "micro": micro_metrics,
}

(RUN_DIR / "m4_baseline_metrics.json").write_text(
    json.dumps(evaluation_report, indent=2, allow_nan=False),
    encoding="utf-8",
)
(RUN_DIR / "m4_baseline_matching_details.json").write_text(
    json.dumps(matching_details, indent=2, allow_nan=False),
    encoding="utf-8",
)


def show_metric(value):
    return "N/A" if value is None else f"{value:.3f}"


print(f"Matching IoU: {MATCH_IOU}")
print(f"{'Class':<8}{'TP':>5}{'FP':>5}{'FN':>5}"
      f"{'Precision':>12}{'Recall':>10}{'F1':>10}")

for name, metrics in [
    *per_class_metrics.items(),
    ("Micro", micro_metrics),
]:
    print(
        f"{name:<8}{metrics['TP']:>5}{metrics['FP']:>5}{metrics['FN']:>5}"
        f"{show_metric(metrics['precision']):>12}"
        f"{show_metric(metrics['recall']):>10}"
        f"{show_metric(metrics['f1']):>10}"
    )

print("\nPASS: Development baseline matching and metrics completed.")
print("D40 support: 5 reference boxes across 2 images.")
print("These results are not AP/mAP or final test performance.")

## 17. Inventory D40 Validation Support

Count D40 reference boxes and images within the approved M4 validation
population, grouped by source. Identify examples outside the initial
development subset and the previously inspected smoke-test image.

This label-based inventory supports planning a separate pothole diagnostic
cohort. Such a cohort is enriched for D40 and must not be presented as
representative overall detection performance.

No inference or test-set evaluation is performed.

In [ ]:
from collections import Counter
import math

existing_dev_images = {
    Path(record["image"]).stem for record in dev_records
}

d40_inventory = []
source_counts = {
    source: Counter()
    for source in DEV_SOURCES
}

for row in sorted(eligible, key=lambda item: item["image_id"]):
    image_id = row["image_id"]
    source = row["country"]
    label_path = VAL_LABELS / f"{image_id}.txt"

    if not label_path.is_file():
        assert image_id in background_evidence, (
            f"Unverified missing label: {image_id}"
        )
        continue

    lines = [
        line.strip()
        for line in label_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    if not lines:
        assert image_id in background_evidence, (
            f"Unverified empty label: {image_id}"
        )
        continue

    d40_boxes = 0

    for line_number, line in enumerate(lines, start=1):
        context = f"{image_id}, line {line_number}"
        fields = line.split()

        assert len(fields) == 5, f"Invalid YOLO row: {context}"
        values = list(map(float, fields))
        assert all(math.isfinite(value) for value in values), context

        class_value, xc, yc, bw, bh = values
        assert class_value.is_integer(), context
        assert int(class_value) in CLASS_NAMES, context
        assert 0 <= xc <= 1 and 0 <= yc <= 1, context
        assert 0 < bw <= 1 and 0 < bh <= 1, context

        tolerance = 1e-6
        assert (
            xc - bw / 2 >= -tolerance
            and yc - bh / 2 >= -tolerance
            and xc + bw / 2 <= 1 + tolerance
            and yc + bh / 2 <= 1 + tolerance
        ), f"Out-of-bounds box: {context}"

        if int(class_value) == 3:
            d40_boxes += 1

    if not d40_boxes:
        continue

    in_initial_subset = image_id in existing_dev_images
    is_smoke_image = image_id == SMOKE_IMAGE.stem
    available_for_extension = not in_initial_subset and not is_smoke_image

    source_counts[source]["images"] += 1
    source_counts[source]["boxes"] += d40_boxes
    source_counts[source]["additional_images"] += int(
        available_for_extension
    )

    d40_inventory.append({
        "image_id": image_id,
        "source": source,
        "image": image_lookup[image_id].relative_to(M4_DIR).as_posix(),
        "label": label_path.relative_to(M4_DIR).as_posix(),
        "label_sha256": file_sha256(label_path),
        "d40_boxes": d40_boxes,
        "in_initial_subset": in_initial_subset,
        "is_smoke_image": is_smoke_image,
        "available_for_extension": available_for_extension,
    })

inventory_report = {
    "purpose": "validation_label_inventory",
    "inclusion_manifest_sha256": file_sha256(INCLUSION_MANIFEST),
    "per_source": {
        source: {
            "d40_images": source_counts[source]["images"],
            "d40_boxes": source_counts[source]["boxes"],
            "additional_images": source_counts[source]["additional_images"],
        }
        for source in DEV_SOURCES
    },
    "records": d40_inventory,
}

inventory_path = RUN_DIR / "d40_validation_inventory.json"
inventory_path.write_text(
    json.dumps(inventory_report, indent=2),
    encoding="utf-8",
)

print(f"{'Source':<20}{'D40 images':>12}{'D40 boxes':>12}{'Additional':>12}")

for source, counts in inventory_report["per_source"].items():
    print(
        f"{source:<20}"
        f"{counts['d40_images']:>12}"
        f"{counts['d40_boxes']:>12}"
        f"{counts['additional_images']:>12}"
    )

print("\nInventory saved to:", inventory_path)
print("PASS: Approved validation D40 support inventoried.")
print("No inference performed. No test data accessed.")

## 18. Select a Fixed D40 Diagnostic Cohort

Select 30 additional D40-positive validation images per source using
seed 42. Exclude the initial development subset and the smoke-test image.

Verify image readability and label fingerprints, then save the selection
with image and label hashes. Retain all reference classes in each image.

This cohort is enriched for potholes. Its results will be reported
separately from the general development subset. It is not a final test
set or a representative estimate of overall false-positive frequency.

In [ ]:
import random
from collections import Counter
from PIL import Image

D40_DIAGNOSTIC_SEED = 42
D40_IMAGES_PER_SOURCE = 30

d40_rng = random.Random(D40_DIAGNOSTIC_SEED)
d40_dev_records = []
d40_source_summary = {}

for source in DEV_SOURCES:
    candidates = sorted(
        (
            record for record in d40_inventory
            if record["source"] == source
            and record["available_for_extension"]
        ),
        key=lambda record: record["image_id"],
    )

    assert len(candidates) >= D40_IMAGES_PER_SOURCE, (
        f"Insufficient additional D40 images for {source}"
    )

    selected = sorted(
        d40_rng.sample(candidates, D40_IMAGES_PER_SOURCE),
        key=lambda record: record["image_id"],
    )

    box_counts = Counter()

    for record in selected:
        image_path = M4_DIR / record["image"]
        label_path = M4_DIR / record["label"]

        assert image_path.is_file()
        assert label_path.is_file()
        assert file_sha256(label_path) == record["label_sha256"], (
            f"Label changed since inventory: {label_path.name}"
        )

        with Image.open(image_path) as image:
            image.load()
            width, height = image.size

        assert width > 0 and height > 0

        # Labels were validated in section 17; hashes must still match.
        # Keep every class, not only D40.
        boxes = []

        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue

            class_value, xc, yc, bw, bh = map(float, line.split())
            class_id = int(class_value)

            boxes.append({
                "class_id": class_id,
                "damage_code": CLASS_NAMES[class_id],
                "bbox_xyxy": [
                    max(0.0, xc - bw / 2) * width,
                    max(0.0, yc - bh / 2) * height,
                    min(1.0, xc + bw / 2) * width,
                    min(1.0, yc + bh / 2) * height,
                ],
            })
            box_counts[class_id] += 1

        d40_count = sum(box["class_id"] == 3 for box in boxes)
        assert d40_count == record["d40_boxes"] and d40_count > 0

        d40_dev_records.append({
            "source": source,
            "image_id": record["image_id"],
            "image": record["image"],
            "label": record["label"],
            "image_sha256": file_sha256(image_path),
            "label_sha256": record["label_sha256"],
            "width": width,
            "height": height,
            "d40_boxes": d40_count,
            "boxes": boxes,
        })

    d40_source_summary[source] = {
        "images": len(selected),
        "d40_boxes": box_counts[3],
        "all_class_boxes": sum(box_counts.values()),
    }

selected_ids = {record["image_id"] for record in d40_dev_records}

assert len(selected_ids) == len(d40_dev_records) == 90
assert not selected_ids.intersection(existing_dev_images)
assert SMOKE_IMAGE.stem not in selected_ids

d40_manifest = {
    "purpose": "d40_enriched_development_diagnostics",
    "seed": D40_DIAGNOSTIC_SEED,
    "images_per_source": D40_IMAGES_PER_SOURCE,
    "selection_basis": "reference D40 presence, not model predictions",
    "inventory_sha256": file_sha256(inventory_path),
    "excluded_general_manifest_sha256": file_sha256(manifest_path),
    "per_source": d40_source_summary,
    "records": d40_dev_records,
}

d40_manifest_path = RUN_DIR / "d40_diagnostic_manifest.json"
d40_manifest_path.write_text(
    json.dumps(d40_manifest, indent=2),
    encoding="utf-8",
)

print(f"{'Source':<20}{'Images':>10}{'D40 boxes':>12}{'All boxes':>12}")

for source, summary in d40_source_summary.items():
    print(
        f"{source:<20}"
        f"{summary['images']:>10}"
        f"{summary['d40_boxes']:>12}"
        f"{summary['all_class_boxes']:>12}"
    )

print("\nTotal diagnostic images:", len(d40_dev_records))
print("Overlap with initial subset: 0")
print("Manifest saved to:", d40_manifest_path)
print("PASS: D40 diagnostic cohort selected and fingerprinted.")
print("No inference performed. No test data accessed.")

## 19. Collect Predictions for the Development Threshold Sweep

Evaluate NMS IoU values 0.50, 0.70, and 0.90 on both fixed development
cohorts. Save predictions at confidence 0.05 for subsequent comparison
of confidence thresholds 0.05, 0.10, 0.25, and 0.50.

Keep the general and D40-enriched cohorts separate when reporting metrics.
Record any images reaching max_det, because truncation can affect results.
This is development inference, not a latency benchmark or final test.

In [ ]:
import math
import time

CONFIDENCE_GRID = [0.05, 0.10, 0.25, 0.50]
NMS_IOU_GRID = [0.50, 0.70, 0.90]

SWEEP_BASE_SETTINGS = {
    "imgsz": 640,
    "conf": min(CONFIDENCE_GRID),
    "max_det": 300,
}

cohort_records = {
    "general": dev_records,
    "d40_diagnostic": d40_dev_records,
}

# Verify inputs before starting the longer inference run.
for cohort_name, records in cohort_records.items():
    for record in records:
        image_path = M4_DIR / record["image"]
        assert file_sha256(image_path) == record["image_sha256"], (
            f"Image changed: {image_path}"
        )

sweep_checkpoint_hash = file_sha256(M4_CHECKPOINT)
assert sweep_checkpoint_hash == checkpoint_hashes["M4"], (
    "M4 checkpoint differs from the previously verified checkpoint."
)

SWEEP_DIR = RUN_DIR / (
    "threshold_sweep_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
SWEEP_DIR.mkdir(parents=True, exist_ok=False)

sweep_predictions = {}
sweep_output_paths = []
total_images = sum(len(records) for records in cohort_records.values())
started = time.perf_counter()

for nms_iou in NMS_IOU_GRID:
    nms_key = f"{nms_iou:.2f}"
    sweep_predictions[nms_key] = {}
    max_det_hits = []
    completed = 0

    for cohort_name, records in cohort_records.items():
        cohort_predictions = {}

        for record in records:
            image_key = record["image"]
            image_path = M4_DIR / image_key
            image_bgr = cv2.imread(str(image_path))

            if image_bgr is None:
                raise ValueError(f"Cannot decode: {image_path}")

            height, width = image_bgr.shape[:2]

            result = m4_model.predict(
                source=image_bgr,
                device=DEVICE,
                **SWEEP_BASE_SETTINGS,
                iou=nms_iou,
                augment=False,
                save=False,
                verbose=False,
            )[0]

            assert tuple(result.orig_shape) == (height, width)
            detections = []

            if result.boxes is not None:
                coordinates = result.boxes.xyxy.cpu().numpy()
                confidences = result.boxes.conf.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy()

                for box, confidence, class_value in zip(
                    coordinates, confidences, classes
                ):
                    assert float(class_value).is_integer()
                    class_id = int(class_value)
                    confidence = float(confidence)
                    box = [float(value) for value in box]

                    assert class_id in CLASS_NAMES
                    assert math.isfinite(confidence)
                    assert 0 <= confidence <= 1
                    assert all(math.isfinite(value) for value in box)

                    x1, y1, x2, y2 = box
                    assert 0 <= x1 < x2 <= width
                    assert 0 <= y1 < y2 <= height

                    detections.append({
                        "class_id": class_id,
                        "damage_code": CLASS_NAMES[class_id],
                        "confidence": confidence,
                        "bbox_xyxy": box,
                    })

            detections.sort(
                key=lambda item: item["confidence"],
                reverse=True,
            )
            cohort_predictions[image_key] = detections

            if len(detections) >= SWEEP_BASE_SETTINGS["max_det"]:
                max_det_hits.append({
                    "cohort": cohort_name,
                    "image": image_key,
                })

            completed += 1

            if completed % 30 == 0 or completed == total_images:
                print(
                    f"NMS IoU={nms_iou:.2f}: "
                    f"{completed}/{total_images} images",
                    flush=True,
                )

        sweep_predictions[nms_key][cohort_name] = cohort_predictions

    output = {
        "purpose": "development_threshold_sweep",
        "settings": {**SWEEP_BASE_SETTINGS, "iou": nms_iou},
        "confidence_grid": CONFIDENCE_GRID,
        "device": DEVICE,
        "checkpoint_sha256": sweep_checkpoint_hash,
        "general_manifest_sha256": file_sha256(manifest_path),
        "d40_manifest_sha256": file_sha256(d40_manifest_path),
        "max_det_hits": max_det_hits,
        "predictions": sweep_predictions[nms_key],
    }

    output_path = SWEEP_DIR / f"predictions_nms_{nms_key}.json"
    output_path.write_text(
        json.dumps(output, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    sweep_output_paths.append(output_path)

    print(f"Saved: {output_path.name}")
    print(f"Images reaching max_det: {len(max_det_hits)}")

elapsed = time.perf_counter() - started

print("\nPASS: Development sweep predictions collected.")
print("NMS settings:", NMS_IOU_GRID)
print("Confidence thresholds to compare:", CONFIDENCE_GRID)
print("Output directory:", SWEEP_DIR)
print(f"Total elapsed time: {elapsed:.1f} seconds (not a benchmark)")
print("No final operating threshold selected. No test data accessed.")

## 20. Compare Development Operating Points

Evaluate the cached predictions at four confidence thresholds for each
NMS setting. Match same-class boxes one-to-one at IoU >= 0.50.

Report per-class and micro metrics separately for the general and
D40-enriched cohorts. Also report D40 false positives per general-cohort
image and the fraction of reference-background images with any detection.

These are development comparisons. No final operating point is selected
automatically, and the enriched cohort does not represent deployment
class prevalence.

In [ ]:
import csv

MATCH_IOU = 0.50

cohort_ground_truth = {
    "general": dev_ground_truth,
    "d40_diagnostic": {
        record["image"]: {
            "source": record["source"],
            "boxes": record["boxes"],
        }
        for record in d40_dev_records
    },
}


def evaluate_operating_point(predictions, ground_truth, confidence):
    counts = {
        class_id: {"TP": 0, "FP": 0, "FN": 0}
        for class_id in CLASS_NAMES
    }

    background_images = 0
    background_images_with_detections = 0

    assert set(predictions) == set(ground_truth)

    for image_key, reference in ground_truth.items():
        gt_boxes = reference["boxes"]
        selected = sorted(
            (
                prediction for prediction in predictions[image_key]
                if prediction["confidence"] >= confidence
            ),
            key=lambda prediction: prediction["confidence"],
            reverse=True,
        )

        if not gt_boxes:
            background_images += 1
            background_images_with_detections += int(bool(selected))

        matched_gt = set()

        for prediction in selected:
            class_id = prediction["class_id"]

            candidates = [
                (
                    index,
                    box_iou(prediction["bbox_xyxy"], gt["bbox_xyxy"]),
                )
                for index, gt in enumerate(gt_boxes)
                if index not in matched_gt
                and gt["class_id"] == class_id
            ]

            best_index, best_iou = max(
                candidates,
                key=lambda item: item[1],
                default=(None, 0.0),
            )

            if best_index is not None and best_iou >= MATCH_IOU:
                matched_gt.add(best_index)
                counts[class_id]["TP"] += 1
            else:
                counts[class_id]["FP"] += 1

        for index, gt in enumerate(gt_boxes):
            if index not in matched_gt:
                counts[gt["class_id"]]["FN"] += 1

    per_class = {}

    for class_id, values in counts.items():
        metrics = calculate_metrics(
            values["TP"], values["FP"], values["FN"]
        )

        expected_references = sum(
            box["class_id"] == class_id
            for reference in ground_truth.values()
            for box in reference["boxes"]
        )
        expected_predictions = sum(
            prediction["class_id"] == class_id
            and prediction["confidence"] >= confidence
            for image_predictions in predictions.values()
            for prediction in image_predictions
        )

        assert metrics["reference_boxes"] == expected_references
        assert metrics["predicted_boxes"] == expected_predictions
        per_class[CLASS_NAMES[class_id]] = metrics

    totals = {
        key: sum(values[key] for values in counts.values())
        for key in ("TP", "FP", "FN")
    }

    return {
        "images": len(ground_truth),
        "per_class": per_class,
        "micro": calculate_metrics(
            totals["TP"], totals["FP"], totals["FN"]
        ),
        "d40_fp_per_image": counts[3]["FP"] / len(ground_truth),
        "background_images": background_images,
        "background_images_with_detections": (
            background_images_with_detections
        ),
        "background_image_fp_rate": (
            background_images_with_detections / background_images
            if background_images else None
        ),
    }


sweep_evaluations = []
comparison_rows = []

# Read the saved files so the comparison uses recorded predictions.
for prediction_file in sweep_output_paths:
    cached = json.loads(prediction_file.read_text(encoding="utf-8"))
    nms_iou = cached["settings"]["iou"]

    assert cached["settings"]["conf"] == min(CONFIDENCE_GRID)
    assert cached["checkpoint_sha256"] == sweep_checkpoint_hash
    assert cached["general_manifest_sha256"] == file_sha256(manifest_path)
    assert cached["d40_manifest_sha256"] == file_sha256(d40_manifest_path)

    for confidence in CONFIDENCE_GRID:
        for cohort_name, ground_truth in cohort_ground_truth.items():
            metrics = evaluate_operating_point(
                cached["predictions"][cohort_name],
                ground_truth,
                confidence,
            )

            sweep_evaluations.append({
                "cohort": cohort_name,
                "confidence": confidence,
                "nms_iou": nms_iou,
                "matching_iou": MATCH_IOU,
                "prediction_file": prediction_file.name,
                "prediction_file_sha256": file_sha256(prediction_file),
                **metrics,
            })

            micro = metrics["micro"]
            d40 = metrics["per_class"]["D40"]

            comparison_rows.append({
                "cohort": cohort_name,
                "conf": confidence,
                "nms_iou": nms_iou,
                "micro_precision": micro["precision"],
                "micro_recall": micro["recall"],
                "micro_f1": micro["f1"],
                "d40_tp": d40["TP"],
                "d40_fp": d40["FP"],
                "d40_fn": d40["FN"],
                "d40_precision": d40["precision"],
                "d40_recall": d40["recall"],
                "d40_f1": d40["f1"],
                "d40_fp_per_image": metrics["d40_fp_per_image"],
                "background_image_fp_rate": (
                    metrics["background_image_fp_rate"]
                ),
            })

(SWEEP_DIR / "operating_point_metrics.json").write_text(
    json.dumps(sweep_evaluations, indent=2, allow_nan=False),
    encoding="utf-8",
)

comparison_path = SWEEP_DIR / "operating_point_comparison.csv"

with comparison_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(comparison_rows[0]))
    writer.writeheader()
    writer.writerows(comparison_rows)


def metric_text(value):
    return "N/A" if value is None else f"{value:.3f}"


for cohort_name in cohort_ground_truth:
    print(f"\nCOHORT: {cohort_name}")
    print(
        " Conf  NMS | Micro-P Micro-R Micro-F1 | "
        "D40-TP D40-FP D40-FN | D40-P D40-R D40-F1 | D40-FP/img BG-FP-rate"
    )

    rows = sorted(
        (
            row for row in comparison_rows
            if row["cohort"] == cohort_name
        ),
        key=lambda row: (row["nms_iou"], row["conf"]),
    )

    for row in rows:
        print(
            f"{row['conf']:5.2f} {row['nms_iou']:4.2f} | "
            f"{metric_text(row['micro_precision']):>7} "
            f"{metric_text(row['micro_recall']):>7} "
            f"{metric_text(row['micro_f1']):>8} | "
            f"{row['d40_tp']:>6} {row['d40_fp']:>6} {row['d40_fn']:>6} | "
            f"{metric_text(row['d40_precision']):>5} "
            f"{metric_text(row['d40_recall']):>5} "
            f"{metric_text(row['d40_f1']):>6} | "
            f"{row['d40_fp_per_image']:>10.3f} "
            f"{metric_text(row['background_image_fp_rate']):>10}"
        )

print("\nSaved:", comparison_path)
print("PASS: Development operating points compared.")
print("BG-FP-rate: fraction of reference-background images with any detection.")
print("No new inference performed. No final threshold selected.")

## 21. Record the Provisional Integration Operating Point

Use confidence 0.10 and NMS IoU 0.50 as the provisional unified M4
integration setting. This combination achieved the highest general-cohort
micro-F1 among the evaluated grid points.

This is a post-hoc development choice, not a predeclared acceptance
criterion or a final test result.

On the D40-enriched cohort, this setting detected 26 of 122 reference
potholes, with 21 false-positive D40 boxes. A detector-gated segmentation
pipeline inherits missed detections from M4.

Confidence 0.05 with NMS IoU 0.50 achieved higher D40 F1 on the enriched
cohort, but produced more D40 false positives in the general cohort.
Retain both findings in the report. No production-readiness claim is made.

In [ ]:
M4_INTEGRATION_SETTINGS = {
    "imgsz": 640,
    "conf": 0.10,
    "iou": 0.50,
    "max_det": 300,
}

selected_results = {}

for cohort_name in ("general", "d40_diagnostic"):
    matches = [
        result for result in sweep_evaluations
        if result["cohort"] == cohort_name
        and abs(
            result["confidence"] - M4_INTEGRATION_SETTINGS["conf"]
        ) < 1e-9
        and abs(
            result["nms_iou"] - M4_INTEGRATION_SETTINGS["iou"]
        ) < 1e-9
    ]

    assert len(matches) == 1
    selected_results[cohort_name] = matches[0]

general_results = [
    result for result in sweep_evaluations
    if result["cohort"] == "general"
]

best_general_f1 = max(
    result["micro"]["f1"]
    for result in general_results
    if result["micro"]["f1"] is not None
)

assert abs(
    selected_results["general"]["micro"]["f1"] - best_general_f1
) < 1e-12

operating_point_record = {
    "status": "provisional_development_choice",
    "selection_timing": "post_hoc_after_development_sweep",
    "selection_basis": (
        "Highest general-cohort micro-F1 among evaluated grid points"
    ),
    "m4_settings": M4_INTEGRATION_SETTINGS,
    "m5_settings": dict(M5_SETTINGS),
    "matching_iou": MATCH_IOU,
    "checkpoint_sha256": sweep_checkpoint_hash,
    "comparison_file_sha256": file_sha256(comparison_path),
    "selected_development_results": selected_results,
    "limitations": [
        "Small development cohorts; no independent performance estimate.",
        "D40-enriched cohort does not represent deployment prevalence.",
        "Detector-gated segmentation cannot recover undetected potholes.",
        "M5 mask quality on M4 images remains unquantified.",
        "No final acceptance or production-readiness decision.",
    ],
}

operating_point_path = RUN_DIR / "provisional_operating_point.json"
operating_point_path.write_text(
    json.dumps(operating_point_record, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("M4 integration settings:", M4_INTEGRATION_SETTINGS)

for cohort_name, result in selected_results.items():
    d40 = result["per_class"]["D40"]
    print(
        f"{cohort_name}: "
        f"Micro-F1={result['micro']['f1']:.3f}, "
        f"D40 TP={d40['TP']}, FP={d40['FP']}, FN={d40['FN']}"
    )

print("Saved:", operating_point_path)
print("PASS: Provisional integration operating point recorded.")
print("No inference performed. No final acceptance decision made.")

## 22. Detector-Gated Segmentation Smoke Test

Select the first diagnostic image, sorted by path, with a cached D40
prediction at the provisional operating point.

Run M5 on each predicted D40 crop using its documented square-resize
preprocessing. Map the predicted masks back to original-image coordinates.

Report mask pixels, crop-area percentage, and full-image union percentage.
These are predicted image areas, not physical area, depth, or severity.

Cropping introduces an input-distribution change for M5. This experiment
checks execution and coordinate mapping, not segmentation accuracy.
The selected example is conditional on M4 detecting D40.

In [ ]:
import numpy as np
from PIL import Image
from IPython.display import display

# Use recorded M4 predictions at the selected operating point.
cached_predictions = sweep_predictions[
    f"{M4_INTEGRATION_SETTINGS['iou']:.2f}"
]["d40_diagnostic"]

integration_candidates = []

for image_key in sorted(cached_predictions):
    detections = [
        prediction for prediction in cached_predictions[image_key]
        if prediction["class_id"] == POTHOLE_M4_CLASS_ID
        and prediction["confidence"] >= M4_INTEGRATION_SETTINGS["conf"]
    ]

    if detections:
        integration_candidates.append((image_key, detections))

assert integration_candidates, (
    "No D40 detections available at the selected operating point."
)

INTEGRATION_IMAGE_KEY, integration_detections = integration_candidates[0]

selected_record = next(
    record for record in d40_dev_records
    if record["image"] == INTEGRATION_IMAGE_KEY
)

integration_image_path = M4_DIR / INTEGRATION_IMAGE_KEY
assert file_sha256(integration_image_path) == selected_record["image_sha256"]

integration_bgr = cv2.imread(str(integration_image_path))
assert integration_bgr is not None

image_height, image_width = integration_bgr.shape[:2]
integration_union_mask = np.zeros(
    (image_height, image_width), dtype=bool
)
integration_crop_records = []

for detection_index, detection in enumerate(integration_detections):
    x1, y1, x2, y2 = detection["bbox_xyxy"]

    # Integer crop bounds: exclusive right and bottom edges.
    left = max(0, int(np.floor(x1)))
    top = max(0, int(np.floor(y1)))
    right = min(image_width, int(np.ceil(x2)))
    bottom = min(image_height, int(np.ceil(y2)))

    assert right > left and bottom > top

    crop_bgr = integration_bgr[top:bottom, left:right].copy()
    crop_height, crop_width = crop_bgr.shape[:2]

    size = int(M5_SETTINGS["imgsz"])
    model_input = cv2.resize(
        crop_bgr, (size, size), interpolation=cv2.INTER_AREA
    )

    segmentation_result = m5_model.predict(
        source=model_input,
        device=DEVICE,
        **M5_SETTINGS,
        retina_masks=True,
        augment=False,
        save=False,
        verbose=False,
    )[0]

    assert tuple(segmentation_result.orig_shape) == (size, size)

    crop_union = np.zeros((crop_height, crop_width), dtype=bool)
    instance_count = 0

    box_count = (
        len(segmentation_result.boxes)
        if segmentation_result.boxes is not None else 0
    )

    if segmentation_result.masks is None:
        assert box_count == 0, "M5 returned boxes without masks."
    else:
        masks = segmentation_result.masks.data.cpu().numpy()
        classes = segmentation_result.boxes.cls.cpu().numpy()

        assert len(masks) == len(classes)
        assert np.isfinite(masks).all()

        for mask, class_value in zip(masks, classes):
            assert float(class_value) == POTHOLE_M5_CLASS_ID

            restored_mask = cv2.resize(
                (mask > 0.5).astype(np.uint8),
                (crop_width, crop_height),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)

            crop_union |= restored_mask
            instance_count += 1

    # Union avoids double-counting pixels in overlapping detection crops.
    integration_union_mask[top:bottom, left:right] |= crop_union

    mask_pixels = int(crop_union.sum())

    integration_crop_records.append({
        "detection_index": detection_index,
        "m4_confidence": detection["confidence"],
        "m4_bbox_xyxy": detection["bbox_xyxy"],
        "crop_bounds_xyxy": [left, top, right, bottom],
        "crop_width": crop_width,
        "crop_height": crop_height,
        "m5_instances": instance_count,
        "mask_pixels": mask_pixels,
        "predicted_mask_percent_of_crop": (
            100.0 * mask_pixels / (crop_width * crop_height)
        ),
        "status": (
            "mask_predicted" if mask_pixels else "no_mask_predicted"
        ),
    })

union_pixels = int(integration_union_mask.sum())
union_percent = 100.0 * union_pixels / (image_width * image_height)

overlay = integration_bgr.copy()
green = np.array([0, 255, 0], dtype=np.float32)

overlay[integration_union_mask] = (
    0.6 * overlay[integration_union_mask].astype(np.float32)
    + 0.4 * green
).astype(np.uint8)

for record in integration_crop_records:
    left, top, right, bottom = record["crop_bounds_xyxy"]
    cv2.rectangle(
        overlay, (left, top), (right - 1, bottom - 1),
        (0, 255, 255), 2,
    )

integration_report = {
    "purpose": "detector_gated_segmentation_smoke_test",
    "selection": "first sorted diagnostic image with a D40 detection",
    "image": INTEGRATION_IMAGE_KEY,
    "image_sha256": selected_record["image_sha256"],
    "m4_settings": dict(M4_INTEGRATION_SETTINGS),
    "m5_settings": dict(M5_SETTINGS),
    "m4_predictions_source": "cached_development_sweep",
    "m5_checkpoint_sha256": file_sha256(M5_CHECKPOINT),
    "image_width": image_width,
    "image_height": image_height,
    "crops": integration_crop_records,
    "union_mask_pixels": union_pixels,
    "predicted_union_percent_of_full_image": union_percent,
    "segmentation_accuracy_evaluated": False,
}

integration_report_path = RUN_DIR / "integration_smoke_report.json"
integration_report_path.write_text(
    json.dumps(integration_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

assert cv2.imwrite(
    str(RUN_DIR / "integration_smoke_mask.png"),
    integration_union_mask.astype(np.uint8) * 255,
)
assert cv2.imwrite(
    str(RUN_DIR / "integration_smoke_overlay.png"), overlay
)

print("Image:", integration_image_path.name)
print("M4 D40 detections:", len(integration_detections))

for record in integration_crop_records:
    print(
        f"Crop {record['detection_index']}: "
        f"M5 instances={record['m5_instances']}, "
        f"mask pixels={record['mask_pixels']}, "
        f"crop area={record['predicted_mask_percent_of_crop']:.3f}%, "
        f"status={record['status']}"
    )

print("Union mask pixels:", union_pixels)
print(f"Predicted union / full image: {union_percent:.3f}%")

display(Image.fromarray(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)))

print("Saved:", integration_report_path)
print("PASS: Detector-gated segmentation smoke test completed.")
print("Yellow: M4 crop bounds. Green: M5 predicted mask.")
print("No physical area, depth, severity, or accuracy inferred.")

## 23. Inspect the Integration Example Against Reference Boxes

Compare the selected M4 D40 predictions with reference D40 boxes using
one-to-one matching at IoU >= 0.50. Display reference boxes and predicted
boxes alongside the segmentation overlay.

This checks detection agreement for one development example.
Reference boxes cannot establish segmentation accuracy.
Multiple predicted masks do not necessarily represent distinct potholes.

In [ ]:
from PIL import ImageDraw

reference_d40 = [
    box for box in selected_record["boxes"]
    if box["class_id"] == POTHOLE_M4_CLASS_ID
]

matched_reference = set()
example_matches = []

for detection in sorted(
    integration_detections,
    key=lambda item: item["confidence"],
    reverse=True,
):
    candidates = [
        (
            index,
            box_iou(detection["bbox_xyxy"], reference["bbox_xyxy"]),
        )
        for index, reference in enumerate(reference_d40)
        if index not in matched_reference
    ]

    best_index, best_iou = max(
        candidates,
        key=lambda item: item[1],
        default=(None, 0.0),
    )

    is_match = best_index is not None and best_iou >= MATCH_IOU

    if is_match:
        matched_reference.add(best_index)

    example_matches.append({
        "confidence": detection["confidence"],
        "bbox_xyxy": detection["bbox_xyxy"],
        "best_available_reference_iou": best_iou,
        "outcome": "TP" if is_match else "FP",
        "matched_reference_index": best_index if is_match else None,
    })

# Original image with boxes, without the segmentation overlay.
reference_preview = Image.fromarray(
    cv2.cvtColor(integration_bgr, cv2.COLOR_BGR2RGB)
)
draw = ImageDraw.Draw(reference_preview)

for reference in reference_d40:
    draw.rectangle(reference["bbox_xyxy"], outline="cyan", width=2)

for detection in integration_detections:
    draw.rectangle(detection["bbox_xyxy"], outline="yellow", width=2)

print("Cyan: reference D40 boxes. Yellow: predicted D40 boxes.")
display(reference_preview)

for index, record in enumerate(integration_crop_records):
    left, top, right, bottom = record["crop_bounds_xyxy"]

    original_crop = Image.fromarray(
        cv2.cvtColor(
            integration_bgr[top:bottom, left:right],
            cv2.COLOR_BGR2RGB,
        )
    )
    overlay_crop = Image.fromarray(
        cv2.cvtColor(
            overlay[top:bottom, left:right],
            cv2.COLOR_BGR2RGB,
        )
    )

    # Nearest-neighbor enlargement for inspection only.
    preview_size = (original_crop.width * 4, original_crop.height * 4)

    print(f"Crop {index}: original, then predicted-mask overlay")
    display(original_crop.resize(preview_size, Image.Resampling.NEAREST))
    display(overlay_crop.resize(preview_size, Image.Resampling.NEAREST))

example_review = {
    "image": INTEGRATION_IMAGE_KEY,
    "matching_iou": MATCH_IOU,
    "reference_d40_boxes": len(reference_d40),
    "matched_reference_boxes": len(matched_reference),
    "missed_reference_boxes": len(reference_d40) - len(matched_reference),
    "prediction_matches": example_matches,
    "segmentation_accuracy_evaluated": False,
}

(RUN_DIR / "integration_example_detection_review.json").write_text(
    json.dumps(example_review, indent=2, allow_nan=False),
    encoding="utf-8",
)

print(json.dumps(example_review, indent=2))
print("PASS: Example detection matching completed.")
print("No new model inference performed.")

## 24. Define the Integrated Inference Function

Accept a decoded uint8 BGR image. Run M4 on the full image, retain all
damage detections, and run M5 on each predicted D40 crop.

Return a JSON-compatible report, an annotated image, and a full-image
union mask. Distinguish no D40 detection from no mask prediction.
Neither outcome proves the absence of potholes.

Mask areas are pixel counts and image-area percentages. Physical area,
depth, and severity are not estimated. Reference labels are not inputs.

In [ ]:
def run_integrated_inference(
    image_bgr,
    detector,
    segmenter,
    detector_settings,
    segmenter_settings,
    device="cpu",
):
    # Validate the input before calling either model.
    if not isinstance(image_bgr, np.ndarray):
        raise TypeError("Input must be a NumPy BGR image.")

    if (
        image_bgr.dtype != np.uint8
        or image_bgr.ndim != 3
        or image_bgr.shape[2] != 3
        or min(image_bgr.shape[:2]) == 0
    ):
        raise ValueError("Expected a non-empty uint8 image with 3 BGR channels.")

    if detector.task != "detect" or segmenter.task != "segment":
        raise ValueError("Expected a detection model and a segmentation model.")

    expected_names = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}

    if dict(detector.names) != expected_names:
        raise ValueError("Unexpected M4 class mapping.")

    if dict(segmenter.names) != {0: "pothole"}:
        raise ValueError("Unexpected M5 class mapping.")

    source = np.ascontiguousarray(image_bgr.copy())
    height, width = source.shape[:2]
    detection_config = dict(detector_settings)
    segmentation_config = dict(segmenter_settings)

    size = segmentation_config["imgsz"]
    if not isinstance(size, int) or size <= 0:
        raise ValueError("M5 imgsz must be a positive integer.")

    result = detector.predict(
        source=source,
        device=device,
        **detection_config,
        augment=False,
        save=False,
        verbose=False,
    )[0]

    if tuple(result.orig_shape) != (height, width):
        raise ValueError("Unexpected detector output dimensions.")

    detections = []
    union_mask = np.zeros((height, width), dtype=bool)
    segmentation_calls = 0
    segmentation_limit_hits = 0

    if result.boxes is not None:
        coordinates = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        for box, confidence, class_value in zip(
            coordinates, confidences, classes
        ):
            if not float(class_value).is_integer():
                raise ValueError("Non-integer detector class.")

            class_id = int(class_value)
            confidence = float(confidence)
            box = [float(value) for value in box]

            if (
                class_id not in expected_names
                or not math.isfinite(confidence)
                or not 0 <= confidence <= 1
                or not all(math.isfinite(value) for value in box)
            ):
                raise ValueError("Invalid detector output.")

            x1, y1, x2, y2 = box
            if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
                raise ValueError("Invalid detector box coordinates.")

            detections.append({
                "class_id": class_id,
                "damage_code": expected_names[class_id],
                "confidence": confidence,
                "bbox_xyxy": box,
                "segmentation": {
                    "status": "not_applicable",
                    "mask_pixels": None,
                    "mask_percent_of_crop": None,
                    "mask_percent_of_full_image": None,
                },
                "severity": None,
                "severity_status": "not_estimated",
            })

    detections.sort(key=lambda item: item["confidence"], reverse=True)

    for detection in detections:
        if detection["class_id"] != 3:
            continue

        x1, y1, x2, y2 = detection["bbox_xyxy"]
        left, top = int(np.floor(x1)), int(np.floor(y1))
        right, bottom = int(np.ceil(x2)), int(np.ceil(y2))

        crop = source[top:bottom, left:right].copy()
        crop_height, crop_width = crop.shape[:2]

        model_input = cv2.resize(
            crop, (size, size), interpolation=cv2.INTER_AREA
        )

        seg_result = segmenter.predict(
            source=model_input,
            device=device,
            **segmentation_config,
            retina_masks=True,
            augment=False,
            save=False,
            verbose=False,
        )[0]

        segmentation_calls += 1

        if tuple(seg_result.orig_shape) != (size, size):
            raise ValueError("Unexpected segmentation output dimensions.")

        crop_mask = np.zeros((crop_height, crop_width), dtype=bool)
        instance_count = (
            len(seg_result.boxes) if seg_result.boxes is not None else 0
        )

        limit_reached = instance_count >= segmentation_config["max_det"]
        segmentation_limit_hits += int(limit_reached)

        if seg_result.masks is None:
            if instance_count:
                raise ValueError("M5 returned boxes without masks.")
        else:
            if seg_result.boxes is None:
                raise ValueError("M5 returned masks without boxes.")

            masks = seg_result.masks.data.cpu().numpy()
            mask_classes = seg_result.boxes.cls.cpu().numpy()

            if len(masks) != instance_count or not np.isfinite(masks).all():
                raise ValueError("Invalid M5 mask output.")

            for mask, mask_class in zip(masks, mask_classes):
                if float(mask_class) != 0:
                    raise ValueError("Unexpected M5 mask class.")

                restored = cv2.resize(
                    (mask > 0.5).astype(np.uint8),
                    (crop_width, crop_height),
                    interpolation=cv2.INTER_NEAREST,
                ).astype(bool)

                crop_mask |= restored

        union_mask[top:bottom, left:right] |= crop_mask
        pixels = int(crop_mask.sum())

        detection["segmentation"] = {
            "status": "mask_predicted" if pixels else "no_mask_predicted",
            "crop_bounds_xyxy": [left, top, right, bottom],
            "predicted_instances": instance_count,
            "max_det_reached": bool(limit_reached),
            "mask_pixels": pixels,
            "mask_percent_of_crop": (
                100.0 * pixels / (crop_width * crop_height)
            ),
            "mask_percent_of_full_image": 100.0 * pixels / (width * height),
        }

    union_pixels = int(union_mask.sum())

    if segmentation_calls == 0:
        pipeline_status = "no_d40_detected"
    elif union_pixels == 0:
        pipeline_status = "d40_detected_no_mask_predicted"
    else:
        pipeline_status = "d40_detected_with_mask_predictions"

    annotated = source.copy()
    annotated[union_mask] = (
        0.6 * source[union_mask].astype(np.float32)
        + 0.4 * np.array([0, 255, 0], dtype=np.float32)
    ).astype(np.uint8)

    for detection in detections:
        x1, y1, x2, y2 = detection["bbox_xyxy"]
        left, top = int(np.floor(x1)), int(np.floor(y1))
        right, bottom = int(np.ceil(x2)), int(np.ceil(y2))
        color = (0, 255, 255) if detection["class_id"] == 3 else (255, 200, 0)

        cv2.rectangle(
            annotated, (left, top), (right - 1, bottom - 1), color, 2
        )
        cv2.putText(
            annotated,
            f"{detection['damage_code']} {detection['confidence']:.2f}",
            (left, max(12, top - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            color,
            1,
            cv2.LINE_AA,
        )

    report = {
        "schema_version": "1.0",
        "status": pipeline_status,
        "image_width": width,
        "image_height": height,
        "m4_settings": detection_config,
        "m5_settings": segmentation_config,
        "detections": detections,
        "detection_count": len(detections),
        "d40_detection_count": segmentation_calls,
        "segmentation_calls": segmentation_calls,
        "m4_max_det_reached": (
            len(detections) >= detection_config["max_det"]
        ),
        "m5_crops_reaching_max_det": segmentation_limit_hits,
        "union_mask_pixels": union_pixels,
        "union_mask_percent_of_full_image": (
            100.0 * union_pixels / (width * height)
        ),
        "area_units": "pixels_and_image_percent",
        "severity_status": "not_estimated",
    }

    # Ensure the report can be returned later by an HTTP service.
    json.dumps(report, allow_nan=False)

    return report, annotated, union_mask


print("PASS: Integrated inference function defined.")
print("No inference performed by this definition cell.")

In [ ]:
pipeline_report, pipeline_overlay, pipeline_mask = run_integrated_inference(
    image_bgr=integration_bgr,
    detector=m4_model,
    segmenter=m5_model,
    detector_settings=M4_INTEGRATION_SETTINGS,
    segmenter_settings=M5_SETTINGS,
    device=DEVICE,
)

assert pipeline_mask.shape == integration_bgr.shape[:2]
assert pipeline_mask.dtype == bool
assert pipeline_report["union_mask_pixels"] == int(pipeline_mask.sum())

(RUN_DIR / "integrated_function_smoke_report.json").write_text(
    json.dumps(pipeline_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

assert cv2.imwrite(
    str(RUN_DIR / "integrated_function_smoke_overlay.png"),
    pipeline_overlay,
)

print("Status:", pipeline_report["status"])
print("All damage detections:", pipeline_report["detection_count"])
print("D40 detections:", pipeline_report["d40_detection_count"])
print("M5 calls:", pipeline_report["segmentation_calls"])
print("Union mask pixels:", pipeline_report["union_mask_pixels"])
print(
    "Union mask / full image:",
    f"{pipeline_report['union_mask_percent_of_full_image']:.3f}%"
)

display(Image.fromarray(
    cv2.cvtColor(pipeline_overlay, cv2.COLOR_BGR2RGB)
))

print("PASS: Integrated function completed real M4-to-M5 inference.")
print("Segmentation accuracy and severity remain unverified.")

## 25. Test Pipeline Edge Cases

Use controlled model doubles to verify:
- No detections and non-D40 detections skip segmentation.
- A D40 detection with no M5 mask is reported explicitly.
- Overlapping crop masks are unioned without double-counting.
- Invalid inputs are rejected before model inference.

These checks validate pipeline logic, not real-model accuracy.

In [ ]:
from types import SimpleNamespace
import numpy as np


class TestArray:
    """Minimal CPU-array interface used by the pipeline."""
    def __init__(self, values):
        self.values = np.asarray(values)

    def cpu(self):
        return self

    def numpy(self):
        return self.values


class TestBoxes:
    def __init__(self, boxes, classes, confidences):
        self.xyxy = TestArray(np.asarray(boxes).reshape(-1, 4))
        self.cls = TestArray(classes)
        self.conf = TestArray(confidences)

    def __len__(self):
        return len(self.cls.values)


class TestDetector:
    task = "detect"
    names = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}

    def __init__(self, boxes, classes):
        self.boxes = boxes
        self.classes = classes
        self.calls = 0

    def predict(self, source, **kwargs):
        self.calls += 1
        return [SimpleNamespace(
            orig_shape=source.shape[:2],
            boxes=TestBoxes(
                self.boxes,
                self.classes,
                [0.9] * len(self.classes),
            ),
        )]


class TestSegmenter:
    task = "segment"
    names = {0: "pothole"}

    def __init__(self, mode):
        self.mode = mode
        self.calls = 0

    def predict(self, source, **kwargs):
        self.calls += 1

        if self.mode == "must_not_run":
            raise AssertionError("Segmentation should have been skipped.")

        height, width = source.shape[:2]

        if self.mode == "empty":
            return [SimpleNamespace(
                orig_shape=(height, width),
                boxes=TestBoxes([], [], []),
                masks=None,
            )]

        if self.mode == "full_mask":
            return [SimpleNamespace(
                orig_shape=(height, width),
                boxes=TestBoxes(
                    [[0, 0, width, height]], [0], [0.9]
                ),
                masks=SimpleNamespace(
                    data=TestArray(
                        np.ones((1, height, width), dtype=np.float32)
                    )
                ),
            )]

        raise ValueError(f"Unknown test mode: {self.mode}")


edge_test_results = []


def record_pass(name):
    edge_test_results.append({"test": name, "status": "PASS"})
    print("PASS:", name)


def run_test_pipeline(image, detector, segmenter):
    return run_integrated_inference(
        image_bgr=image,
        detector=detector,
        segmenter=segmenter,
        detector_settings=M4_INTEGRATION_SETTINGS,
        segmenter_settings=M5_SETTINGS,
        device="cpu",
    )


test_image = np.full((100, 120, 3), 80, dtype=np.uint8)
original_test_image = test_image.copy()

# 1. No detections: M5 must not run.
detector = TestDetector([], [])
segmenter = TestSegmenter("must_not_run")

report, annotated, mask = run_test_pipeline(
    test_image, detector, segmenter
)

assert report["status"] == "no_d40_detected"
assert report["detection_count"] == 0
assert report["segmentation_calls"] == segmenter.calls == 0
assert not mask.any()
assert np.array_equal(annotated, test_image)
record_pass("No detections skip M5")

# 2. A crack detection is retained without segmentation.
detector = TestDetector([[10, 10, 30, 30]], [1])
segmenter = TestSegmenter("must_not_run")

report, _, mask = run_test_pipeline(
    test_image, detector, segmenter
)

assert report["status"] == "no_d40_detected"
assert report["detection_count"] == 1
assert report["detections"][0]["damage_code"] == "D10"
assert report["detections"][0]["segmentation"]["status"] == "not_applicable"
assert segmenter.calls == 0
assert not mask.any()
record_pass("Non-D40 detection retained; M5 skipped")

# 3. D40 exists, but M5 predicts no mask.
detector = TestDetector([[10, 10, 30, 30]], [3])
segmenter = TestSegmenter("empty")

report, _, mask = run_test_pipeline(
    test_image, detector, segmenter
)

assert report["status"] == "d40_detected_no_mask_predicted"
assert report["d40_detection_count"] == 1
assert report["segmentation_calls"] == segmenter.calls == 1
assert report["detections"][0]["segmentation"]["status"] == "no_mask_predicted"
assert report["union_mask_pixels"] == 0
assert not mask.any()
record_pass("D40 with no mask reported explicitly")

# 4. Two overlapping 20x20 crops: union = 400 + 400 - 100 = 700.
detector = TestDetector(
    [[10, 10, 30, 30], [20, 20, 40, 40]], [3, 3]
)
segmenter = TestSegmenter("full_mask")

report, _, mask = run_test_pipeline(
    test_image, detector, segmenter
)

expected_mask = np.zeros((100, 120), dtype=bool)
expected_mask[10:30, 10:30] = True
expected_mask[20:40, 20:40] = True

assert np.array_equal(mask, expected_mask)
assert report["union_mask_pixels"] == 700
assert report["segmentation_calls"] == segmenter.calls == 2
assert all(
    detection["segmentation"]["mask_pixels"] == 400
    for detection in report["detections"]
)
assert abs(
    report["union_mask_percent_of_full_image"] - 100 * 700 / 12000
) < 1e-12
record_pass("Mask coordinates and overlap union are correct")

# 5. Invalid inputs must fail before either model runs.
invalid_inputs = [
    ("None input", None, TypeError),
    ("Grayscale input", np.zeros((10, 10), dtype=np.uint8), ValueError),
    ("Float input", np.zeros((10, 10, 3), dtype=np.float32), ValueError),
    ("Empty input", np.zeros((0, 10, 3), dtype=np.uint8), ValueError),
    ("Four-channel input", np.zeros((10, 10, 4), dtype=np.uint8), ValueError),
]

for name, invalid_image, expected_error in invalid_inputs:
    detector = TestDetector([], [])
    segmenter = TestSegmenter("must_not_run")

    try:
        run_test_pipeline(invalid_image, detector, segmenter)
    except expected_error:
        pass
    else:
        raise AssertionError(f"Expected {expected_error.__name__}: {name}")

    assert detector.calls == 0
    assert segmenter.calls == 0
    record_pass(f"{name} rejected before inference")

assert np.array_equal(test_image, original_test_image)
record_pass("Input image remains unchanged")

edge_test_path = RUN_DIR / "pipeline_edge_case_tests.json"
edge_test_path.write_text(
    json.dumps({
        "test_type": "controlled_model_doubles",
        "real_models_used": False,
        "tests": edge_test_results,
        "passed": len(edge_test_results),
    }, indent=2),
    encoding="utf-8",
)

print(f"\nCompleted: {len(edge_test_results)} checks passed.")
print("Saved:", edge_test_path)
print("These checks validate pipeline logic, not model accuracy.")

## 26. Create and Verify the Standalone Pipeline Module

Extract the integrated inference function from the saved notebook into
a standalone Python module. Save model paths relative to the project
root, checkpoint fingerprints, and inference settings in a JSON config.

Import the generated module and compare its real-model output with the
previous notebook result on the same image. This checks extraction
consistency, not accuracy or clean-environment reproducibility.

In [ ]:
import ast
import importlib.util
import os
from pathlib import Path

NOTEBOOK_PATH = M8_DIR / "M8.ipynb"
saved_notebook = json.loads(
    NOTEBOOK_PATH.read_text(encoding="utf-8")
)

function_sources = []

for cell in saved_notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source_text = "".join(cell.get("source", []))

    if "def run_integrated_inference(" not in source_text:
        continue

    parsed = ast.parse(source_text)

    for node in parsed.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "run_integrated_inference"
        ):
            function_sources.append(
                ast.get_source_segment(source_text, node)
            )

assert len(function_sources) == 1, (
    "Expected one saved run_integrated_inference definition. "
    "Save the notebook and check for duplicate definitions."
)

module_source = (
    '"""M8 integrated road-damage inference pipeline."""\n\n'
    "import json\n"
    "import math\n"
    "import cv2\n"
    "import numpy as np\n\n"
    + function_sources[0]
    + "\n"
)

# Syntax-check before writing.
compile(module_source, "pipeline.py", "exec")

BUNDLE_DIR = RUN_DIR / "service_bundle"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

pipeline_module_path = BUNDLE_DIR / "pipeline.py"


def write_if_identical_or_new(path, content):
    """Protect existing files from accidental replacement."""
    if path.exists():
        if path.read_text(encoding="utf-8") != content:
            raise FileExistsError(
                f"Existing file differs; inspect before replacing: {path}"
            )
    else:
        path.write_text(content, encoding="utf-8")


write_if_identical_or_new(pipeline_module_path, module_source)

# Verify that the files still match the checkpoints checked earlier.
for model_name, checkpoint in (
    ("M4", M4_CHECKPOINT),
    ("M5", M5_CHECKPOINT),
):
    assert file_sha256(checkpoint) == checkpoint_hashes[model_name]


def project_relative(path):
    return Path(path).resolve().relative_to(
        PROJECT_ROOT.resolve()
    ).as_posix()


service_config = {
    "schema_version": "1.0",
    "status": "development_prototype",
    "device": DEVICE,
    "m4": {
        "checkpoint": project_relative(M4_CHECKPOINT),
        "sha256": file_sha256(M4_CHECKPOINT),
        "settings": dict(M4_INTEGRATION_SETTINGS),
    },
    "m5": {
        "checkpoint": project_relative(M5_CHECKPOINT),
        "sha256": file_sha256(M5_CHECKPOINT),
        "settings": dict(M5_SETTINGS),
    },
    "input": {
        "decoded_format": "uint8_BGR",
    },
    "output": {
        "area_units": "pixels_and_image_percent",
        "severity_status": "not_estimated",
    },
}

service_config_path = BUNDLE_DIR / "config.json"
write_if_identical_or_new(
    service_config_path,
    json.dumps(service_config, indent=2, allow_nan=False) + "\n",
)

# Local launcher hint only. Keep this separate from portable config.
local_launch = {
    "project_root_relative_to_bundle": Path(
        os.path.relpath(PROJECT_ROOT, BUNDLE_DIR)
    ).as_posix()
}
write_if_identical_or_new(
    BUNDLE_DIR / "launch.local.json",
    json.dumps(local_launch, indent=2) + "\n",
)

# Import the standalone module without executing notebook cells.
spec = importlib.util.spec_from_file_location(
    "m8_standalone_pipeline", pipeline_module_path
)
standalone_pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(standalone_pipeline)

standalone_report, standalone_overlay, standalone_mask = (
    standalone_pipeline.run_integrated_inference(
        image_bgr=integration_bgr,
        detector=m4_model,
        segmenter=m5_model,
        detector_settings=M4_INTEGRATION_SETTINGS,
        segmenter_settings=M5_SETTINGS,
        device=DEVICE,
    )
)

# Same models, image, settings and process: require exact agreement.
assert standalone_report == pipeline_report, (
    "Standalone report differs from the notebook result."
)
assert np.array_equal(standalone_mask, pipeline_mask), (
    "Standalone mask differs from the notebook result."
)
assert np.array_equal(standalone_overlay, pipeline_overlay), (
    "Standalone annotated image differs from the notebook result."
)

standalone_verification = {
    "module_sha256": file_sha256(pipeline_module_path),
    "config_sha256": file_sha256(service_config_path),
    "source_notebook_sha256": file_sha256(NOTEBOOK_PATH),
    "image": INTEGRATION_IMAGE_KEY,
    "report_exact_match": True,
    "mask_exact_match": True,
    "overlay_exact_match": True,
    "verification_scope": (
        "Same-process module extraction and real-model output consistency"
    ),
    "clean_environment_tested": False,
}

(RUN_DIR / "standalone_pipeline_verification.json").write_text(
    json.dumps(standalone_verification, indent=2),
    encoding="utf-8",
)

print("Module:", pipeline_module_path)
print("Config:", service_config_path)
print("PASS: Standalone module imported successfully.")
print("PASS: Report, mask, and annotated image match exactly.")
print("Model paths in config are relative to PROJECT_ROOT.")
print("HTTP service has not been created or tested yet.")

## 27. Create the HTTP Inference Service

Create a FastAPI application that verifies checkpoint hashes and loads
both models during startup. Serialize inference with a lock.

Expose GET /health and POST /predict. Accept JPEG/PNG images and return
the pipeline report, annotated PNG, and binary-mask PNG as Base64 strings.

Reject empty, invalid, oversized, and unsupported images. Uploaded images
are not saved by the application. This cell creates the service file;
it does not start or verify the HTTP server.

In [ ]:
app_source = r'''
import base64
import hashlib
import io
import json
import os
import threading
from contextlib import asynccontextmanager
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, UnidentifiedImageError
from fastapi import FastAPI, HTTPException, UploadFile
from ultralytics import YOLO

from pipeline import run_integrated_inference

BUNDLE_DIR = Path(__file__).resolve().parent
MAX_UPLOAD_BYTES = 10 * 1024 * 1024
MAX_IMAGE_PIXELS = 12_000_000
INFERENCE_LOCK = threading.Lock()


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def get_project_root():
    configured = os.environ.get("M8_PROJECT_ROOT")
    if configured:
        return Path(configured).resolve()

    launch = json.loads(
        (BUNDLE_DIR / "launch.local.json").read_text(encoding="utf-8")
    )
    return (
        BUNDLE_DIR / launch["project_root_relative_to_bundle"]
    ).resolve()


@asynccontextmanager
async def lifespan(app):
    app.state.ready = False

    config = json.loads(
        (BUNDLE_DIR / "config.json").read_text(encoding="utf-8")
    )
    project_root = get_project_root()
    models = {}

    for name in ("m4", "m5"):
        checkpoint = (project_root / config[name]["checkpoint"]).resolve()

        if not checkpoint.is_relative_to(project_root):
            raise ValueError("Checkpoint path must remain inside project root.")

        if not checkpoint.is_file():
            raise FileNotFoundError(f"Missing {name} checkpoint: {checkpoint}")

        if sha256(checkpoint) != config[name]["sha256"]:
            raise ValueError(f"{name} checkpoint fingerprint mismatch.")

        models[name] = YOLO(str(checkpoint))
        models[name].to(config["device"])

    if models["m4"].task != "detect":
        raise ValueError("M4 must be a detector.")
    if models["m5"].task != "segment":
        raise ValueError("M5 must be a segmenter.")
    if dict(models["m4"].names) != {
        0: "D00", 1: "D10", 2: "D20", 3: "D40"
    }:
        raise ValueError("Unexpected M4 classes.")
    if dict(models["m5"].names) != {0: "pothole"}:
        raise ValueError("Unexpected M5 classes.")

    app.state.config = config
    app.state.models = models
    app.state.ready = True

    try:
        yield
    finally:
        app.state.ready = False
        app.state.models.clear()


app = FastAPI(
    title="M8 Road Damage Inference",
    version="0.1.0",
    lifespan=lifespan,
)


@app.get("/health")
def health():
    if not getattr(app.state, "ready", False):
        raise HTTPException(status_code=503, detail="Models are not ready.")

    config = app.state.config
    return {
        "status": "ready",
        "device": config["device"],
        "m4_sha256": config["m4"]["sha256"],
        "m5_sha256": config["m5"]["sha256"],
        "max_upload_bytes": MAX_UPLOAD_BYTES,
        "max_image_pixels": MAX_IMAGE_PIXELS,
    }


def decode_image(payload):
    try:
        with Image.open(io.BytesIO(payload)) as image:
            if image.format not in {"JPEG", "PNG"}:
                raise HTTPException(
                    status_code=415,
                    detail="Only JPEG and PNG images are supported.",
                )

            width, height = image.size
            if width <= 0 or height <= 0:
                raise HTTPException(status_code=400, detail="Invalid dimensions.")

            if width * height > MAX_IMAGE_PIXELS:
                raise HTTPException(
                    status_code=413, detail="Image exceeds pixel limit."
                )

            if getattr(image, "n_frames", 1) != 1:
                raise HTTPException(
                    status_code=415, detail="Animated images are not supported."
                )

            # Use encoded pixel orientation, consistent with the notebook.
            image.load()
            rgb = np.array(image.convert("RGB"))

    except HTTPException:
        raise
    except Image.DecompressionBombError:
        raise HTTPException(status_code=413, detail="Image exceeds pixel limit.")
    except (UnidentifiedImageError, OSError, ValueError):
        raise HTTPException(
            status_code=400, detail="File is not a readable image."
        )

    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def png_base64(array):
    success, encoded = cv2.imencode(".png", array)
    if not success:
        raise RuntimeError("PNG encoding failed.")
    return base64.b64encode(encoded.tobytes()).decode("ascii")


@app.post("/predict")
def predict(file: UploadFile):
    if not getattr(app.state, "ready", False):
        raise HTTPException(status_code=503, detail="Models are not ready.")

    try:
        payload = file.file.read(MAX_UPLOAD_BYTES + 1)
    finally:
        file.file.close()

    if not payload:
        raise HTTPException(status_code=400, detail="Uploaded file is empty.")

    if len(payload) > MAX_UPLOAD_BYTES:
        raise HTTPException(status_code=413, detail="Upload exceeds 10 MiB.")

    image_bgr = decode_image(payload)
    config = app.state.config

    # Models are shared within this server process.
    with INFERENCE_LOCK:
        report, annotated, mask = run_integrated_inference(
            image_bgr=image_bgr,
            detector=app.state.models["m4"],
            segmenter=app.state.models["m5"],
            detector_settings=config["m4"]["settings"],
            segmenter_settings=config["m5"]["settings"],
            device=config["device"],
        )

    return {
        "report": report,
        "model_fingerprints": {
            "m4": config["m4"]["sha256"],
            "m5": config["m5"]["sha256"],
        },
        "annotated_png_base64": png_base64(annotated),
        "mask_png_base64": png_base64(mask.astype(np.uint8) * 255),
    }
'''

compile(app_source, "app.py", "exec")

app_path = BUNDLE_DIR / "app.py"
write_if_identical_or_new(app_path, app_source)

print("Service file:", app_path)
print("PASS: app.py written and Python syntax checked.")
print("Endpoints: GET /health, POST /predict")
print("Server has not been started. HTTP behavior is not tested yet.")

## 28. Start the Local Service and Verify HTTP Inference

Start Uvicorn in a separate process on localhost using the notebook's
Python interpreter. Verify model fingerprints through /health.

Send the previous integration image as a lossless PNG to /predict.
Compare the returned report, mask, and annotated image with the notebook
result. This verifies HTTP integration in the current environment,
not clean-environment reproducibility or performance.

In [ ]:
import base64
import os
import socket
import subprocess
import sys
import time
import requests

SERVICE_HOST = "127.0.0.1"
SERVICE_PORT = 8765
SERVICE_URL = f"http://{SERVICE_HOST}:{SERVICE_PORT}"
SERVICE_LOG = RUN_DIR / "http_service.log"

# Avoid starting duplicate servers or using an unrelated service.
if (
    "service_process" in globals()
    and service_process.poll() is None
):
    raise RuntimeError(
        "The service started by this notebook is already running. "
        "Do not start another copy."
    )

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_check:
    try:
        port_check.bind((SERVICE_HOST, SERVICE_PORT))
    except OSError as error:
        raise RuntimeError(
            f"Port {SERVICE_PORT} is occupied. No process was stopped."
        ) from error

service_environment = os.environ.copy()
service_environment["M8_PROJECT_ROOT"] = str(PROJECT_ROOT.resolve())

command = [
    sys.executable,
    "-m", "uvicorn",
    "app:app",
    "--host", SERVICE_HOST,
    "--port", str(SERVICE_PORT),
    "--workers", "1",
]

with SERVICE_LOG.open("ab") as log_stream:
    service_process = subprocess.Popen(
        command,
        cwd=str(BUNDLE_DIR),
        env=service_environment,
        stdout=log_stream,
        stderr=subprocess.STDOUT,
        creationflags=(
            subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0
        ),
    )

session = requests.Session()
session.trust_env = False  # Local requests should not use a proxy.

try:
    health_data = None
    deadline = time.monotonic() + 60

    while time.monotonic() < deadline:
        if service_process.poll() is not None:
            raise RuntimeError("Service process exited during startup.")

        try:
            response = session.get(
                f"{SERVICE_URL}/health", timeout=2
            )
            if response.status_code == 200:
                health_data = response.json()
                break
        except requests.RequestException:
            pass

        time.sleep(0.5)

    if health_data is None:
        raise TimeoutError("Service was not ready within 60 seconds.")

    assert health_data["status"] == "ready"
    assert health_data["m4_sha256"] == checkpoint_hashes["M4"]
    assert health_data["m5_sha256"] == checkpoint_hashes["M5"]

    print("PASS: HTTP health check and model fingerprints verified.")

    # PNG preserves the exact input pixels used in the notebook.
    encoded_ok, encoded_image = cv2.imencode(".png", integration_bgr)
    assert encoded_ok

    response = session.post(
        f"{SERVICE_URL}/predict",
        files={
            "file": (
                "integration_check.png",
                encoded_image.tobytes(),
                "image/png",
            )
        },
        timeout=120,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Prediction request failed: HTTP {response.status_code}\n"
            f"{response.text[:1000]}"
        )

    http_result = response.json()

    def decode_returned_png(encoded, mode):
        raw = base64.b64decode(encoded, validate=True)
        image = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), mode)
        assert image is not None, "Unreadable returned PNG."
        return image

    http_overlay = decode_returned_png(
        http_result["annotated_png_base64"], cv2.IMREAD_COLOR
    )
    http_mask = decode_returned_png(
        http_result["mask_png_base64"], cv2.IMREAD_GRAYSCALE
    )

    assert set(np.unique(http_mask)).issubset({0, 255})

    parity_checks = {
        "report_exact_match": http_result["report"] == pipeline_report,
        "mask_exact_match": np.array_equal(
            http_mask, pipeline_mask.astype(np.uint8) * 255
        ),
        "overlay_exact_match": np.array_equal(
            http_overlay, pipeline_overlay
        ),
        "model_fingerprints_match": http_result["model_fingerprints"] == {
            "m4": checkpoint_hashes["M4"],
            "m5": checkpoint_hashes["M5"],
        },
    }

    verification = {
        "service_url": SERVICE_URL,
        "process_id": service_process.pid,
        "interpreter": sys.executable,
        "health": health_data,
        "http_status": response.status_code,
        "image": INTEGRATION_IMAGE_KEY,
        "checks": parity_checks,
        "all_passed": all(parity_checks.values()),
        "scope": "localhost HTTP inference in the current environment",
    }

    (RUN_DIR / "http_inference_verification.json").write_text(
        json.dumps(verification, indent=2),
        encoding="utf-8",
    )
    (RUN_DIR / "http_prediction_report.json").write_text(
        json.dumps(http_result["report"], indent=2),
        encoding="utf-8",
    )

    assert cv2.imwrite(str(RUN_DIR / "http_overlay.png"), http_overlay)
    assert cv2.imwrite(str(RUN_DIR / "http_mask.png"), http_mask)

    for name, passed in parity_checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {name}")

    assert all(parity_checks.values()), (
        "HTTP output differs from the notebook result. "
        "Evidence was saved for investigation."
    )

    print("\nPASS: Real HTTP inference matches the notebook result.")
    print("Service PID:", service_process.pid)
    print("Interactive API:", SERVICE_URL + "/docs")
    print("Log:", SERVICE_LOG)
    print("Service remains running locally for subsequent tests.")

except Exception:
    # Stop only the process created by this cell.
    if service_process.poll() is None:
        service_process.terminate()
        try:
            service_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            service_process.kill()
            service_process.wait(timeout=10)

    print("\nService stopped after an error. Recent log:")
    print(
        SERVICE_LOG.read_text(
            encoding="utf-8", errors="replace"
        )[-5000:]
    )
    raise

finally:
    session.close()

## 29. Verify HTTP Input Validation

Check that the running service rejects missing uploads, empty files,
invalid image bytes, unsupported formats, oversized uploads, and images
exceeding the pixel limit.

Verify the expected HTTP status codes and confirm service health after
each request. These are API validation tests, not model-quality tests.

In [ ]:
import io
import requests
from PIL import Image

assert service_process.poll() is None, "The service is not running."

http_validation_results = []


def make_image_bytes(image, image_format):
    buffer = io.BytesIO()
    image.save(buffer, format=image_format)
    return buffer.getvalue()


def check_http_case(session, name, expected_status, files=None):
    response = session.post(
        f"{SERVICE_URL}/predict",
        files=files,
        timeout=30,
    )

    health_response = session.get(
        f"{SERVICE_URL}/health", timeout=5
    )

    healthy = (
        health_response.status_code == 200
        and health_response.json().get("status") == "ready"
    )

    try:
        response_detail = response.json()
    except ValueError:
        response_detail = response.text[:500]

    passed = response.status_code == expected_status and healthy

    http_validation_results.append({
        "test": name,
        "expected_status": expected_status,
        "actual_status": response.status_code,
        "service_healthy_after_request": healthy,
        "response": response_detail,
        "passed": passed,
    })

    print(
        f"{'PASS' if passed else 'FAIL'}: {name} — "
        f"HTTP {response.status_code}, healthy={healthy}"
    )


with requests.Session() as test_session:
    test_session.trust_env = False

    health_response = test_session.get(
        f"{SERVICE_URL}/health", timeout=5
    )
    health_response.raise_for_status()
    limits = health_response.json()

    check_http_case(
        test_session,
        "Missing file field",
        422,
    )

    check_http_case(
        test_session,
        "Empty file",
        400,
        files={"file": ("empty.png", b"", "image/png")},
    )

    check_http_case(
        test_session,
        "Invalid bytes disguised as PNG",
        400,
        files={
            "file": (
                "invalid.png",
                b"This is not an image.",
                "image/png",
            )
        },
    )

    gif_bytes = make_image_bytes(
        Image.new("RGB", (8, 8), "white"), "GIF"
    )
    check_http_case(
        test_session,
        "Unsupported GIF format",
        415,
        files={"file": ("image.gif", gif_bytes, "image/gif")},
    )

    oversized_bytes = b"x" * (limits["max_upload_bytes"] + 1)
    check_http_case(
        test_session,
        "Upload exceeds byte limit",
        413,
        files={
            "file": (
                "oversized.png",
                oversized_bytes,
                "image/png",
            )
        },
    )
    del oversized_bytes

    # Highly compressible PNG: small upload, too many decoded pixels.
    large_width = 4000
    large_height = limits["max_image_pixels"] // large_width + 1

    with Image.new("RGB", (large_width, large_height), "black") as large_image:
        large_png = make_image_bytes(large_image, "PNG")

    assert large_width * large_height > limits["max_image_pixels"]
    assert len(large_png) < limits["max_upload_bytes"]

    check_http_case(
        test_session,
        "Image exceeds pixel limit",
        413,
        files={"file": ("large.png", large_png, "image/png")},
    )
    del large_png

validation_report = {
    "purpose": "http_input_validation",
    "service_url": SERVICE_URL,
    "tests": http_validation_results,
    "passed": sum(item["passed"] for item in http_validation_results),
    "total": len(http_validation_results),
}

validation_path = RUN_DIR / "http_input_validation.json"
validation_path.write_text(
    json.dumps(validation_report, indent=2),
    encoding="utf-8",
)

print(
    f"\nPassed: {validation_report['passed']}"
    f"/{validation_report['total']}"
)
print("Saved:", validation_path)

assert all(item["passed"] for item in http_validation_results), (
    "At least one HTTP validation test failed. Review the saved report."
)

print("PASS: HTTP input validation checks completed.")
print("Service remains running locally.")

## 30. Export M4 and M5 to ONNX

Export fresh model instances from verified checkpoint copies.
Use FP32, batch size 1, fixed 640x640 input, opset 18, no graph
simplification, and no embedded NMS.

Validate ONNX structure and load each graph with ONNX Runtime CPU.
Record file fingerprints and input/output signatures.

Successful export and loading do not establish numerical parity,
prediction quality, or deployment performance.

In [ ]:
import gc
import shutil
import onnx
import onnxruntime as ort
from ultralytics import YOLO

ONNX_EXPORT_SETTINGS = {
    "format": "onnx",
    "imgsz": 640,
    "batch": 1,
    "dynamic": False,
    "half": False,
    "simplify": False,
    "opset": 18,
    "nms": False,
    "device": "cpu",
}

ONNX_DIR = RUN_DIR / (
    "onnx_export_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
ONNX_DIR.mkdir(parents=True, exist_ok=False)

onnx_paths = {}
onnx_export_records = {}

export_sources = {
    "M4": M4_CHECKPOINT,
    "M5": M5_CHECKPOINT,
}

for model_name, original_checkpoint in export_sources.items():
    print(f"\nExporting {model_name}...", flush=True)

    expected_hash = checkpoint_hashes[model_name]
    assert file_sha256(original_checkpoint) == expected_hash

    model_dir = ONNX_DIR / model_name.lower()
    model_dir.mkdir()

    copied_checkpoint = model_dir / f"{model_name.lower()}.pt"
    shutil.copy2(original_checkpoint, copied_checkpoint)

    assert file_sha256(copied_checkpoint) == expected_hash

    # Separate instance: do not export the live notebook models.
    export_model = YOLO(str(copied_checkpoint))

    exported_path = Path(
        export_model.export(**ONNX_EXPORT_SETTINGS)
    ).resolve()

    assert exported_path.is_file()
    assert exported_path.is_relative_to(ONNX_DIR.resolve())

    # Structural validation.
    onnx.checker.check_model(str(exported_path))

    # Runtime loading validation; no inference yet.
    runtime_session = ort.InferenceSession(
        str(exported_path),
        providers=["CPUExecutionProvider"],
    )

    inputs = [
        {
            "name": item.name,
            "shape": item.shape,
            "type": item.type,
        }
        for item in runtime_session.get_inputs()
    ]
    outputs = [
        {
            "name": item.name,
            "shape": item.shape,
            "type": item.type,
        }
        for item in runtime_session.get_outputs()
    ]

    assert len(inputs) == 1
    assert inputs[0]["shape"] == [1, 3, 640, 640]
    assert inputs[0]["type"] == "tensor(float)"

    expected_output_count = 1 if model_name == "M4" else 2
    assert len(outputs) == expected_output_count

    onnx_paths[model_name] = exported_path
    onnx_export_records[model_name] = {
        "source_checkpoint_sha256": expected_hash,
        "onnx_path": exported_path.relative_to(ONNX_DIR).as_posix(),
        "onnx_sha256": file_sha256(exported_path),
        "size_bytes": exported_path.stat().st_size,
        "export_settings": dict(ONNX_EXPORT_SETTINGS),
        "runtime_providers": runtime_session.get_providers(),
        "inputs": inputs,
        "outputs": outputs,
        "structure_check": "PASS",
        "runtime_load": "PASS",
        "numerical_parity": "NOT_TESTED",
    }

    # Save progress after each model.
    (ONNX_DIR / "export_manifest.json").write_text(
        json.dumps(onnx_export_records, indent=2),
        encoding="utf-8",
    )

    print(f"PASS: {model_name} exported, checked, and loaded.")
    print("Input:", inputs)
    print("Outputs:", outputs)
    print("File:", exported_path)

    del runtime_session, export_model
    gc.collect()

print("\nPASS: Both ONNX models exported and loaded on CPU.")
print("Manifest:", ONNX_DIR / "export_manifest.json")
print("Numerical parity has not been tested yet.")
print("The HTTP service still uses the PyTorch checkpoints.")

## 31. Check Raw ONNX Numerical Parity

Select one general and one D40-diagnostic image per source, sorted by path.
Feed identical FP32 RGB tensors to fresh export-mode PyTorch models and
ONNX Runtime CPU sessions.

For this numerical check, resize each image to 640x640 with INTER_AREA.
This isolates graph agreement; it does not validate the complete M4
preprocessing pipeline or final postprocessed predictions.

Compare every output element using:
abs(ONNX - PyTorch) <= 0.001 + 0.0001 * abs(PyTorch).

Record errors and failures before asserting success. Final box and mask
agreement will be checked separately.

In [ ]:
from ultralytics.nn.modules.head import Detect

PARITY_ATOL = 1e-3
PARITY_RTOL = 1e-4

# Fixed selection independent of predictions.
parity_records = []

for source in DEV_SOURCES:
    for cohort_name, records in (
        ("general", dev_records),
        ("d40_diagnostic", d40_dev_records),
    ):
        candidates = sorted(
            (record for record in records if record["source"] == source),
            key=lambda record: record["image"],
        )
        assert candidates
        parity_records.append({
            "cohort": cohort_name,
            **candidates[0],
        })

assert len({record["image"] for record in parity_records}) == 6

parity_inputs = []

for record in parity_records:
    image_path = M4_DIR / record["image"]
    assert file_sha256(image_path) == record["image_sha256"]

    image_bgr = cv2.imread(str(image_path))
    assert image_bgr is not None

    resized = cv2.resize(
        image_bgr, (640, 640), interpolation=cv2.INTER_AREA
    )
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

    tensor_array = np.ascontiguousarray(
        rgb.transpose(2, 0, 1)[None],
        dtype=np.float32,
    ) / np.float32(255.0)

    parity_inputs.append((record, tensor_array))

raw_parity_results = []

for model_name in ("M4", "M5"):
    copied_checkpoint = (
        ONNX_DIR / model_name.lower() / f"{model_name.lower()}.pt"
    )
    assert file_sha256(copied_checkpoint) == checkpoint_hashes[model_name]
    assert file_sha256(onnx_paths[model_name]) == (
        onnx_export_records[model_name]["onnx_sha256"]
    )

    # Mirror the relevant export-mode settings on a fresh model.
    reference_wrapper = YOLO(str(copied_checkpoint))
    reference_model = reference_wrapper.model.cpu().float().eval()
    reference_model = reference_model.fuse(imgsz=[640, 640])

    for module in reference_model.modules():
        if isinstance(module, Detect):
            module.export = True
            module.format = "onnx"
            module.dynamic = False

    runtime_session = ort.InferenceSession(
        str(onnx_paths[model_name]),
        providers=["CPUExecutionProvider"],
    )
    input_name = runtime_session.get_inputs()[0].name
    output_names = [
        output.name for output in runtime_session.get_outputs()
    ]

    for record, tensor_array in parity_inputs:
        with torch.inference_mode():
            reference_outputs = reference_model(
                torch.from_numpy(tensor_array)
            )

        if isinstance(reference_outputs, torch.Tensor):
            reference_outputs = [reference_outputs]
        else:
            reference_outputs = list(reference_outputs)

        assert all(
            isinstance(output, torch.Tensor)
            for output in reference_outputs
        ), "Unexpected export-mode output structure."

        reference_arrays = [
            output.detach().cpu().numpy()
            for output in reference_outputs
        ]
        runtime_arrays = runtime_session.run(
            None, {input_name: tensor_array}
        )

        assert len(reference_arrays) == len(runtime_arrays)

        for output_name, reference, actual in zip(
            output_names, reference_arrays, runtime_arrays
        ):
            assert reference.shape == actual.shape
            assert np.isfinite(reference).all()
            assert np.isfinite(actual).all()

            difference = np.abs(
                actual.astype(np.float64)
                - reference.astype(np.float64)
            )
            allowed = (
                PARITY_ATOL
                + PARITY_RTOL * np.abs(reference.astype(np.float64))
            )
            failing_elements = int(np.count_nonzero(difference > allowed))
            passed = failing_elements == 0

            raw_parity_results.append({
                "model": model_name,
                "cohort": record["cohort"],
                "image": record["image"],
                "image_sha256": record["image_sha256"],
                "input_tensor_sha256": hashlib.sha256(
                    tensor_array.tobytes()
                ).hexdigest(),
                "output": output_name,
                "shape": list(reference.shape),
                "max_absolute_error": float(difference.max()),
                "mean_absolute_error": float(difference.mean()),
                "failing_elements": failing_elements,
                "total_elements": int(reference.size),
                "passed": passed,
            })

            print(
                f"{'PASS' if passed else 'FAIL'}: "
                f"{model_name} | {Path(record['image']).name} | "
                f"{output_name} | "
                f"max error={difference.max():.6g} | "
                f"outside tolerance={failing_elements}",
                flush=True,
            )

    del runtime_session, reference_model, reference_wrapper
    gc.collect()

raw_parity_report = {
    "purpose": "raw_export_graph_numerical_parity",
    "atol": PARITY_ATOL,
    "rtol": PARITY_RTOL,
    "preprocessing": "640x640 INTER_AREA, BGR-to-RGB, BCHW FP32 / 255",
    "providers": ["CPUExecutionProvider"],
    "models": {
        name: {
            "checkpoint_sha256": checkpoint_hashes[name],
            "onnx_sha256": file_sha256(onnx_paths[name]),
        }
        for name in ("M4", "M5")
    },
    "comparisons": raw_parity_results,
    "all_passed": all(row["passed"] for row in raw_parity_results),
    "postprocessed_parity_tested": False,
}

raw_parity_path = ONNX_DIR / "raw_numerical_parity.json"
raw_parity_path.write_text(
    json.dumps(raw_parity_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("\nSaved:", raw_parity_path)
print(
    "Passed comparisons:",
    sum(row["passed"] for row in raw_parity_results),
    "/", len(raw_parity_results),
)

assert raw_parity_report["all_passed"], (
    "Numerical differences exceeded the recorded tolerance. "
    "Inspect the saved results before continuing."
)

print("PASS: Raw ONNX numerical parity passed on the six selected images.")
print("Final boxes, masks, and full-pipeline parity are not tested yet.")

## 32. Check End-to-End ONNX Prediction Parity

Compare fresh PyTorch and ONNX-backed pipelines on the six numerical
parity images plus the previous successful integration example.

Use rect=False for both backends so M4 receives fixed 640x640 letterboxed
inputs. This explicitly controls preprocessing for the static ONNX graph.
M5 retains its documented square-resize preprocessing.

Require identical detection counts and same-class one-to-one matches,
box-coordinate error <= 0.5 pixels, confidence error <= 0.0001,
matching segmentation statuses, and full-image union-mask IoU >= 0.99.
Two empty union masks count as agreement; one empty mask does not.

These checks measure backend agreement, not prediction accuracy.
The running HTTP service remains on its existing PyTorch configuration.

In [ ]:
BOX_PARITY_ATOL = 0.5
CONF_PARITY_ATOL = 1e-4
MASK_PARITY_MIN_IOU = 0.99

parity_m4_settings = {
    **M4_INTEGRATION_SETTINGS,
    "rect": False,
}
parity_m5_settings = {
    **M5_SETTINGS,
    "rect": False,
}

# Fresh instances; do not change the notebook's live models.
parity_pt_m4 = YOLO(str(M4_CHECKPOINT), task="detect")
parity_pt_m5 = YOLO(str(M5_CHECKPOINT), task="segment")
parity_onnx_m4 = YOLO(str(onnx_paths["M4"]), task="detect")
parity_onnx_m5 = YOLO(str(onnx_paths["M5"]), task="segment")

final_parity_cases = {
    record["image"]: record
    for record in parity_records
}
integration_case = next(
    record for record in d40_dev_records
    if record["image"] == INTEGRATION_IMAGE_KEY
)
final_parity_cases[INTEGRATION_IMAGE_KEY] = integration_case

final_parity_results = []
FINAL_PARITY_DIR = ONNX_DIR / "pipeline_parity"
FINAL_PARITY_DIR.mkdir(exist_ok=True)

for case_index, (image_key, record) in enumerate(
    sorted(final_parity_cases.items()), start=1
):
    image_path = M4_DIR / image_key
    assert file_sha256(image_path) == record["image_sha256"]
    image_bgr = cv2.imread(str(image_path))
    assert image_bgr is not None

    pt_report, pt_overlay, pt_mask = (
        standalone_pipeline.run_integrated_inference(
            image_bgr,
            parity_pt_m4,
            parity_pt_m5,
            parity_m4_settings,
            parity_m5_settings,
            device="cpu",
        )
    )

    onnx_report, onnx_overlay, onnx_mask = (
        standalone_pipeline.run_integrated_inference(
            image_bgr,
            parity_onnx_m4,
            parity_onnx_m5,
            parity_m4_settings,
            parity_m5_settings,
            device="cpu",
        )
    )

    reference_detections = pt_report["detections"]
    candidate_detections = onnx_report["detections"]

    matched_candidates = set()
    detection_comparisons = []
    boxes_pass = len(reference_detections) == len(candidate_detections)

    for reference in reference_detections:
        candidates = [
            (
                index,
                box_iou(
                    reference["bbox_xyxy"],
                    candidate["bbox_xyxy"],
                ),
            )
            for index, candidate in enumerate(candidate_detections)
            if index not in matched_candidates
            and candidate["class_id"] == reference["class_id"]
        ]

        best_index, overlap = max(
            candidates,
            key=lambda item: item[1],
            default=(None, 0.0),
        )

        if best_index is None:
            boxes_pass = False
            detection_comparisons.append({
                "class_id": reference["class_id"],
                "passed": False,
                "reason": "No unmatched same-class ONNX detection",
            })
            continue

        matched_candidates.add(best_index)
        candidate = candidate_detections[best_index]

        coordinate_error = max(
            abs(a - b)
            for a, b in zip(
                reference["bbox_xyxy"], candidate["bbox_xyxy"]
            )
        )
        confidence_error = abs(
            reference["confidence"] - candidate["confidence"]
        )
        segmentation_status_match = (
            reference["segmentation"]["status"]
            == candidate["segmentation"]["status"]
        )

        passed = (
            coordinate_error <= BOX_PARITY_ATOL
            and confidence_error <= CONF_PARITY_ATOL
            and segmentation_status_match
        )
        boxes_pass = boxes_pass and passed

        detection_comparisons.append({
            "class_id": reference["class_id"],
            "box_iou": overlap,
            "max_coordinate_error_pixels": coordinate_error,
            "confidence_error": confidence_error,
            "segmentation_status_match": segmentation_status_match,
            "passed": passed,
        })

    boxes_pass = boxes_pass and (
        len(matched_candidates) == len(candidate_detections)
    )

    assert pt_mask.shape == onnx_mask.shape

    intersection = int(np.logical_and(pt_mask, onnx_mask).sum())
    union = int(np.logical_or(pt_mask, onnx_mask).sum())
    mask_iou = intersection / union if union else 1.0

    status_match = pt_report["status"] == onnx_report["status"]
    calls_match = (
        pt_report["segmentation_calls"]
        == onnx_report["segmentation_calls"]
    )

    passed = (
        boxes_pass
        and status_match
        and calls_match
        and mask_iou >= MASK_PARITY_MIN_IOU
    )

    case_result = {
        "image": image_key,
        "image_sha256": record["image_sha256"],
        "pytorch_detection_count": len(reference_detections),
        "onnx_detection_count": len(candidate_detections),
        "pytorch_mask_pixels": int(pt_mask.sum()),
        "onnx_mask_pixels": int(onnx_mask.sum()),
        "union_mask_iou": mask_iou,
        "both_masks_empty": union == 0,
        "pipeline_status_match": status_match,
        "segmentation_calls_match": calls_match,
        "detection_comparisons": detection_comparisons,
        "passed": passed,
    }
    final_parity_results.append(case_result)

    case_dir = FINAL_PARITY_DIR / f"case_{case_index:02d}"
    case_dir.mkdir(exist_ok=True)

    for backend, report, annotated, mask in (
        ("pytorch", pt_report, pt_overlay, pt_mask),
        ("onnx", onnx_report, onnx_overlay, onnx_mask),
    ):
        (case_dir / f"{backend}_report.json").write_text(
            json.dumps(report, indent=2, allow_nan=False),
            encoding="utf-8",
        )
        assert cv2.imwrite(
            str(case_dir / f"{backend}_overlay.png"), annotated
        )
        assert cv2.imwrite(
            str(case_dir / f"{backend}_mask.png"),
            mask.astype(np.uint8) * 255,
        )

    print(
        f"{'PASS' if passed else 'FAIL'}: {image_path.name} | "
        f"boxes PT/ONNX={len(reference_detections)}/"
        f"{len(candidate_detections)} | "
        f"mask IoU={mask_iou:.6f} | "
        f"pixels PT/ONNX={int(pt_mask.sum())}/{int(onnx_mask.sum())}",
        flush=True,
    )

nonempty_reference_cases = sum(
    row["pytorch_mask_pixels"] > 0
    for row in final_parity_results
)

final_parity_report = {
    "purpose": "end_to_end_backend_agreement",
    "m4_settings": parity_m4_settings,
    "m5_settings": parity_m5_settings,
    "box_coordinate_atol_pixels": BOX_PARITY_ATOL,
    "confidence_atol": CONF_PARITY_ATOL,
    "minimum_union_mask_iou": MASK_PARITY_MIN_IOU,
    "nonempty_reference_mask_cases": nonempty_reference_cases,
    "cases": final_parity_results,
    "all_passed": (
        all(row["passed"] for row in final_parity_results)
        and nonempty_reference_cases > 0
    ),
}

final_parity_path = ONNX_DIR / "pipeline_parity.json"
final_parity_path.write_text(
    json.dumps(final_parity_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("\nNonempty PyTorch mask cases:", nonempty_reference_cases)
print("Saved:", final_parity_path)

assert final_parity_report["all_passed"], (
    "Pipeline parity did not pass all checks. "
    "Review the saved cases before continuing."
)

print("PASS: Pipeline backend agreement passed on the selected images.")
print("Prediction accuracy and deployment performance are separate checks.")

## 33. Expand Segmentation Parity on Fixed Detection Crops

Use every cached D40 detection in the fixed diagnostic cohort at the
selected operating point. Feed identical crops to PyTorch and ONNX M5,
using the same preprocessing and inference settings.

Compare union masks after restoration to crop dimensions at IoU >= 0.99.
Report nonempty and empty cases separately. Record instance counts, but
do not interpret them as verified numbers of potholes.

This isolates M5 backend agreement on detector-generated crops.
It does not measure segmentation accuracy or full-pipeline recall.

In [ ]:
SEGMENTATION_PARITY_MIN_IOU = 0.99
segmentation_parity_results = []

diagnostic_predictions = sweep_predictions[
    f"{M4_INTEGRATION_SETTINGS['iou']:.2f}"
]["d40_diagnostic"]

diagnostic_record_lookup = {
    record["image"]: record for record in d40_dev_records
}


def predict_crop_union(model, crop_bgr):
    height, width = crop_bgr.shape[:2]
    size = int(M5_SETTINGS["imgsz"])

    resized = cv2.resize(
        crop_bgr, (size, size), interpolation=cv2.INTER_AREA
    )

    result = model.predict(
        source=resized,
        device="cpu",
        **parity_m5_settings,
        retina_masks=True,
        augment=False,
        save=False,
        verbose=False,
    )[0]

    assert tuple(result.orig_shape) == (size, size)

    union_mask = np.zeros((height, width), dtype=bool)
    instance_count = len(result.boxes) if result.boxes is not None else 0

    if result.masks is None:
        assert instance_count == 0
    else:
        masks = result.masks.data.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        assert len(masks) == len(classes) == instance_count
        assert np.isfinite(masks).all()

        for mask, class_value in zip(masks, classes):
            assert float(class_value) == 0

            restored = cv2.resize(
                (mask > 0.5).astype(np.uint8),
                (width, height),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)

            union_mask |= restored

    return union_mask, instance_count


SEG_PARITY_DIR = ONNX_DIR / "crop_mask_parity"
SEG_PARITY_DIR.mkdir(exist_ok=True)

for image_key in sorted(diagnostic_predictions):
    d40_predictions = [
        prediction for prediction in diagnostic_predictions[image_key]
        if prediction["class_id"] == 3
        and prediction["confidence"] >= M4_INTEGRATION_SETTINGS["conf"]
    ]

    if not d40_predictions:
        continue

    record = diagnostic_record_lookup[image_key]
    image_path = M4_DIR / image_key
    assert file_sha256(image_path) == record["image_sha256"]

    image_bgr = cv2.imread(str(image_path))
    assert image_bgr is not None
    height, width = image_bgr.shape[:2]

    for detection_index, prediction in enumerate(d40_predictions):
        x1, y1, x2, y2 = prediction["bbox_xyxy"]
        left = max(0, int(np.floor(x1)))
        top = max(0, int(np.floor(y1)))
        right = min(width, int(np.ceil(x2)))
        bottom = min(height, int(np.ceil(y2)))

        assert right > left and bottom > top
        crop = image_bgr[top:bottom, left:right].copy()

        pt_mask, pt_instances = predict_crop_union(parity_pt_m5, crop)
        ort_mask, ort_instances = predict_crop_union(parity_onnx_m5, crop)

        intersection = int(np.logical_and(pt_mask, ort_mask).sum())
        union = int(np.logical_or(pt_mask, ort_mask).sum())
        mask_iou = intersection / union if union else 1.0

        passed = mask_iou >= SEGMENTATION_PARITY_MIN_IOU
        case_number = len(segmentation_parity_results) + 1

        segmentation_parity_results.append({
            "case": case_number,
            "source": record["source"],
            "image": image_key,
            "image_sha256": record["image_sha256"],
            "detection_index": detection_index,
            "crop_bounds_xyxy": [left, top, right, bottom],
            "m4_confidence": prediction["confidence"],
            "pytorch_instances": pt_instances,
            "onnx_instances": ort_instances,
            "pytorch_mask_pixels": int(pt_mask.sum()),
            "onnx_mask_pixels": int(ort_mask.sum()),
            "both_empty": union == 0,
            "mask_iou": mask_iou,
            "passed": passed,
        })

        # Preserve every mask pair for review.
        assert cv2.imwrite(
            str(SEG_PARITY_DIR / f"{case_number:03d}_pytorch.png"),
            pt_mask.astype(np.uint8) * 255,
        )
        assert cv2.imwrite(
            str(SEG_PARITY_DIR / f"{case_number:03d}_onnx.png"),
            ort_mask.astype(np.uint8) * 255,
        )

        if case_number % 10 == 0 or not passed:
            print(
                f"Processed {case_number} crops; "
                f"latest mask IoU={mask_iou:.6f}",
                flush=True,
            )

assert segmentation_parity_results, "No diagnostic D40 crops found."

nonempty_cases = [
    row for row in segmentation_parity_results
    if row["pytorch_mask_pixels"] > 0
]
failed_cases = [
    row for row in segmentation_parity_results
    if not row["passed"]
]

segmentation_parity_report = {
    "purpose": "M5_backend_agreement_on_fixed_M4_crops",
    "minimum_mask_iou": SEGMENTATION_PARITY_MIN_IOU,
    "m4_settings": dict(M4_INTEGRATION_SETTINGS),
    "m5_settings": dict(parity_m5_settings),
    "m5_checkpoint_sha256": file_sha256(M5_CHECKPOINT),
    "m5_onnx_sha256": file_sha256(onnx_paths["M5"]),
    "total_crops": len(segmentation_parity_results),
    "nonempty_pytorch_cases": len(nonempty_cases),
    "both_empty_cases": sum(
        row["both_empty"] for row in segmentation_parity_results
    ),
    "instance_count_difference_cases": sum(
        row["pytorch_instances"] != row["onnx_instances"]
        for row in segmentation_parity_results
    ),
    "failed_cases": len(failed_cases),
    "minimum_observed_mask_iou": min(
        row["mask_iou"] for row in segmentation_parity_results
    ),
    "cases": segmentation_parity_results,
}

segmentation_parity_path = ONNX_DIR / "crop_mask_parity.json"
segmentation_parity_path.write_text(
    json.dumps(segmentation_parity_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

for key in (
    "total_crops",
    "nonempty_pytorch_cases",
    "both_empty_cases",
    "instance_count_difference_cases",
    "failed_cases",
    "minimum_observed_mask_iou",
):
    print(f"{key}: {segmentation_parity_report[key]}")

print("Saved:", segmentation_parity_path)

assert not failed_cases, (
    "Some crop masks failed parity. Inspect the saved cases."
)
assert nonempty_cases, "No nonempty reference masks were tested."

print("PASS: Expanded M5 union-mask parity passed.")
print("This measures backend agreement, not segmentation accuracy.")

## 34. Freeze the HTTP Benchmark Protocol

Benchmark the existing PyTorch HTTP service on CPU with one client,
one server worker, and sequential requests.

Use both fixed development cohorts: 90 general and 90 D40-enriched images.
Perform 10 warm-up requests, followed by three shuffled repetitions of
all 180 images. Pre-encode inputs as PNG outside the timed requests.

Measure client-observed HTTP latency through full response-body receipt.
Exclude subsequent JSON parsing and output validation from that latency.
Report p50/p95/p99, total wall-time throughput, response sizes, and sampled
server-process RSS. Sampled peak RSS is not an exact allocator peak.

Report cohorts separately. The combined workload is an artificial 50/50
mixture, not deployment prevalence. No latency or memory acceptance target
has been specified. Functional acceptance requires all measured requests
to succeed and return structurally valid outputs.

This protocol covers HTTP performance. Per-stage timing and an isolated
PyTorch-versus-ONNX comparison require separate measurements.

In [ ]:
import platform
import random
import psutil

assert service_process.poll() is None, "HTTP service is not running."

benchmark_cases = []

for cohort_name, records in (
    ("general", dev_records),
    ("d40_diagnostic", d40_dev_records),
):
    for record in sorted(records, key=lambda item: item["image"]):
        benchmark_cases.append({
            "cohort": cohort_name,
            "image": record["image"],
            "image_sha256": record["image_sha256"],
        })

assert len(benchmark_cases) == 180
assert len({case["image"] for case in benchmark_cases}) == 180

BENCHMARK_SEED = 42
BENCHMARK_REPEATS = 3
BENCHMARK_WARMUP_REQUESTS = 10

benchmark_rng = random.Random(BENCHMARK_SEED)
benchmark_schedule = []

for repeat in range(1, BENCHMARK_REPEATS + 1):
    order = list(range(len(benchmark_cases)))
    benchmark_rng.shuffle(order)

    benchmark_schedule.extend({
        "repeat": repeat,
        "case_index": index,
    } for index in order)

# Include examples from both cohorts during warm-up.
warmup_indices = []

for cohort_name in ("general", "d40_diagnostic"):
    indices = [
        index for index, case in enumerate(benchmark_cases)
        if case["cohort"] == cohort_name
    ]
    warmup_indices.extend(benchmark_rng.sample(indices, 5))

benchmark_rng.shuffle(warmup_indices)

assert len(warmup_indices) == BENCHMARK_WARMUP_REQUESTS
assert len(benchmark_schedule) == 540

# Record the actual configuration used by the running service.
benchmark_service_config = json.loads(
    service_config_path.read_text(encoding="utf-8")
)
assert benchmark_service_config["device"] == "cpu"

benchmark_protocol = {
    "purpose": "development_HTTP_performance_measurement",
    "backend": "PyTorch",
    "service_url": SERVICE_URL,
    "server_workers": 1,
    "concurrent_requests": 1,
    "seed": BENCHMARK_SEED,
    "warmup_requests": BENCHMARK_WARMUP_REQUESTS,
    "repeats_per_image": BENCHMARK_REPEATS,
    "measured_requests": len(benchmark_schedule),
    "request_timeout_seconds": 120,
    "input_encoding": "PNG prepared before timing",
    "latency_scope": (
        "Client POST start through full response-body receipt; "
        "excludes subsequent JSON parsing and output validation"
    ),
    "throughput_scope": (
        "Completed measured requests divided by total measurement-loop "
        "wall time, including client validation overhead"
    ),
    "latency_percentiles": [50, 95, 99],
    "memory": {
        "metric": "server-process RSS",
        "sampling_interval_seconds": 0.05,
        "limitation": "Sampled peak, not exact peak; excludes client process",
    },
    "acceptance": {
        "required_successful_requests": len(benchmark_schedule),
        "require_structurally_valid_outputs": True,
        "latency_target_ms": None,
        "memory_target_bytes": None,
        "production_readiness_assessed": False,
    },
    "environment": {
        "os": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "logical_cpu_count": psutil.cpu_count(logical=True),
        "physical_cpu_count": psutil.cpu_count(logical=False),
        "total_ram_bytes": psutil.virtual_memory().total,
        "python": platform.python_version(),
    },
    "service_config": benchmark_service_config,
    "artifact_hashes": {
        "app": file_sha256(app_path),
        "pipeline": file_sha256(pipeline_module_path),
        "config": file_sha256(service_config_path),
        "general_manifest": file_sha256(manifest_path),
        "d40_manifest": file_sha256(d40_manifest_path),
    },
    "cases": benchmark_cases,
    "warmup_case_indices": warmup_indices,
    "schedule": benchmark_schedule,
}

BENCHMARK_DIR = RUN_DIR / "http_benchmark"
BENCHMARK_DIR.mkdir(exist_ok=True)

benchmark_protocol_path = BENCHMARK_DIR / "protocol.json"
write_if_identical_or_new(
    benchmark_protocol_path,
    json.dumps(benchmark_protocol, indent=2, allow_nan=False) + "\n",
)

print("Backend:", benchmark_protocol["backend"])
print("Images:", len(benchmark_cases))
print("Warm-up requests:", len(warmup_indices))
print("Measured requests:", len(benchmark_schedule))
print("Concurrency:", benchmark_protocol["concurrent_requests"])
print("Saved:", benchmark_protocol_path)
print("PASS: HTTP benchmark protocol recorded.")
print("No benchmark requests have been sent.")

## 35. Run the Local HTTP Benchmark

Execute the frozen benchmark protocol against the local PyTorch service.
Prepare PNG inputs before timing, perform warm-up requests, and measure
sequential HTTP requests through full response-body receipt.

Validate response contents after each timed request. Sample server-process
RSS separately. Preserve failed requests in the results without retrying.

This measures local development performance, not prediction accuracy
or production readiness.

In [ ]:
import base64
import hashlib
import json
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import psutil
import requests

protocol = json.loads(benchmark_protocol_path.read_text(encoding="utf-8"))

if service_process.poll() is not None:
    raise RuntimeError("The HTTP service is not running.")

benchmark_result_path = BENCHMARK_DIR / "measurements.json"
if benchmark_result_path.exists():
    raise FileExistsError(
        f"Existing benchmark results will not be overwritten: "
        f"{benchmark_result_path}"
    )

# Verify that the service files still match the frozen protocol.
for key, path in {
    "app": app_path,
    "pipeline": pipeline_module_path,
    "config": service_config_path,
}.items():
    actual_hash = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    if actual_hash != protocol["artifact_hashes"][key]:
        raise RuntimeError(f"Artifact changed after protocol creation: {key}")

session = requests.Session()
session.trust_env = False
timeout = protocol["request_timeout_seconds"]

health_response = session.get(f"{SERVICE_URL}/health", timeout=10)
health_response.raise_for_status()
health_before = health_response.json()

expected_fingerprints = {
    "m4": protocol["service_config"]["m4"]["sha256"],
    "m5": protocol["service_config"]["m5"]["sha256"],
}

assert health_before["status"] == "ready"
assert health_before["device"] == "cpu"
assert health_before["m4_sha256"] == expected_fingerprints["m4"]
assert health_before["m5_sha256"] == expected_fingerprints["m5"]

server = psutil.Process(service_process.pid)
server_created = server.create_time()

# Image reading, fingerprint verification, and PNG encoding are untimed.
prepared_inputs = []
for case in protocol["cases"]:
    image_path = M4_DIR / case["image"]
    image_bytes = image_path.read_bytes()

    if hashlib.sha256(image_bytes).hexdigest() != case["image_sha256"]:
        raise RuntimeError(f"Image fingerprint mismatch: {image_path}")

    image = cv2.imdecode(
        np.frombuffer(image_bytes, dtype=np.uint8), cv2.IMREAD_COLOR
    )
    if image is None:
        raise ValueError(f"Cannot decode image: {image_path}")

    ok, encoded = cv2.imencode(".png", image)
    if not ok:
        raise RuntimeError(f"PNG encoding failed: {image_path}")

    prepared_inputs.append({
        "payload": encoded.tobytes(),
        "height": int(image.shape[0]),
        "width": int(image.shape[1]),
    })

print(f"Prepared {len(prepared_inputs)} PNG inputs outside timing.")


def validate_benchmark_response(response, prepared):
    response.raise_for_status()
    body = response.json()
    report = body["report"]

    if body["model_fingerprints"] != expected_fingerprints:
        raise ValueError("Response model fingerprints differ.")

    height, width = prepared["height"], prepared["width"]
    if (report["image_height"], report["image_width"]) != (height, width):
        raise ValueError("Report image dimensions differ.")

    detections = report["detections"]
    if report["detection_count"] != len(detections):
        raise ValueError("Detection count differs from detection list.")

    d40_count = sum(d["class_id"] == 3 for d in detections)
    if report["d40_detection_count"] != d40_count:
        raise ValueError("D40 count mismatch.")
    if report["segmentation_calls"] != d40_count:
        raise ValueError("Segmentation call count mismatch.")

    for detection in detections:
        if detection["class_id"] not in (0, 1, 2, 3):
            raise ValueError("Unexpected class ID.")

    def decode_output(key, flag):
        raw = base64.b64decode(body[key], validate=True)
        decoded = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), flag)
        if decoded is None:
            raise ValueError(f"Invalid encoded image: {key}")
        return decoded

    mask = decode_output("mask_png_base64", cv2.IMREAD_UNCHANGED)
    overlay = decode_output("annotated_png_base64", cv2.IMREAD_COLOR)

    if mask.shape != (height, width):
        raise ValueError("Mask dimensions differ.")
    if overlay.shape != (height, width, 3):
        raise ValueError("Overlay dimensions differ.")
    if not np.isin(mask, [0, 255]).all():
        raise ValueError("Mask is not binary.")
    if int(np.count_nonzero(mask)) != report["union_mask_pixels"]:
        raise ValueError("Mask area differs from report.")

    # Reject non-finite numeric values in the JSON response.
    json.dumps(report, allow_nan=False)
    return report


def send_benchmark_request(case_index):
    prepared = prepared_inputs[case_index]
    return session.post(
        f"{SERVICE_URL}/predict",
        files={"file": ("input.png", prepared["payload"], "image/png")},
        timeout=timeout,
    )


# Warm-up is excluded from all measured request statistics.
for number, case_index in enumerate(protocol["warmup_case_indices"], 1):
    response = send_benchmark_request(case_index)
    try:
        validate_benchmark_response(response, prepared_inputs[case_index])
    finally:
        response.close()
    print(f"Warm-up {number}/{protocol['warmup_requests']} complete")

rss_samples = []
sampling_errors = []
sampling_stop = threading.Event()
sampling_interval = protocol["memory"]["sampling_interval_seconds"]
measurement_origin = time.perf_counter()


def sample_server_rss():
    while not sampling_stop.is_set():
        try:
            if server.create_time() != server_created:
                raise RuntimeError("Server process identity changed.")
            rss_samples.append({
                "elapsed_seconds": time.perf_counter() - measurement_origin,
                "rss_bytes": int(server.memory_info().rss),
            })
        except Exception as exc:
            sampling_errors.append(f"{type(exc).__name__}: {exc}")
            return
        sampling_stop.wait(sampling_interval)


baseline_rss = int(server.memory_info().rss)
sampler = threading.Thread(target=sample_server_rss, daemon=True)
request_measurements = []
interrupted = False
sampler.start()
loop_start = time.perf_counter()

try:
    for number, scheduled in enumerate(protocol["schedule"], 1):
        case_index = scheduled["case_index"]
        case = protocol["cases"][case_index]
        prepared = prepared_inputs[case_index]

        row = {
            "request_number": number,
            "repeat": scheduled["repeat"],
            "case_index": case_index,
            "cohort": case["cohort"],
            "image": case["image"],
            "input_bytes": len(prepared["payload"]),
            "http_status": None,
            "response_bytes": None,
            "latency_ms": None,
            "success": False,
            "error": None,
        }

        response = None
        started = time.perf_counter()

        try:
            response = send_benchmark_request(case_index)
            # requests uses stream=False: the full response body is received.
            row["latency_ms"] = (time.perf_counter() - started) * 1000
            row["http_status"] = response.status_code
            row["response_bytes"] = len(response.content)

            # Validation below is outside the per-request latency timer.
            report = validate_benchmark_response(response, prepared)
            row.update({
                "success": True,
                "detection_count": report["detection_count"],
                "d40_detection_count": report["d40_detection_count"],
                "segmentation_calls": report["segmentation_calls"],
                "union_mask_pixels": report["union_mask_pixels"],
            })

        except Exception as exc:
            # A transport failure has no completed-response latency.
            row["error"] = f"{type(exc).__name__}: {exc}"
        finally:
            row["attempt_and_validation_ms"] = (
                time.perf_counter() - started
            ) * 1000
            if response is not None:
                response.close()
            request_measurements.append(row)

        if number % 30 == 0:
            failures = sum(not r["success"] for r in request_measurements)
            print(
                f"Measured {number}/{protocol['measured_requests']} "
                f"| failed requests: {failures}"
            )

except KeyboardInterrupt:
    interrupted = True
    print("Interrupted: saving partial measurements.")

finally:
    loop_wall_seconds = time.perf_counter() - loop_start
    sampling_stop.set()
    sampler.join(timeout=5)
    session.close()

    successful = sum(r["success"] for r in request_measurements)
    sampled_peak = max(
        [baseline_rss] + [s["rss_bytes"] for s in rss_samples]
    )

    measurements = {
        "purpose": "development_HTTP_performance_measurement",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "protocol_sha256": hashlib.sha256(
            benchmark_protocol_path.read_bytes()
        ).hexdigest(),
        "backend": "PyTorch",
        "server_pid": service_process.pid,
        "server_create_time": server_created,
        "health_before": health_before,
        "interrupted": interrupted,
        "expected_requests": protocol["measured_requests"],
        "attempted_requests": len(request_measurements),
        "successful_requests": successful,
        "failed_requests": len(request_measurements) - successful,
        "measurement_loop_wall_seconds": loop_wall_seconds,
        "successful_requests_per_second": (
            successful / loop_wall_seconds if loop_wall_seconds else None
        ),
        "server_rss_baseline_bytes": baseline_rss,
        "server_rss_sampled_peak_bytes": sampled_peak,
        "rss_sampling_errors": sampling_errors,
        "rss_samples": rss_samples,
        "requests": request_measurements,
    }

    benchmark_result_path.write_text(
        json.dumps(measurements, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

print("\nSuccessful requests:", successful)
print("Failed requests:", measurements["failed_requests"])
print("Measured loop seconds:", round(loop_wall_seconds, 3))
print("Server sampled peak RSS MiB:", round(sampled_peak / 1024**2, 2))
print("Saved:", benchmark_result_path)

complete = (
    not interrupted
    and successful == protocol["measured_requests"]
    and not sampling_errors
)

print(
    "PASS: HTTP benchmark collection completed."
    if complete
    else "INCOMPLETE/FAIL: Inspect saved failures or sampling errors."
)
print("No latency or memory acceptance target was specified.")

## 36. Summarize the HTTP Benchmark

Report successful-request latency percentiles separately for the general
and D40 diagnostic cohorts, and by whether segmentation was invoked.

Overall throughput includes client-side response validation.
Memory is sampled server-process RSS, not an exact peak or total system usage.
Repeated requests are repeated measurements of the same 180 images.

No latency or memory acceptance target was specified.

In [ ]:
import hashlib
import json
import numpy as np

data = json.loads(
    (BENCHMARK_DIR / "measurements.json").read_text(encoding="utf-8")
)
protocol = json.loads(
    benchmark_protocol_path.read_text(encoding="utf-8")
)

actual_protocol_hash = hashlib.sha256(
    benchmark_protocol_path.read_bytes()
).hexdigest()

if data["protocol_sha256"] != actual_protocol_hash:
    raise RuntimeError("Benchmark protocol fingerprint mismatch.")

rows = data["requests"]
successful_rows = [row for row in rows if row["success"]]

if len(rows) != protocol["measured_requests"]:
    raise RuntimeError("Benchmark request count is incomplete.")

if len(successful_rows) != len(rows):
    raise RuntimeError("Failed requests exist; inspect measurements first.")

if data["rss_sampling_errors"]:
    raise RuntimeError("Memory sampling errors exist.")

latencies = np.array(
    [row["latency_ms"] for row in successful_rows], dtype=float
)
if not np.isfinite(latencies).all() or np.any(latencies < 0):
    raise ValueError("Invalid latency measurements.")


def summarize_latency(selected):
    if not selected:
        return {"requests": 0, "unique_images": 0}

    values = np.array([row["latency_ms"] for row in selected], dtype=float)
    p50, p95, p99 = np.percentile(values, [50, 95, 99])

    return {
        "requests": len(selected),
        "unique_images": len({row["image"] for row in selected}),
        "mean_ms": float(values.mean()),
        "p50_ms": float(p50),
        "p95_ms": float(p95),
        "p99_ms": float(p99),
        "min_ms": float(values.min()),
        "max_ms": float(values.max()),
    }


groups = {
    "all_mixed": successful_rows,
    "general": [
        row for row in successful_rows if row["cohort"] == "general"
    ],
    "d40_diagnostic": [
        row for row in successful_rows
        if row["cohort"] == "d40_diagnostic"
    ],
    "M5_skipped": [
        row for row in successful_rows if row["segmentation_calls"] == 0
    ],
    "M5_invoked": [
        row for row in successful_rows if row["segmentation_calls"] > 0
    ],
}

group_summaries = {
    name: summarize_latency(selected)
    for name, selected in groups.items()
}

repeat_summaries = {
    str(repeat): summarize_latency([
        row for row in successful_rows if row["repeat"] == repeat
    ])
    for repeat in sorted({row["repeat"] for row in successful_rows})
}

wall_seconds = data["measurement_loop_wall_seconds"]
baseline_rss = data["server_rss_baseline_bytes"]
peak_rss = data["server_rss_sampled_peak_bytes"]

summary = {
    "purpose": "development_HTTP_benchmark_summary",
    "backend": data["backend"],
    "concurrency": protocol["concurrent_requests"],
    "successful_requests": len(successful_rows),
    "failed_requests": data["failed_requests"],
    "unique_images": len({row["image"] for row in successful_rows}),
    "measurement_loop_wall_seconds": wall_seconds,
    "successful_requests_per_second": len(successful_rows) / wall_seconds,
    "latency_groups": group_summaries,
    "latency_by_repeat": repeat_summaries,
    "memory": {
        "server_baseline_rss_mib": baseline_rss / 1024**2,
        "server_sampled_peak_rss_mib": peak_rss / 1024**2,
        "sampled_peak_above_baseline_mib": (
            peak_rss - baseline_rss
        ) / 1024**2,
        "sample_count": len(data["rss_samples"]),
    },
    "latency_target_ms": None,
    "memory_target_bytes": None,
    "production_readiness_assessed": False,
    "limitations": [
        "Local CPU service with one worker and sequential requests.",
        "PNG preparation and response validation excluded from latency.",
        "Measurement-loop throughput includes client validation overhead.",
        "RSS is sampled for the server process only.",
        "The mixed cohort has an artificial 50/50 composition.",
        "Three repeats per image are not independent image samples.",
        "Backend speed and prediction accuracy are separate evaluations.",
    ],
}

summary_path = BENCHMARK_DIR / "summary.json"
write_if_identical_or_new(
    summary_path,
    json.dumps(summary, indent=2, allow_nan=False) + "\n",
)

print(
    f"{'Group':<20} {'Requests':>8} {'Images':>7}"
    f" {'Mean ms':>10} {'p50 ms':>10} {'p95 ms':>10} {'p99 ms':>10}"
)

for name, result in group_summaries.items():
    if result["requests"] == 0:
        print(f"{name:<20} {'0':>8} {'0':>7}  N/A")
        continue

    print(
        f"{name:<20} {result['requests']:>8} "
        f"{result['unique_images']:>7}"
        f" {result['mean_ms']:>10.2f}"
        f" {result['p50_ms']:>10.2f}"
        f" {result['p95_ms']:>10.2f}"
        f" {result['p99_ms']:>10.2f}"
    )

print("\nPer-repeat latency:")
for repeat, result in repeat_summaries.items():
    print(
        f"Repeat {repeat}: "
        f"p50={result['p50_ms']:.2f} ms, "
        f"p95={result['p95_ms']:.2f} ms"
    )

print(
    "\nSuccessful requests / measurement-loop second:",
    f"{summary['successful_requests_per_second']:.3f}",
)
print("Server baseline RSS MiB:", f"{baseline_rss / 1024**2:.2f}")
print("Server sampled peak RSS MiB:", f"{peak_rss / 1024**2:.2f}")
print("Saved:", summary_path)
print("PASS: HTTP benchmark summary generated.")
print("Performance acceptance was not assessed; no targets were specified.")

## 37. HTTP Benchmark Findings

The local PyTorch HTTP service completed all 540 measured requests
without failures after 10 warm-up requests. The workload contained
180 development images, each requested three times, with concurrency 1.

### Latency

| Group | Requests | Mean (ms) | p50 (ms) | p95 (ms) | p99 (ms) |
|---|---:|---:|---:|---:|---:|
| General development | 270 | 130.84 | 122.27 | 222.07 | 355.50 |
| D40 diagnostic | 270 | 172.55 | 140.83 | 375.85 | 489.23 |
| M5 skipped | 423 | 120.20 | 121.03 | 160.55 | 177.80 |
| M5 invoked | 117 | 265.56 | 246.45 | 446.87 | 496.83 |

Latency includes the HTTP request through full response-body receipt.
Input PNG preparation and subsequent client-side output validation
are excluded from individual request latency.

Requests invoking M5 had higher observed latency. These groups contain
different images and crop counts, so their difference does not isolate
M5 execution time.

### Throughput and Memory

- Successful requests per measurement-loop second: 6.003.
- Throughput includes client-side response validation overhead.
- Server baseline RSS after warm-up: 470.70 MiB.
- Sampled server peak RSS: 525.10 MiB.
- Sampled peak above baseline: 54.40 MiB.

Memory measurements cover the server process only. Sampling may miss
short-lived peaks; these values do not establish a memory leak.

### Interpretation and Limits

All measured requests passed the implemented response checks.
No latency or memory acceptance target was specified, so this is a
performance characterization rather than a performance acceptance pass.

Repeat-level p95 latency varied from 260.84 to 343.42 ms.
The measurements do not establish the cause of this variation.

The combined workload contains equal numbers of general and
D40-enriched images and does not represent deployment prevalence.
Repeated requests are not independent image samples.

These results apply to the local CPU PyTorch service with one worker
and sequential requests. They do not measure concurrent-load capacity,
ONNX service performance, or prediction accuracy.

---------------------
## 38. Compare PyTorch and ONNX Pipeline Latency

Benchmark both backends on the same 180 development images using
identical settings, including rect=False for fixed-size ONNX compatibility.

Preload images, warm up each backend, and run three paired repeats.
Alternate backend execution order to reduce systematic order effects.

Measure the complete pipeline call, including preprocessing, detection,
crop segmentation, mask reconstruction, report creation, and annotation.
Exclude disk reads, HTTP, and output file encoding.

This is a sequential in-process comparison, separate from the HTTP
benchmark. No speed advantage or performance acceptance is assumed.

In [ ]:
import hashlib
import json
import random
import time
from datetime import datetime, timezone

import cv2
import numpy as np

# Reuse the backend instances that passed pipeline parity.
backend_models = {
    "PyTorch": (parity_pt_m4, parity_pt_m5),
    "ONNX": (parity_onnx_m4, parity_onnx_m5),
}

comparison_m4_settings = {**M4_INTEGRATION_SETTINGS, "rect": False}
comparison_m5_settings = {**M5_SETTINGS, "rect": False}

comparison_dir = ONNX_DIR / (
    "backend_benchmark_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
comparison_dir.mkdir(parents=True, exist_ok=False)

# Reuse the same cases as the HTTP benchmark.
http_protocol = json.loads(
    benchmark_protocol_path.read_text(encoding="utf-8")
)
comparison_cases = http_protocol["cases"]

comparison_images = []
for case in comparison_cases:
    image_path = M4_DIR / case["image"]
    raw = image_path.read_bytes()

    if hashlib.sha256(raw).hexdigest() != case["image_sha256"]:
        raise RuntimeError(f"Image fingerprint mismatch: {image_path}")

    image = cv2.imdecode(
        np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR
    )
    if image is None:
        raise ValueError(f"Cannot decode: {image_path}")

    comparison_images.append(image)

assert len(comparison_images) == 180

rng = random.Random(42)
paired_schedule = []

for repeat in range(1, 4):
    indices = list(range(len(comparison_images)))
    rng.shuffle(indices)

    for case_index in indices:
        pair_number = len(paired_schedule)
        order = (
            ["PyTorch", "ONNX"]
            if pair_number % 2 == 0
            else ["ONNX", "PyTorch"]
        )
        paired_schedule.append({
            "repeat": repeat,
            "case_index": case_index,
            "backend_order": order,
        })

comparison_protocol = {
    "purpose": "development_backend_latency_comparison",
    "device": "cpu",
    "images": len(comparison_images),
    "repeats": 3,
    "warmup_calls_per_backend": 10,
    "measured_calls_per_backend": len(paired_schedule),
    "m4_settings": comparison_m4_settings,
    "m5_settings": comparison_m5_settings,
    "timing_scope": "Complete standalone pipeline function call",
    "excluded": ["disk reads", "HTTP", "PNG encoding"],
    "runtime_threads": "Existing backend defaults; not normalized",
    "cases": comparison_cases,
    "schedule": paired_schedule,
}

(comparison_dir / "protocol.json").write_text(
    json.dumps(comparison_protocol, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)


def call_comparison_pipeline(backend, image):
    detector, segmenter = backend_models[backend]
    return standalone_pipeline.run_integrated_inference(
        image,
        detector,
        segmenter,
        comparison_m4_settings,
        comparison_m5_settings,
        device="cpu",
    )


# Identical warm-up images for both backends.
for position, case_index in enumerate(
    http_protocol["warmup_case_indices"], 1
):
    for backend in ("PyTorch", "ONNX"):
        warm_output = call_comparison_pipeline(
            backend, comparison_images[case_index]
        )
        del warm_output

    print(f"Both backends warmed: {position}/10")

backend_rows = []
comparison_interrupted = False

try:
    for pair_number, scheduled in enumerate(paired_schedule, 1):
        case_index = scheduled["case_index"]
        case = comparison_cases[case_index]

        for backend in scheduled["backend_order"]:
            row = {
                "backend": backend,
                "repeat": scheduled["repeat"],
                "case_index": case_index,
                "cohort": case["cohort"],
                "image": case["image"],
                "success": False,
                "latency_ms": None,
                "error": None,
            }

            started = time.perf_counter()
            try:
                output = call_comparison_pipeline(
                    backend, comparison_images[case_index]
                )
                elapsed_ms = (time.perf_counter() - started) * 1000

                report, annotated, mask = output
                row.update({
                    "success": True,
                    "latency_ms": elapsed_ms,
                    "detection_count": report["detection_count"],
                    "segmentation_calls": report["segmentation_calls"],
                    "union_mask_pixels": report["union_mask_pixels"],
                })
                del output, report, annotated, mask

            except Exception as exc:
                row["error"] = f"{type(exc).__name__}: {exc}"

            backend_rows.append(row)

        if pair_number % 30 == 0:
            failures = sum(not row["success"] for row in backend_rows)
            print(
                f"Completed {pair_number}/{len(paired_schedule)} pairs"
                f" | failed calls: {failures}"
            )

except KeyboardInterrupt:
    comparison_interrupted = True
    print("Interrupted: preserving partial measurements.")

finally:
    result_path = comparison_dir / "measurements.json"
    result_path.write_text(
        json.dumps({
            "interrupted": comparison_interrupted,
            "expected_calls": 2 * len(paired_schedule),
            "recorded_calls": len(backend_rows),
            "requests": backend_rows,
        }, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

print("\nSaved:", result_path)

expected_calls = 2 * len(paired_schedule)
complete = (
    not comparison_interrupted
    and len(backend_rows) == expected_calls
    and all(row["success"] for row in backend_rows)
)

if not complete:
    print("INCOMPLETE/FAIL: Inspect saved measurements before comparison.")
else:
    print(
        f"\n{'Cohort':<18} {'Backend':<10} {'Calls':>6}"
        f" {'Mean ms':>10} {'p50 ms':>10} {'p95 ms':>10}"
    )

    for cohort in ("general", "d40_diagnostic"):
        means = {}

        for backend in ("PyTorch", "ONNX"):
            selected = [
                row for row in backend_rows
                if row["cohort"] == cohort and row["backend"] == backend
            ]
            values = np.array(
                [row["latency_ms"] for row in selected], dtype=float
            )
            means[backend] = float(values.mean())
            p50, p95 = np.percentile(values, [50, 95])

            print(
                f"{cohort:<18} {backend:<10} {len(selected):>6}"
                f" {values.mean():>10.2f}"
                f" {p50:>10.2f} {p95:>10.2f}"
            )

        print(
            f"  PyTorch mean / ONNX mean: "
            f"{means['PyTorch'] / means['ONNX']:.3f}x"
            " (>1 means ONNX was faster)"
        )

    print("\nPASS: Paired backend latency measurements completed.")
    print("Runtime thread defaults were not normalized.")
    print("This is not an HTTP or concurrent-load benchmark.")

## 39. Backend Performance Findings and Deployment Decision

Both backends completed 540 measured pipeline calls each without failures.
The comparison used identical images and settings, including rect=False,
with alternating execution order and three repeats per image.

| Cohort | Backend | Mean (ms) | p50 (ms) | p95 (ms) |
|---|---|---:|---:|---:|
| General | PyTorch | 142.52 | 95.46 | 429.46 |
| General | ONNX | 80.97 | 30.87 | 352.54 |
| D40 diagnostic | PyTorch | 216.02 | 129.97 | 494.66 |
| D40 diagnostic | ONNX | 188.08 | 36.85 | 660.28 |

### Findings

The PyTorch-to-ONNX mean latency ratio was 1.760 for the general cohort
and 1.149 for the D40 diagnostic cohort.

ONNX reduced mean and median latency in both cohorts. However, its
D40 diagnostic p95 latency was higher than PyTorch's. These measurements
do not establish the cause of the latency variation.

Runtime thread settings were not normalized, and both backends were
resident in the same notebook process. The results characterize this
configuration rather than an isolated runtime comparison.

The timing run recorded output counts but did not perform full output
parity checks on all 180 images. Earlier parity checks remain separate
evidence with their documented sample coverage.

### Provisional Deployment Decision

Retain PyTorch for the current local HTTP prototype because its HTTP
behavior and performance have been measured directly.

Keep ONNX as a validated export and a candidate for subsequent deployment
testing. Switching the service requires ONNX-specific HTTP verification
and performance measurements.

This decision does not establish that PyTorch is universally faster.
Neither backend has received a production-readiness acceptance decision.
Detection recall and segmentation accuracy remain separate limitations.

----------------------
## 40. Historical Verification Summary — Original Pipeline

> This section records the original pipeline before the changes in
> sections 44 onward. Its severity limitations and remaining-work list
> describe that earlier implementation, not the final schema 1.2 service.
>
> Subsequent sections add the pothole image-area proxy, CPU stage timing,
> updated functional checks, isolated backend benchmarks, model card,
> failure gallery, and verified delivery package.
>
> For the updated system status, consult the model card in section 56,
> the delivery checklist in section 60, and the submission index.
> Earlier measurements retain their original version and experiment scope.
### Verification Results

| Area | Evidence | Result and scope |
|---|---|---|
| Pipeline edge cases | 10 checks passed | Tested skip logic, mask placement, overlap union, invalid inputs, and input preservation |
| Standalone module | Report, mask, and overlay matched notebook outputs | Verified on the selected example |
| HTTP inference | Real HTTP outputs matched notebook outputs | Verified on the selected example |
| HTTP input validation | 6/6 checks passed | Expected error responses returned; service remained healthy |
| ONNX export | Both models exported, structurally checked, and loaded on CPU | Export compatibility verified |
| Raw numerical parity | 18/18 comparisons passed on 6 images | Within the configured tolerances |
| Full-pipeline parity | 7 selected images passed | Included one nonempty predicted union mask |
| Expanded M5 mask parity | 47 crops passed: 37 nonempty, 10 both empty | Minimum union-mask IoU was 1.0 |
| PyTorch HTTP benchmark | 540/540 measured requests succeeded | Local sequential workload; no performance acceptance targets specified |
| Backend timing comparison | 540 calls per backend completed | ONNX improved mean latency, but diagnostic p95 was higher |

These results establish the tested implementation behavior and backend
agreement within the recorded coverage. They do not establish prediction
accuracy, production readiness, or universal backend equivalence.

### Detection Quality

The provisional M4 integration settings are:

- Image size: 640.
- Confidence threshold: 0.10.
- NMS IoU: 0.50.
- Maximum detections: 300.

On the general development subset, micro-F1 was approximately 0.350.
D40 results were TP=3, FP=8, and FN=2, based on only five reference boxes.

On the D40-enriched diagnostic cohort, D40 results were TP=26, FP=21,
and FN=96. D40 recall was approximately 0.213.

These are development operating-point measurements, not AP/mAP or
independent final-test results. Threshold selection used these development
cohorts, so their results must not be presented as unbiased final performance.

### Segmentation and Integration Limits

The pipeline invokes M5 only for M4-predicted D40 regions. Consequently,
a pothole missed by M4 may never reach the segmentation stage.

M5 predictions showed false-positive regions during the independent
full-image smoke test. Detector-gated cropping has been implemented,
but its segmentation accuracy has not been established on these road images.

No reference segmentation masks were used to quantify integrated mask
accuracy. Agreement between PyTorch and ONNX does not establish agreement
with ground truth.

Reported area is predicted mask area in pixels or as a percentage of
the crop/full image. Physical area, depth, and severity are not estimated.

### Deployment Status

The current local HTTP prototype uses PyTorch and provides:

- GET /health
- POST /predict

The prototype passed the recorded functional checks. Its measured
sequential HTTP throughput was approximately 6.003 successful requests
per measurement-loop second, including client-side validation overhead.

No latency, memory, or model-quality acceptance targets were specified
for final deployment acceptance. Therefore, production readiness remains
unassessed.

ONNX remains an export candidate. ONNX-specific HTTP behavior, service
performance, and concurrent-load capacity have not been tested.

### Remaining Work

1. Verify reproducibility from a fresh process using the saved configuration.
2. Finalize dependency records and execution instructions.
3. Document the output schema and known failure examples.
4. Define deployment acceptance criteria before any final evaluation.
5. Perform an independent final evaluation if required by the project scope,
   keeping final-test data separate from further development decisions.

-----------------
## 41. Verify Inference in a Fresh Python Process

Start a new Python process using the current interpreter. Load the standalone
pipeline, configuration, and checkpoints from disk, verify their fingerprints,
and run inference on the saved smoke-test input.

Compare its report, mask, and annotated image with the notebook reference.
This checks independence from notebook variables, using the existing installed
environment. It does not verify installation in a clean environment.

In [ ]:
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone

import cv2
import numpy as np

fresh_dir = RUN_DIR / (
    "fresh_process_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
fresh_dir.mkdir(parents=True, exist_ok=False)

# Lossless input preserves the exact pixels used by the notebook reference.
input_path = fresh_dir / "input.png"
if not cv2.imwrite(str(input_path), integration_bgr):
    raise RuntimeError("Could not save the verification input.")

fresh_script = r'''
import hashlib
import json
import sys
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

bundle = Path(sys.argv[1]).resolve()
project_root = Path(sys.argv[2]).resolve()
input_path = Path(sys.argv[3]).resolve()
output_dir = Path(sys.argv[4]).resolve()

sys.path.insert(0, str(bundle))
from pipeline import run_integrated_inference

config = json.loads((bundle / "config.json").read_text(encoding="utf-8"))

models = {}
verified_hashes = {}

for key in ("m4", "m5"):
    checkpoint = (project_root / config[key]["checkpoint"]).resolve()
    if not checkpoint.is_relative_to(project_root):
        raise ValueError(f"Checkpoint outside project root: {key}")

    actual_hash = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    if actual_hash != config[key]["sha256"]:
        raise ValueError(f"Checkpoint fingerprint mismatch: {key}")

    verified_hashes[key] = actual_hash
    models[key] = YOLO(str(checkpoint)).to(config["device"])

image = cv2.imread(str(input_path), cv2.IMREAD_COLOR)
if image is None:
    raise ValueError("Cannot read verification input.")

report, overlay, mask = run_integrated_inference(
    image,
    models["m4"],
    models["m5"],
    config["m4"]["settings"],
    config["m5"]["settings"],
    device=config["device"],
)

(output_dir / "report.json").write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
np.savez_compressed(output_dir / "outputs.npz", mask=mask, overlay=overlay)
(output_dir / "process_info.json").write_text(
    json.dumps({
        "python_executable": sys.executable,
        "python_version": sys.version,
        "verified_checkpoint_hashes": verified_hashes,
        "pipeline_sha256": hashlib.sha256(
            (bundle / "pipeline.py").read_bytes()
        ).hexdigest(),
        "config_sha256": hashlib.sha256(
            (bundle / "config.json").read_bytes()
        ).hexdigest(),
    }, indent=2) + "\n",
    encoding="utf-8",
)
print("Fresh-process inference completed.")
'''

script_path = fresh_dir / "verify_fresh_process.py"
script_path.write_text(fresh_script, encoding="utf-8")

command = [
    sys.executable,
    str(script_path),
    str(BUNDLE_DIR),
    str(PROJECT_ROOT),
    str(input_path),
    str(fresh_dir),
]

print("Starting fresh Python process...")

try:
    completed = subprocess.run(
        command,
        cwd=str(fresh_dir),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=180,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
except subprocess.TimeoutExpired:
    raise RuntimeError("Fresh-process verification exceeded 180 seconds.")

log_path = fresh_dir / "process.log"
log_path.write_text(
    completed.stdout + "\n" + completed.stderr,
    encoding="utf-8",
)

if completed.returncode != 0:
    print(completed.stderr[-4000:])
    raise RuntimeError(f"Fresh process failed. See: {log_path}")

fresh_report = json.loads(
    (fresh_dir / "report.json").read_text(encoding="utf-8")
)

with np.load(fresh_dir / "outputs.npz", allow_pickle=False) as outputs:
    checks = {
        "report_exact_match": fresh_report == pipeline_report,
        "mask_exact_match": np.array_equal(outputs["mask"], pipeline_mask),
        "overlay_exact_match": np.array_equal(
            outputs["overlay"], pipeline_overlay
        ),
    }

verification = {
    "purpose": "fresh_process_reproducibility",
    "same_installed_environment": True,
    "clean_environment_installation_tested": False,
    "return_code": completed.returncode,
    "checks": checks,
    "passed": all(checks.values()),
}

verification_path = fresh_dir / "verification.json"
verification_path.write_text(
    json.dumps(verification, indent=2) + "\n",
    encoding="utf-8",
)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("Saved:", verification_path)
print("Log:", log_path)

if not verification["passed"]:
    raise AssertionError("Fresh-process outputs differ; inspect before proceeding.")

print("PASS: Standalone inference reproduced in a fresh Python process.")
print("A clean-environment installation has not been tested.")

## 42. Record Dependencies and Execution Instructions

Record installed versions of direct service dependencies and an inventory
of the current Python environment. Create instructions for launching the
local PyTorch service and interpreting its outputs.

These files document the verified environment. They do not establish that
installation will succeed on a different machine or in a clean environment.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import platform
import sys

# Use the current experiment, without a hard-coded run path.
if "RUN_DIR" not in globals():
    raise RuntimeError(
        "RUN_DIR is undefined. Initialize or explicitly resume the intended run."
    )

active_run = Path(RUN_DIR).resolve()
results_root = (Path(M8_DIR) / "results").resolve()

if not active_run.is_relative_to(results_root):
    raise ValueError("RUN_DIR must be inside M8/results.")

BUNDLE_DIR = active_run / "service_bundle"

for filename in ("pipeline.py", "app.py", "config.json", "launch.local.json"):
    path = BUNDLE_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Required service file missing: {path}")

service_packages = [
    "ultralytics",
    "torch",
    "torchvision",
    "numpy",
    "opencv-python",
    "Pillow",
    "fastapi",
    "uvicorn",
    "python-multipart",
]

verification_packages = [
    "onnx",
    "onnxruntime",
    "requests",
    "psutil",
]

service_versions = {
    name: metadata.version(name)
    for name in service_packages
}

verification_versions = {
    name: metadata.version(name)
    for name in verification_packages
}

requirements_service = (
    "# Observed direct service dependencies.\n"
    "# Not a complete lock; clean installation is unverified.\n"
    "# PyTorch wheel availability depends on Python, platform and index.\n"
    + "\n".join(
        f"{name}=={version}"
        for name, version in service_versions.items()
    )
    + "\n"
)

requirements_verification = (
    "-r requirements-service.txt\n"
    "# Additional dependencies for ONNX and verification.\n"
    + "\n".join(
        f"{name}=={version}"
        for name, version in verification_versions.items()
    )
    + "\n"
)

installed_packages = sorted(
    [
        {
            "name": dist.metadata.get("Name") or "UNKNOWN",
            "version": dist.version,
        }
        for dist in metadata.distributions()
    ],
    key=lambda item: (item["name"].lower(), item["version"]),
)

environment = {
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "service_direct_dependencies": service_versions,
    "verification_direct_dependencies": verification_versions,
    "installed_packages": installed_packages,
    "clean_installation_verified": False,
    "note": (
        "Current environment inventory, not a portable dependency lock. "
        "May include unrelated packages."
    ),
}

readme = r"""# M8 Road Damage Inference

Local development prototype using PyTorch on CPU.

## Bundle files

- pipeline.py: M4 detection and detector-gated M5 segmentation.
- app.py: FastAPI service.
- config.json: model paths, fingerprints and inference settings.
- launch.local.json: local project-root location.
- requirements-service.txt: observed direct service dependency versions.
- requirements-verification.txt: additional verification dependencies.
- environment_inventory.json: recorded environment.

Model checkpoints and datasets are not included in this bundle.
Preserve the configured checkpoint paths under the project root.

## Installation

Use the Python version in environment_inventory.json as the reference.
Create and activate a virtual environment, then run from this directory:

    python -m pip install -r requirements-service.txt

For ONNX and verification dependencies:

    python -m pip install -r requirements-verification.txt

Compatible wheels must be available for the selected Python and platform,
particularly for the recorded PyTorch CPU versions.

These requirements record direct dependencies, not a complete environment
lock. Clean installation and portability remain unverified.
Record dependency changes and rerun verification after changes.

## Start the service

Open PowerShell in this bundle directory with the intended environment
activated. Replace the example path with the actual phase2 directory:

    $env:M8_PROJECT_ROOT = "C:\path\to\phase2"
    python -m uvicorn app:app --host 127.0.0.1 --port 8765 --workers 1

The application reads config.json and verifies checkpoint fingerprints
during startup.

If port 8765 is occupied by the notebook server, use that server or stop
its owned process before starting another instance.

When relocating this bundle, set M8_PROJECT_ROOT explicitly.
The fallback launch.local.json path describes the original folder layout.

## Health check

    Invoke-RestMethod "http://127.0.0.1:8765/health"

Interactive API: http://127.0.0.1:8765/docs

## Prediction request

Replace the image path, then run:

    curl.exe -f -X POST "http://127.0.0.1:8765/predict" -F "file=@C:\path\to\image.png" -o prediction.json

Accepted formats: supported single-frame JPEG and PNG.
File limit: 10 MiB. Decoded image limit: 12,000,000 pixels.

The byte check occurs after multipart parsing; it is not a network-level
request-body limit. This service is a local development prototype.

## Response fields

- report: detections, segmentation status and predicted mask area.
- model_fingerprints: checkpoint SHA-256 fingerprints.
- annotated_png_base64: annotated PNG encoded as base64.
- mask_png_base64: binary union-mask PNG encoded as base64.

M4 classes are D00, D10, D20 and D40.
M5 runs only for M4-predicted D40 regions.

Skipped segmentation or an empty mask does not establish that an image
contains no potholes.

Area is reported in pixels and crop/full-image percentages.
In the current implementation, severity is marked not_estimated.
Physical area and depth are not estimated.

## Verification evidence

The parent development-run directory contains verification and benchmark
reports. Their scope, settings and artifact fingerprints determine which
implementation each result supports.

After changing pipeline.py, app.py, models or configuration, rerun affected
checks. Earlier results must not be treated as verification of changed code.

A fresh-process test using the installed environment is distinct from
installation and verification in a clean environment.

The current HTTP service uses PyTorch. ONNX export and backend-comparison
evidence are stored separately in the ONNX export directory.

## Limitations

Detection misses can prevent actual potholes from reaching M5.
Integrated segmentation accuracy on road images has not been measured
against reference masks.

Backend agreement does not establish prediction accuracy.
Development results do not establish independent final-test performance
or production readiness.

Use anonymized images for demonstrations and shared examples.
The current annotation code does not blur faces or license plates.

## Stop the service

For a foreground server, press Ctrl+C in its terminal.
For a notebook-launched server, stop only the process owned by that notebook.
Do not terminate unrelated Python processes.
"""

files_to_save = {
    "requirements-service.txt": requirements_service,
    "requirements-verification.txt": requirements_verification,
    "environment_inventory.json": (
        json.dumps(environment, indent=2, ensure_ascii=False) + "\n"
    ),
    "README.md": readme,
}

# Preserve existing versions before replacing changed documentation.
for filename, content in files_to_save.items():
    target = BUNDLE_DIR / filename

    if target.exists():
        existing = target.read_text(encoding="utf-8")

        if existing == content:
            print("Already current:", target)
            continue

        backup = target.with_name(target.name + ".previous")
        number = 1

        while backup.exists():
            backup = target.with_name(
                target.name + f".previous.{number}"
            )
            number += 1

        backup.write_bytes(target.read_bytes())
        print("Previous version preserved:", backup)

    target.write_text(content, encoding="utf-8")
    print("Saved:", target)

print("\nActive run:", active_run)
print("PASS: Dependency records and README created.")
print("No packages installed. Service code and config unchanged.")
print("Clean-environment installation remains unverified.")

## 43. Package and Verify the Service Bundle

Package the service source, configuration, dependency records, and README.
Record SHA-256 fingerprints and verify every packaged file after ZIP creation.

Check that configured checkpoints exist and match their recorded fingerprints.
Model checkpoints, datasets, and evaluation evidence are not included in this
service-only archive. Packaging does not establish clean-install portability.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile

# Package only the active experiment's service bundle.
if "RUN_DIR" not in globals():
    raise RuntimeError(
        "RUN_DIR is undefined. Initialize or explicitly resume the intended run."
    )

active_run = Path(RUN_DIR).resolve()
results_root = (Path(M8_DIR) / "results").resolve()

if not active_run.is_relative_to(results_root):
    raise ValueError("RUN_DIR must be inside M8/results.")

BUNDLE_DIR = active_run / "service_bundle"

if not BUNDLE_DIR.is_dir():
    raise FileNotFoundError(BUNDLE_DIR)

required_files = [
    "pipeline.py",
    "app.py",
    "config.json",
    "launch.local.json",
    "requirements-service.txt",
    "requirements-verification.txt",
    "environment_inventory.json",
    "README.md",
]

# Snapshot the exact bytes that will be packaged.
bundle_contents = {}

for filename in required_files:
    path = BUNDLE_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(f"Required bundle file missing: {path}")

    bundle_contents[filename] = path.read_bytes()

# Check Python syntax without executing the service.
for filename in ("pipeline.py", "app.py"):
    compile(
        bundle_contents[filename].decode("utf-8"),
        filename,
        "exec",
    )
    print("PASS: Python syntax:", filename)

config = json.loads(
    bundle_contents["config.json"].decode("utf-8")
)

root = Path(PROJECT_ROOT).resolve()
checkpoint_records = {}

for model_name in ("m4", "m5"):
    entry = config[model_name]
    checkpoint = (root / entry["checkpoint"]).resolve()

    if not checkpoint.is_relative_to(root):
        raise ValueError(
            f"Checkpoint outside project root: {model_name}"
        )

    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)

    with checkpoint.open("rb") as stream:
        actual_hash = hashlib.file_digest(
            stream, "sha256"
        ).hexdigest()

    if actual_hash != entry["sha256"]:
        raise ValueError(
            f"Checkpoint fingerprint mismatch: {model_name}"
        )

    checkpoint_records[model_name] = {
        "path_relative_to_project_root": entry["checkpoint"],
        "sha256": actual_hash,
        "included_in_archive": False,
    }

    print("PASS: Checkpoint verified:", model_name.upper())

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)
package_dir = active_run / f"service_package_{timestamp}"
package_dir.mkdir(parents=True, exist_ok=False)

manifest = {
    "purpose": "service_bundle_integrity",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_run": active_run.name,
    "backend": "PyTorch",
    "files": [
        {
            "path": filename,
            "size_bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
        }
        for filename, content in bundle_contents.items()
    ],
    "external_checkpoints": checkpoint_records,
    "datasets_included": False,
    "evaluation_evidence_included": False,
    "clean_installation_verified": False,
    "production_readiness_assessed": False,
    "verification_scope": (
        "Archive integrity, Python syntax and checkpoint fingerprints only. "
        "Packaging does not rerun inference or establish requirements completion."
    ),
}

manifest_bytes = (
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n"
).encode("utf-8")

manifest_path = package_dir / "bundle_manifest.json"
manifest_path.write_bytes(manifest_bytes)

archive_path = package_dir / "M8_service_bundle.zip"

with zipfile.ZipFile(
    archive_path,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for filename, content in bundle_contents.items():
        archive.writestr(
            f"service_bundle/{filename}",
            content,
        )

    archive.writestr(
        "service_bundle/bundle_manifest.json",
        manifest_bytes,
    )

expected_members = {
    f"service_bundle/{filename}"
    for filename in required_files
}
expected_members.add("service_bundle/bundle_manifest.json")

# Read back the archive and verify its exact contents.
with zipfile.ZipFile(archive_path, "r") as archive:
    members = archive.namelist()

    if len(members) != len(expected_members):
        raise AssertionError("Unexpected or duplicate ZIP members.")

    if set(members) != expected_members:
        raise AssertionError("ZIP file list differs from the package plan.")

    bad_member = archive.testzip()
    if bad_member is not None:
        raise AssertionError(f"ZIP integrity failure: {bad_member}")

    for record in manifest["files"]:
        content = archive.read(
            f"service_bundle/{record['path']}"
        )
        actual_hash = hashlib.sha256(content).hexdigest()

        if actual_hash != record["sha256"]:
            raise AssertionError(
                f"ZIP fingerprint mismatch: {record['path']}"
            )

    packaged_manifest = archive.read(
        "service_bundle/bundle_manifest.json"
    )
    if packaged_manifest != manifest_bytes:
        raise AssertionError("Packaged manifest differs.")

with archive_path.open("rb") as stream:
    archive_hash = hashlib.file_digest(
        stream, "sha256"
    ).hexdigest()

checksum_path = package_dir / "M8_service_bundle.zip.sha256"
checksum_path.write_text(
    f"{archive_hash}  {archive_path.name}\n",
    encoding="utf-8",
)

print("\nActive run:", active_run)
print("PASS: Service archive contents and fingerprints verified.")
print("Packaged service files:", len(required_files))
print("ZIP:", archive_path)
print("Manifest:", manifest_path)
print("ZIP SHA-256:", archive_hash)
print("Models, datasets, and evaluation evidence are external.")
print("Packaging does not establish completion of all M8 requirements.")

## 44. Add an Explicit Pothole Area-Based Severity Proxy

For D40 detections with a nonempty predicted mask, report severity as an
image-area proxy: predicted mask pixels divided by full-image pixels × 100.

This is not physical area, depth, or an engineering severity category.
Values depend on image framing, perspective, and prediction quality.
They are not directly comparable across arbitrary images.

For D40 detections without a predicted mask, severity is unavailable.
Crack severity remains unsupported by the pothole segmentation model.

Create a candidate pipeline version and verify that existing predictions,
masks, and annotations remain unchanged.

In [ ]:
from pathlib import Path
import copy
import importlib.util
import json

BUNDLE_DIR = Path(RUN_DIR) / "service_bundle"
original_path = BUNDLE_DIR / "pipeline.py"
candidate_path = BUNDLE_DIR / "pipeline_v2.py"

source = original_path.read_text(encoding="utf-8")


def replace_once(text, old, new):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(
            f"Expected one matching source block; found {count}. "
            "No candidate file has been written."
        )
    return text.replace(old, new, 1)


# Distinguish unsupported crack severity from a missing pothole mask.
source = replace_once(
    source,
    '"severity_status": "not_estimated",',
    (
        '"severity_status": (\n'
        '                    "unavailable_no_predicted_mask"\n'
        '                    if class_id == 3\n'
        '                    else "unsupported_damage_class"\n'
        '                ),'
    ),
) if source.count('"severity_status": "not_estimated",') == 1 else source

# The original implementation has both per-detection and report statuses.
old_detection_block = '''                "severity": None,
                "severity_status": "not_estimated",
'''

new_detection_block = '''                "severity": None,
                "severity_status": (
                    "unavailable_no_predicted_mask"
                    if class_id == 3
                    else "unsupported_damage_class"
                ),
'''

if old_detection_block in source:
    source = replace_once(
        source, old_detection_block, new_detection_block
    )

# Add the proxy only after a nonempty D40 mask has been produced.
anchor = '    union_pixels = int(union_mask.sum())'

severity_code = '''    for detection in detections:
        if detection["class_id"] != 3:
            continue

        segmentation = detection["segmentation"]
        if segmentation["mask_pixels"] > 0:
            detection["severity"] = {
                "measure": "predicted_pothole_area_percent_of_full_image",
                "value": segmentation["mask_percent_of_full_image"],
                "units": "percent",
                "mask_pixels": segmentation["mask_pixels"],
                "denominator_pixels": int(width * height),
                "interpretation": "image_area_proxy_only",
                "physical_area_estimated": False,
                "depth_estimated": False,
                "engineering_category": None,
            }
            detection["severity_status"] = "area_proxy_estimated"

    union_pixels = int(union_mask.sum())'''

source = replace_once(source, anchor, severity_code)

source = replace_once(
    source,
    '"schema_version": "1.0",',
    '"schema_version": "1.1",',
)

source = replace_once(
    source,
    '        "severity_status": "not_estimated",',
    '''        "severity_status": (
            "area_proxy_estimated"
            if any(
                d["severity_status"] == "area_proxy_estimated"
                for d in detections
            )
            else (
                "unavailable_no_predicted_mask"
                if segmentation_calls
                else "no_d40_detected"
            )
        ),''',
)

compile(source, str(candidate_path), "exec")

# Preserve an existing candidate if it differs.
if candidate_path.exists():
    if candidate_path.read_text(encoding="utf-8") != source:
        raise FileExistsError(
            f"A different candidate already exists: {candidate_path}"
        )
else:
    candidate_path.write_text(source, encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "m8_pipeline_v2", candidate_path
)
pipeline_v2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline_v2)

candidate_report, candidate_overlay, candidate_mask = (
    pipeline_v2.run_integrated_inference(
        integration_bgr,
        m4_model,
        m5_model,
        M4_INTEGRATION_SETTINGS,
        M5_SETTINGS,
        device=DEVICE,
    )
)

# Compare predictions while excluding the intentionally changed schema fields.
def prediction_only(report):
    normalized = copy.deepcopy(report)
    normalized.pop("schema_version", None)
    normalized.pop("severity_status", None)

    for detection in normalized["detections"]:
        detection.pop("severity", None)
        detection.pop("severity_status", None)

    return normalized


import math

# Explicit tolerances for repeated inference on this regression example.
BOX_ATOL_PIXELS = 1e-4
CONFIDENCE_ATOL = 1e-6

old_report = prediction_only(pipeline_report)
new_report = prediction_only(candidate_report)

old_detections = old_report.pop("detections")
new_detections = new_report.pop("detections")

# All report fields outside detections must match exactly.
other_fields_exact = old_report == new_report

same_count = len(old_detections) == len(new_detections)
boxes_close = same_count
confidences_close = same_count
detection_metadata_exact = same_count

for old_detection, new_detection in zip(
    old_detections, new_detections
):
    old_detection = copy.deepcopy(old_detection)
    new_detection = copy.deepcopy(new_detection)

    old_box = old_detection.pop("bbox_xyxy")
    new_box = new_detection.pop("bbox_xyxy")

    boxes_close = boxes_close and (
        len(old_box) == len(new_box) == 4
        and all(
            math.isclose(
                old_value,
                new_value,
                rel_tol=0.0,
                abs_tol=BOX_ATOL_PIXELS,
            )
            for old_value, new_value in zip(old_box, new_box)
        )
    )

    old_confidence = old_detection.pop("confidence")
    new_confidence = new_detection.pop("confidence")

    confidences_close = confidences_close and math.isclose(
        old_confidence,
        new_confidence,
        rel_tol=0.0,
        abs_tol=CONFIDENCE_ATOL,
    )

    # Includes classes, segmentation statuses, crop bounds and mask areas.
    detection_metadata_exact = (
        detection_metadata_exact
        and old_detection == new_detection
    )

checks = {
    "other_report_fields_exact": other_fields_exact,
    "detection_count_exact": same_count,
    "detection_metadata_exact": detection_metadata_exact,
    "boxes_within_0.0001_pixel": boxes_close,
    "confidence_within_0.000001": confidences_close,
    "mask_unchanged": np.array_equal(candidate_mask, pipeline_mask),
    "overlay_unchanged": np.array_equal(
        candidate_overlay, pipeline_overlay
    ),
}
severity_checks = []
for detection in candidate_report["detections"]:
    if detection["class_id"] != 3:
        severity_checks.append(
            detection["severity"] is None
            and detection["severity_status"] == "unsupported_damage_class"
        )
        continue

    pixels = detection["segmentation"]["mask_pixels"]

    if pixels > 0:
        severity = detection["severity"]
        expected = (
            100.0 * pixels
            / (
                candidate_report["image_width"]
                * candidate_report["image_height"]
            )
        )
        severity_checks.append(
            severity is not None
            and abs(severity["value"] - expected) < 1e-10
            and severity["units"] == "percent"
            and detection["severity_status"] == "area_proxy_estimated"
        )
        print("D40 area-based severity proxy:", severity)
    else:
        severity_checks.append(
            detection["severity"] is None
            and detection["severity_status"] == "unavailable_no_predicted_mask"
        )

checks["severity_fields_consistent"] = all(severity_checks)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

if not all(checks.values()):
    raise AssertionError("Candidate verification failed; service unchanged.")

candidate_report_path = Path(RUN_DIR) / "severity_proxy_candidate_report.json"
candidate_report_path.write_text(
    json.dumps(candidate_report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("\nCandidate module:", candidate_path)
print("Report:", candidate_report_path)
print("PASS: Area-based severity proxy verified on the smoke-test example.")
print("Proxy accuracy against ground truth has not been evaluated here.")
print("The running HTTP service still uses the original pipeline.")

## 45. Add CPU Per-Stage Pipeline Timing

Measure input validation, detection, detection postprocessing, crop preparation,
segmentation, mask restoration, severity calculation, annotation, and report creation.

Model predict timings include each model's internal preprocessing and
postprocessing. Crop-stage timings accumulate across all detected potholes.

Timing excludes model loading, file decoding, PNG encoding, HTTP, and final
timing-metadata assembly. These smoke-test timings are not benchmark results.

In [ ]:
from pathlib import Path
import importlib.util
import json
import math

BUNDLE_DIR = Path(RUN_DIR) / "service_bundle"
source_path = BUNDLE_DIR / "pipeline_v2.py"
timed_path = BUNDLE_DIR / "pipeline_v3.py"

source = source_path.read_text(encoding="utf-8")


def insert_before(anchor, addition):
    global source
    count = source.count(anchor)
    if count != 1:
        raise RuntimeError(
            f"Expected one insertion point, found {count}: {anchor}"
        )
    source = source.replace(anchor, addition + "\n" + anchor, 1)


if source.count("import math\n") != 1:
    raise RuntimeError("Unexpected source imports.")

source = source.replace(
    "import math\n", "import math\nimport time\n", 1
)

insert_before(
    "    # Validate the input before calling either model.",
    '''    if str(device) != "cpu":
        raise ValueError("Stage timing currently supports CPU only.")

    timing_start = time.perf_counter()
    timing_last = timing_start
    stage_ms = dict.fromkeys([
        "input_validation",
        "m4_predict",
        "detection_postprocess",
        "crop_preparation",
        "m5_predict",
        "mask_restoration",
        "severity_calculation",
        "annotation",
        "report_creation",
    ], 0.0)

    def mark_stage(name):
        nonlocal timing_last
        now = time.perf_counter()
        stage_ms[name] += (now - timing_last) * 1000.0
        timing_last = now
''',
)

insert_before(
    "    result = detector.predict(",
    '    mark_stage("input_validation")',
)

insert_before(
    "    if tuple(result.orig_shape)",
    '    mark_stage("m4_predict")',
)

insert_before(
    '    for detection in detections:\n'
    '        if detection["class_id"] != 3:\n'
    '            continue\n\n'
    '        x1, y1',
    '    mark_stage("detection_postprocess")',
)

insert_before(
    "        seg_result = segmenter.predict(",
    '        mark_stage("crop_preparation")',
)

insert_before(
    "        segmentation_calls += 1",
    '        mark_stage("m5_predict")',
)

# This marker is inside the crop loop, after each crop's mask processing.
insert_before(
    '    for detection in detections:\n'
    '        if detection["class_id"] != 3:\n'
    '            continue\n\n'
    '        segmentation =',
    '        mark_stage("mask_restoration")\n',
)

insert_before(
    "    annotated = source.copy()",
    '    mark_stage("severity_calculation")',
)

insert_before(
    "    report = {",
    '    mark_stage("annotation")',
)

insert_before(
    "    return report, annotated, union_mask",
    '''    mark_stage("report_creation")

    report["timing"] = {
        "device": "cpu",
        "units": "milliseconds",
        "stages": stage_ms,
        "total_ms": (timing_last - timing_start) * 1000.0,
        "scope": (
            "Input validation through prediction-report JSON validation"
        ),
        "excluded": [
            "model loading",
            "file decoding",
            "PNG encoding",
            "HTTP",
            "timing metadata assembly",
        ],
        "predict_scope": (
            "Model predict calls include their preprocessing "
            "and postprocessing"
        ),
    }
''',
)

if source.count('"schema_version": "1.1"') != 1:
    raise RuntimeError("Expected candidate schema version 1.1.")

source = source.replace(
    '"schema_version": "1.1"',
    '"schema_version": "1.2"',
    1,
)

compile(source, str(timed_path), "exec")

if timed_path.exists():
    if timed_path.read_text(encoding="utf-8") != source:
        raise FileExistsError(
            f"A different timing candidate already exists: {timed_path}"
        )
else:
    timed_path.write_text(source, encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "m8_pipeline_v3", timed_path
)
pipeline_v3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline_v3)

timed_report, timed_overlay, timed_mask = (
    pipeline_v3.run_integrated_inference(
        integration_bgr,
        m4_model,
        m5_model,
        M4_INTEGRATION_SETTINGS,
        M5_SETTINGS,
        device="cpu",
    )
)

timing = timed_report["timing"]
stages = timing["stages"]

if not all(
    math.isfinite(value) and value >= 0
    for value in [*stages.values(), timing["total_ms"]]
):
    raise AssertionError("Invalid timing values.")

if not math.isclose(
    sum(stages.values()),
    timing["total_ms"],
    rel_tol=1e-9,
    abs_tol=1e-6,
):
    raise AssertionError("Stage timings do not sum to measured total.")

# These checks cover timing structure and unchanged image outputs.
# Full report and edge-case regression checks follow separately.
import numpy as np

if not np.array_equal(timed_mask, candidate_mask):
    raise AssertionError("Mask changed after adding timing.")

if not np.array_equal(timed_overlay, candidate_overlay):
    raise AssertionError("Annotated image changed after adding timing.")

json.dumps(timed_report, allow_nan=False)

for name, value in stages.items():
    print(f"{name:<26} {value:>10.3f} ms")

print(f"{'TOTAL':<26} {timing['total_ms']:>10.3f} ms")

timed_report_path = Path(RUN_DIR) / "stage_timing_candidate_report.json"
timed_report_path.write_text(
    json.dumps(timed_report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("\nPASS: Stage timings are finite, nonnegative, and sum to total.")
print("PASS: Mask and annotated image remain unchanged.")
print("Candidate:", timed_path)
print("Report:", timed_report_path)
print("Single-example timing only; not a benchmark.")
print("The running HTTP service has not been changed.")

## 46. Verify Severity and Timing Edge Cases

Use controlled model doubles to check skipped segmentation, missing masks,
overlapping masks, area-proxy denominators, input preservation, and timing
consistency.

Compare candidate versions 2 and 3 after excluding schema version and timing.
These checks validate implementation logic, not model accuracy or speed.

In [ ]:
from types import SimpleNamespace
from pathlib import Path
import copy
import hashlib
import json
import math
import numpy as np


class TestTensor:
    def __init__(self, values):
        self.values = np.asarray(values, dtype=np.float32)

    def cpu(self):
        return self

    def numpy(self):
        return self.values


class TestBoxes:
    def __init__(self, boxes, classes, confidences):
        self.xyxy = TestTensor(np.asarray(boxes).reshape(-1, 4))
        self.cls = TestTensor(classes)
        self.conf = TestTensor(confidences)

    def __len__(self):
        return len(self.cls.values)


class TestDetector:
    task = "detect"
    names = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}

    def __init__(self, boxes, classes):
        self.boxes = boxes
        self.classes = classes
        self.calls = 0

    def predict(self, source, **kwargs):
        self.calls += 1
        return [SimpleNamespace(
            orig_shape=source.shape[:2],
            boxes=TestBoxes(
                self.boxes,
                self.classes,
                [0.9] * len(self.classes),
            ),
        )]


class TestSegmenter:
    task = "segment"
    names = {0: "pothole"}

    def __init__(self, produce_mask):
        self.produce_mask = produce_mask
        self.calls = 0

    def predict(self, source, **kwargs):
        self.calls += 1
        height, width = source.shape[:2]

        if self.produce_mask:
            boxes = TestBoxes([[0, 0, width, height]], [0], [0.9])
            masks = SimpleNamespace(
                data=TestTensor(np.ones((1, height, width)))
            )
        else:
            boxes = TestBoxes([], [], [])
            masks = None

        return [SimpleNamespace(
            orig_shape=(height, width),
            boxes=boxes,
            masks=masks,
        )]


test_image = np.zeros((40, 40, 3), dtype=np.uint8)
test_m4_settings = {
    "imgsz": 32, "conf": 0.1, "iou": 0.5, "max_det": 300
}
test_m5_settings = {
    "imgsz": 32, "conf": 0.05, "iou": 0.7, "max_det": 100
}

test_results = []


def check(name, condition):
    passed = bool(condition)
    test_results.append({"test": name, "passed": passed})
    print(f"{'PASS' if passed else 'FAIL'}: {name}")


def run_case(module, boxes, classes, produce_mask):
    detector = TestDetector(boxes, classes)
    segmenter = TestSegmenter(produce_mask)
    image = test_image.copy()

    report, overlay, mask = module.run_integrated_inference(
        image,
        detector,
        segmenter,
        test_m4_settings,
        test_m5_settings,
        device="cpu",
    )
    return report, overlay, mask, image, detector, segmenter


def validate_timing(report):
    timing = report["timing"]
    values = list(timing["stages"].values())

    return (
        timing["device"] == "cpu"
        and timing["units"] == "milliseconds"
        and all(
            math.isfinite(value) and value >= 0
            for value in values + [timing["total_ms"]]
        )
        and math.isclose(
            sum(values), timing["total_ms"],
            rel_tol=1e-9, abs_tol=1e-6,
        )
    )


cases = [
    ("no_detections", [], [], False),
    ("crack_only", [[2, 2, 12, 12]], [0], False),
    ("pothole_without_mask", [[0, 0, 20, 20]], [3], False),
    (
        "overlapping_potholes",
        [[0, 0, 20, 20], [10, 10, 30, 30]],
        [3, 3],
        True,
    ),
]

case_outputs = {}

for name, boxes, classes, produce_mask in cases:
    old = run_case(pipeline_v2, boxes, classes, produce_mask)
    new = run_case(pipeline_v3, boxes, classes, produce_mask)

    old_report = copy.deepcopy(old[0])
    new_report = copy.deepcopy(new[0])
    old_report.pop("schema_version")
    new_report.pop("schema_version")
    new_report.pop("timing")

    check(
        f"{name}: prediction report unchanged",
        old_report == new_report,
    )
    check(
        f"{name}: image outputs unchanged",
        np.array_equal(old[1], new[1])
        and np.array_equal(old[2], new[2]),
    )
    check(f"{name}: timing valid", validate_timing(new[0]))
    check(
        f"{name}: input preserved",
        np.array_equal(new[3], test_image),
    )

    case_outputs[name] = new

for name in ("no_detections", "crack_only"):
    report, _, _, _, _, segmenter = case_outputs[name]
    stages = report["timing"]["stages"]

    check(
        f"{name}: segmentation skipped with zero crop-stage times",
        segmenter.calls == 0
        and report["segmentation_calls"] == 0
        and all(
            stages[key] == 0.0
            for key in (
                "crop_preparation", "m5_predict", "mask_restoration"
            )
        ),
    )

crack = case_outputs["crack_only"][0]["detections"][0]
check(
    "Crack severity explicitly unsupported",
    crack["severity"] is None
    and crack["severity_status"] == "unsupported_damage_class",
)

empty_report = case_outputs["pothole_without_mask"][0]
empty_detection = empty_report["detections"][0]
check(
    "Missing mask means unavailable severity, not zero severity",
    empty_detection["severity"] is None
    and empty_detection["severity_status"]
        == "unavailable_no_predicted_mask"
    and empty_report["severity_status"]
        == "unavailable_no_predicted_mask",
)

overlap_report, _, overlap_mask, _, _, segmenter = (
    case_outputs["overlapping_potholes"]
)

check(
    "Overlapping masks union to 700 pixels",
    int(overlap_mask.sum()) == 700
    and overlap_report["union_mask_pixels"] == 700
    and overlap_report["union_mask_percent_of_full_image"] == 43.75
    and segmenter.calls == 2,
)

check(
    "Each pothole proxy uses the full-image denominator",
    all(
        detection["severity"]["mask_pixels"] == 400
        and detection["severity"]["denominator_pixels"] == 1600
        and detection["severity"]["value"] == 25.0
        and detection["severity"]["engineering_category"] is None
        and detection["severity_status"] == "area_proxy_estimated"
        for detection in overlap_report["detections"]
    ),
)

# Invalid inputs must fail before either model is called.
invalid_inputs = {
    "None": None,
    "grayscale": np.zeros((40, 40), dtype=np.uint8),
    "float": np.zeros((40, 40, 3), dtype=np.float32),
    "empty": np.zeros((0, 40, 3), dtype=np.uint8),
    "four_channel": np.zeros((40, 40, 4), dtype=np.uint8),
}

for name, invalid_image in invalid_inputs.items():
    detector = TestDetector([], [])
    segmenter = TestSegmenter(False)
    rejected = False

    try:
        pipeline_v3.run_integrated_inference(
            invalid_image,
            detector,
            segmenter,
            test_m4_settings,
            test_m5_settings,
            device="cpu",
        )
    except (TypeError, ValueError):
        rejected = True

    check(
        f"{name}: rejected before model inference",
        rejected and detector.calls == 0 and segmenter.calls == 0,
    )

verification_path = Path(RUN_DIR) / "pipeline_v3_edge_case_tests.json"
verification = {
    "test_type": "controlled_model_doubles",
    "real_models_used": False,
    "pipeline_sha256": hashlib.sha256(
        (Path(RUN_DIR) / "service_bundle" / "pipeline_v3.py").read_bytes()
    ).hexdigest(),
    "passed": sum(item["passed"] for item in test_results),
    "total": len(test_results),
    "tests": test_results,
}

verification_path.write_text(
    json.dumps(verification, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print(
    f"\nPassed: {verification['passed']}/{verification['total']}"
)
print("Saved:", verification_path)

if verification["passed"] != verification["total"]:
    raise AssertionError("Candidate edge-case verification failed.")

print("PASS: Severity and timing edge cases verified.")
print("The running HTTP service remains unchanged.")

## 47. Start and Verify the Updated HTTP Service

Create a separate service bundle containing the verified timing/severity
pipeline and updated output metadata. Preserve the original bundle and evidence.

Start the updated PyTorch service on localhost:8766 and verify its response
schema, model fingerprints, mask, annotated image, severity proxy, and timing.

This is a functional smoke test, not a new performance benchmark.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import json
import math
import os
import socket
import subprocess
import sys
import time

import cv2
import numpy as np
import requests

run_root = Path(RUN_DIR).resolve()
original_bundle = run_root / "service_bundle"
candidate_file = original_bundle / "pipeline_v3.py"

candidate_bytes = candidate_file.read_bytes()
candidate_hash = hashlib.sha256(candidate_bytes).hexdigest()

edge_test = json.loads(
    (run_root / "pipeline_v3_edge_case_tests.json").read_text(
        encoding="utf-8"
    )
)

if (
    edge_test["passed"] != edge_test["total"]
    or edge_test["pipeline_sha256"] != candidate_hash
):
    raise RuntimeError("Candidate does not match the passing edge-case tests.")

if (
    "service_v3_process" in globals()
    and service_v3_process.poll() is None
):
    raise RuntimeError("The updated service is already running.")

V3_SERVICE_URL = "http://127.0.0.1:8766"

# Check availability before creating or starting the new service.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
    probe.bind(("127.0.0.1", 8766))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
V3_BUNDLE_DIR = run_root / f"service_bundle_v3_{stamp}"
V3_BUNDLE_DIR.mkdir(exist_ok=False)

(V3_BUNDLE_DIR / "pipeline.py").write_bytes(candidate_bytes)
(V3_BUNDLE_DIR / "app.py").write_bytes(
    (original_bundle / "app.py").read_bytes()
)

v3_config = json.loads(
    (original_bundle / "config.json").read_text(encoding="utf-8")
)
v3_config["schema_version"] = "1.2"
v3_config["output"].update({
    "severity_status": "area_proxy_only",
    "severity_measure": "predicted_pothole_area_percent_of_full_image",
    "physical_area_estimated": False,
    "depth_estimated": False,
    "per_stage_timing": True,
    "timing_device": "cpu",
})

if v3_config["device"] != "cpu":
    raise ValueError("This timing implementation requires CPU.")

(V3_BUNDLE_DIR / "config.json").write_text(
    json.dumps(v3_config, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

(V3_BUNDLE_DIR / "launch.local.json").write_text(
    json.dumps({
        "project_root_relative_to_bundle": os.path.relpath(
            Path(PROJECT_ROOT).resolve(), V3_BUNDLE_DIR
        )
    }, indent=2) + "\n",
    encoding="utf-8",
)

v3_artifact_hashes = {
    name: hashlib.sha256((V3_BUNDLE_DIR / name).read_bytes()).hexdigest()
    for name in ("pipeline.py", "app.py", "config.json")
}

V3_SERVICE_LOG = V3_BUNDLE_DIR / "http_service.log"
environment = os.environ.copy()
environment["M8_PROJECT_ROOT"] = str(Path(PROJECT_ROOT).resolve())

with V3_SERVICE_LOG.open("ab") as log:
    service_v3_process = subprocess.Popen(
        [
            sys.executable, "-m", "uvicorn", "app:app",
            "--host", "127.0.0.1",
            "--port", "8766",
            "--workers", "1",
        ],
        cwd=str(V3_BUNDLE_DIR),
        env=environment,
        stdout=log,
        stderr=subprocess.STDOUT,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )

session = requests.Session()
session.trust_env = False
v3_checks = {}

try:
    deadline = time.monotonic() + 60

    while True:
        if service_v3_process.poll() is not None:
            raise RuntimeError(
                f"Updated service exited. Inspect {V3_SERVICE_LOG}"
            )

        try:
            response = session.get(
                f"{V3_SERVICE_URL}/health", timeout=2
            )
            response.raise_for_status()
            v3_health = response.json()
            break
        except requests.RequestException:
            if time.monotonic() >= deadline:
                raise TimeoutError("Updated service startup timed out.")
            time.sleep(0.5)

    expected_hashes = {
        "m4": v3_config["m4"]["sha256"],
        "m5": v3_config["m5"]["sha256"],
    }

    if (
        v3_health["status"] != "ready"
        or v3_health["m4_sha256"] != expected_hashes["m4"]
        or v3_health["m5_sha256"] != expected_hashes["m5"]
    ):
        raise AssertionError("Health or model fingerprint mismatch.")

    ok, encoded = cv2.imencode(".png", integration_bgr)
    if not ok:
        raise RuntimeError("Input PNG encoding failed.")

    response = session.post(
        f"{V3_SERVICE_URL}/predict",
        files={"file": ("input.png", encoded.tobytes(), "image/png")},
        timeout=120,
    )
    response.raise_for_status()
    body = response.json()
    v3_http_report = body["report"]

    def decode_png(field, flag):
        raw = base64.b64decode(body[field], validate=True)
        decoded = cv2.imdecode(
            np.frombuffer(raw, dtype=np.uint8), flag
        )
        if decoded is None:
            raise ValueError(f"Invalid output PNG: {field}")
        return decoded

    http_mask = decode_png("mask_png_base64", cv2.IMREAD_UNCHANGED)
    http_overlay = decode_png(
        "annotated_png_base64", cv2.IMREAD_COLOR
    )

    timing = v3_http_report["timing"]
    stage_values = list(timing["stages"].values())

    v3_checks["schema_version_1_2"] = (
        v3_http_report["schema_version"] == "1.2"
    )
    v3_checks["model_fingerprints_match"] = (
        body["model_fingerprints"] == expected_hashes
    )
    v3_checks["mask_exact_match"] = np.array_equal(
        http_mask, timed_mask.astype(np.uint8) * 255
    )
    v3_checks["overlay_exact_match"] = np.array_equal(
        http_overlay, timed_overlay
    )
    v3_checks["mask_area_consistent"] = (
        int(np.count_nonzero(http_mask))
        == v3_http_report["union_mask_pixels"]
    )
    v3_checks["detection_and_segmentation_counts_match"] = (
        v3_http_report["detection_count"]
        == timed_report["detection_count"]
        and v3_http_report["segmentation_calls"]
        == timed_report["segmentation_calls"]
    )
    v3_checks["timing_valid"] = (
        timing["device"] == "cpu"
        and timing["units"] == "milliseconds"
        and all(
            math.isfinite(value) and value >= 0
            for value in stage_values + [timing["total_ms"]]
        )
        and math.isclose(
            sum(stage_values), timing["total_ms"],
            rel_tol=1e-9, abs_tol=1e-6,
        )
    )

    potholes = [
        detection for detection in v3_http_report["detections"]
        if detection["class_id"] == 3
    ]
    denominator = (
        v3_http_report["image_width"]
        * v3_http_report["image_height"]
    )

    v3_checks["nonempty_pothole_proxy_verified"] = (
        bool(potholes)
        and any(
            d["segmentation"]["mask_pixels"] > 0 for d in potholes
        )
        and all(
            (
                d["severity"] is not None
                and d["severity_status"] == "area_proxy_estimated"
                and d["severity"]["units"] == "percent"
                and d["severity"]["denominator_pixels"] == denominator
                and math.isclose(
                    d["severity"]["value"],
                    100.0 * d["segmentation"]["mask_pixels"] / denominator,
                    rel_tol=0.0, abs_tol=1e-10,
                )
            )
            if d["segmentation"]["mask_pixels"] > 0
            else (
                d["severity"] is None
                and d["severity_status"] == "unavailable_no_predicted_mask"
            )
            for d in potholes
        )
    )

    for name, passed in v3_checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {name}")

    (V3_BUNDLE_DIR / "http_smoke_report.json").write_text(
        json.dumps(v3_http_report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    (V3_BUNDLE_DIR / "http_verification.json").write_text(
        json.dumps({
            "purpose": "updated_HTTP_functional_smoke_test",
            "service_url": V3_SERVICE_URL,
            "pid": service_v3_process.pid,
            "artifact_sha256": v3_artifact_hashes,
            "checks": v3_checks,
            "passed": all(v3_checks.values()),
            "full_report_numerical_comparison_performed": False,
        }, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

    if not all(v3_checks.values()):
        raise AssertionError("Updated HTTP smoke test failed.")

except BaseException:
    # Stop only the new process created by this cell.
    if service_v3_process.poll() is None:
        service_v3_process.terminate()
        try:
            service_v3_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            service_v3_process.kill()
            service_v3_process.wait()
    raise

finally:
    session.close()

print("\nPASS: Updated HTTP service smoke test completed.")
print("New bundle:", V3_BUNDLE_DIR)
print("Interactive API:", V3_SERVICE_URL + "/docs")
print("Log:", V3_SERVICE_LOG)
print("Original bundle and previous evidence preserved.")
print("Both services may be running; do not benchmark them concurrently.")

## 48. Retire the Previous Service and Validate Updated HTTP Inputs

Stop the previous notebook-owned server after verifying that the updated
server is healthy. Repeat input-validation checks against the updated service.

Record the active service bundle explicitly. Preserve all previous artifacts.

In [ ]:
from pathlib import Path
from PIL import Image
import io
import json
import subprocess
import requests

if service_v3_process.poll() is not None:
    raise RuntimeError("The updated service is not running.")

session = requests.Session()
session.trust_env = False

try:
    response = session.get(f"{V3_SERVICE_URL}/health", timeout=10)
    response.raise_for_status()
    health = response.json()

    if health["status"] != "ready":
        raise RuntimeError("The updated service is not ready.")

    # Stop only the original process owned by this notebook.
    old_process = globals().get("service_process")

    if old_process is not None and old_process.poll() is None:
        command = old_process.args

        if (
            old_process.pid == service_v3_process.pid
            or not isinstance(command, (list, tuple))
            or "uvicorn" not in command
            or "8765" not in command
        ):
            raise RuntimeError(
                "Could not verify the old process identity; it was not stopped."
            )

        old_process.terminate()
        try:
            old_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            old_process.kill()
            old_process.wait(timeout=10)

        print("PASS: Previous notebook-owned service stopped.")
    else:
        print("Previous notebook-owned service is already stopped.")

    byte_limit = int(health["max_upload_bytes"])
    pixel_limit = int(health["max_image_pixels"])

    gif_buffer = io.BytesIO()
    Image.new("RGB", (2, 2)).save(gif_buffer, format="GIF")

    # Exceed the pixel limit while keeping the compressed PNG small.
    large_width = 4000
    large_height = pixel_limit // large_width + 1

    png_buffer = io.BytesIO()
    large_image = Image.new("RGB", (large_width, large_height))
    large_image.save(png_buffer, format="PNG")
    large_image.close()

    if len(png_buffer.getvalue()) > byte_limit:
        raise RuntimeError(
            "Pixel-limit test also exceeds byte limit; test is ambiguous."
        )

    cases = [
        ("Missing file field", None, 422),
        ("Empty file", ("empty.png", b"", "image/png"), 400),
        (
            "Invalid image bytes",
            ("invalid.png", b"not an image", "image/png"),
            400,
        ),
        (
            "Unsupported GIF",
            ("image.gif", gif_buffer.getvalue(), "image/gif"),
            415,
        ),
        (
            "Upload exceeds byte limit",
            ("large.png", b"x" * (byte_limit + 1), "image/png"),
            413,
        ),
        (
            "Image exceeds pixel limit",
            ("large_pixels.png", png_buffer.getvalue(), "image/png"),
            413,
        ),
    ]

    validation_results = []

    for name, upload, expected_status in cases:
        arguments = {"timeout": 60}
        if upload is not None:
            arguments["files"] = {"file": upload}

        response = session.post(
            f"{V3_SERVICE_URL}/predict", **arguments
        )

        health_response = session.get(
            f"{V3_SERVICE_URL}/health", timeout=10
        )
        healthy = (
            health_response.status_code == 200
            and health_response.json().get("status") == "ready"
        )

        passed = response.status_code == expected_status and healthy

        validation_results.append({
            "test": name,
            "expected_status": expected_status,
            "actual_status": response.status_code,
            "healthy_after_request": healthy,
            "passed": passed,
        })

        print(
            f"{'PASS' if passed else 'FAIL'}: {name}"
            f" — HTTP {response.status_code}, healthy={healthy}"
        )

    validation_path = Path(V3_BUNDLE_DIR) / "http_input_validation.json"
    validation_path.write_text(
        json.dumps({
            "service_url": V3_SERVICE_URL,
            "passed": sum(row["passed"] for row in validation_results),
            "total": len(validation_results),
            "tests": validation_results,
        }, indent=2) + "\n",
        encoding="utf-8",
    )

    if not all(row["passed"] for row in validation_results):
        raise AssertionError("Updated HTTP input-validation checks failed.")

    # Explicit selection for subsequent testing and final packaging.
    ACTIVE_SERVICE_BUNDLE = Path(V3_BUNDLE_DIR).resolve()
    ACTIVE_SERVICE_URL = V3_SERVICE_URL

    selection_path = Path(RUN_DIR) / "active_service.json"
    selection_path.write_text(
        json.dumps({
            "bundle_relative_to_run": ACTIVE_SERVICE_BUNDLE.relative_to(
                Path(RUN_DIR).resolve()
            ).as_posix(),
            "service_url": ACTIVE_SERVICE_URL,
            "backend": "PyTorch",
            "report_schema_version": "1.2",
            "status": "development_candidate",
            "production_approved": False,
        }, indent=2) + "\n",
        encoding="utf-8",
    )

    print("\nPASS: Updated service input validation completed.")
    print("Active bundle:", ACTIVE_SERVICE_BUNDLE)
    print("Active API:", ACTIVE_SERVICE_URL + "/docs")
    print("Selection saved:", selection_path)
    print("Previous benchmark results remain evidence for the previous version.")

finally:
    session.close()

## 49. Benchmark the Updated HTTP Service and Pipeline Stages

Reuse the original 180-image workload, ten warm-up requests, and three
measurement repeats. Record HTTP latency and the server's per-stage timings.

HTTP latency includes response-body receipt. Response validation is outside
individual request timing but inside measurement-loop wall time.

Sample updated-server RSS. Preserve the previous version's measurements.
No performance acceptance thresholds are introduced retrospectively.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import threading
import time

import cv2
import numpy as np
import psutil
import requests

if service_v3_process.poll() is not None:
    raise RuntimeError("Updated HTTP service is not running.")

original_protocol_path = Path(RUN_DIR) / "http_benchmark" / "protocol.json"
protocol = json.loads(original_protocol_path.read_text(encoding="utf-8"))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
V3_BENCHMARK_DIR = Path(RUN_DIR) / f"http_benchmark_v3_{stamp}"
V3_BENCHMARK_DIR.mkdir(exist_ok=False)

config = json.loads(
    (ACTIVE_SERVICE_BUNDLE / "config.json").read_text(encoding="utf-8")
)
fingerprints = {
    name: config[name]["sha256"] for name in ("m4", "m5")
}

protocol.update({
    "purpose": "updated_HTTP_and_stage_performance_measurement",
    "service_url": ACTIVE_SERVICE_URL,
    "report_schema_version": "1.2",
    "service_config": config,
    "artifact_hashes": {
        name: hashlib.sha256(
            (ACTIVE_SERVICE_BUNDLE / name).read_bytes()
        ).hexdigest()
        for name in ("pipeline.py", "app.py", "config.json")
    },
    "source_workload_protocol_sha256": hashlib.sha256(
        original_protocol_path.read_bytes()
    ).hexdigest(),
})

(V3_BENCHMARK_DIR / "protocol.json").write_text(
    json.dumps(protocol, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

inputs = []
for case in protocol["cases"]:
    raw = (Path(M4_DIR) / case["image"]).read_bytes()
    if hashlib.sha256(raw).hexdigest() != case["image_sha256"]:
        raise ValueError(f"Image fingerprint mismatch: {case['image']}")

    image = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Cannot decode: {case['image']}")

    ok, png = cv2.imencode(".png", image)
    if not ok:
        raise RuntimeError("PNG encoding failed.")

    inputs.append((png.tobytes(), image.shape[:2]))

print("Prepared inputs:", len(inputs))

session = requests.Session()
session.trust_env = False

health_response = session.get(
    ACTIVE_SERVICE_URL + "/health", timeout=10
)
health_response.raise_for_status()
health = health_response.json()
assert health["status"] == "ready"
assert health["m4_sha256"] == fingerprints["m4"]
assert health["m5_sha256"] == fingerprints["m5"]

stage_names = {
    "input_validation", "m4_predict", "detection_postprocess",
    "crop_preparation", "m5_predict", "mask_restoration",
    "severity_calculation", "annotation", "report_creation",
}


def post_case(index):
    return session.post(
        ACTIVE_SERVICE_URL + "/predict",
        files={"file": ("input.png", inputs[index][0], "image/png")},
        timeout=protocol["request_timeout_seconds"],
    )


def validate_response(response, index):
    # Reuse the original image/report validation from section 35.
    height, width = inputs[index][1]
    report = validate_benchmark_response(
        response, {"height": height, "width": width}
    )

    if report["schema_version"] != "1.2":
        raise ValueError("Unexpected report version.")

    timing = report["timing"]
    stages = timing["stages"]

    if set(stages) != stage_names:
        raise ValueError("Unexpected timing stages.")

    if not all(
        math.isfinite(value) and value >= 0
        for value in [*stages.values(), timing["total_ms"]]
    ):
        raise ValueError("Invalid timing values.")

    if not math.isclose(
        sum(stages.values()), timing["total_ms"],
        rel_tol=1e-9, abs_tol=1e-6,
    ):
        raise ValueError("Timing sum mismatch.")

    for detection in report["detections"]:
        severity = detection["severity"]
        status = detection["severity_status"]

        if detection["class_id"] != 3:
            if severity is not None or status != "unsupported_damage_class":
                raise ValueError("Unexpected crack severity.")
            continue

        pixels = detection["segmentation"]["mask_pixels"]
        if pixels == 0:
            if severity is not None or status != "unavailable_no_predicted_mask":
                raise ValueError("Missing mask must have unavailable severity.")
        else:
            if (
                severity is None
                or status != "area_proxy_estimated"
                or severity["denominator_pixels"] != height * width
                or severity["units"] != "percent"
                or not math.isclose(
                    severity["value"], 100.0 * pixels / (height * width),
                    rel_tol=0.0, abs_tol=1e-10,
                )
            ):
                raise ValueError("Invalid area-based severity proxy.")

    return report


# The reused validator checks this global fingerprint mapping.
expected_fingerprints = fingerprints

for number, index in enumerate(protocol["warmup_case_indices"], 1):
    with post_case(index) as response:
        validate_response(response, index)
    print(f"Warm-up {number}/{protocol['warmup_requests']}")

server = psutil.Process(service_v3_process.pid)
server_identity = server.create_time()
rss_baseline = server.memory_info().rss

samples = []
sampling_errors = []
stop_sampling = threading.Event()
origin = time.perf_counter()


def sample_memory():
    while not stop_sampling.is_set():
        try:
            if server.create_time() != server_identity:
                raise RuntimeError("Server process identity changed.")

            samples.append({
                "elapsed_seconds": time.perf_counter() - origin,
                "rss_bytes": int(server.memory_info().rss),
            })
        except Exception as exc:
            sampling_errors.append(str(exc))
            return

        stop_sampling.wait(0.05)


rows = []
interrupted = False
sampler = threading.Thread(target=sample_memory, daemon=True)
sampler.start()
loop_start = time.perf_counter()

try:
    for number, item in enumerate(protocol["schedule"], 1):
        index = item["case_index"]
        case = protocol["cases"][index]
        row = {
            "request_number": number,
            "repeat": item["repeat"],
            "case_index": index,
            "image": case["image"],
            "cohort": case["cohort"],
            "success": False,
            "latency_ms": None,
            "http_status": None,
            "error": None,
        }

        started = time.perf_counter()
        try:
            with post_case(index) as response:
                row["latency_ms"] = (
                    time.perf_counter() - started
                ) * 1000.0
                row["http_status"] = response.status_code
                report = validate_response(response, index)

            row.update({
                "success": True,
                "timing": report["timing"],
                "detection_count": report["detection_count"],
                "segmentation_calls": report["segmentation_calls"],
                "union_mask_pixels": report["union_mask_pixels"],
            })
        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"

        rows.append(row)

        if number % 30 == 0:
            failed = sum(not row["success"] for row in rows)
            print(f"Measured {number}/540 | failed requests: {failed}")

except KeyboardInterrupt:
    interrupted = True

finally:
    wall_seconds = time.perf_counter() - loop_start
    stop_sampling.set()
    sampler.join(timeout=5)
    session.close()

    result = {
        "purpose": "updated_HTTP_and_stage_performance_measurement",
        "interrupted": interrupted,
        "artifact_hashes": protocol["artifact_hashes"],
        "server_pid": service_v3_process.pid,
        "expected_requests": len(protocol["schedule"]),
        "successful_requests": sum(row["success"] for row in rows),
        "measurement_loop_wall_seconds": wall_seconds,
        "server_baseline_rss_bytes": int(rss_baseline),
        "server_sampled_peak_rss_bytes": max(
            [int(rss_baseline)] + [s["rss_bytes"] for s in samples]
        ),
        "sampling_errors": sampling_errors,
        "rss_samples": samples,
        "requests": rows,
    }

    result_path = V3_BENCHMARK_DIR / "measurements.json"
    result_path.write_text(
        json.dumps(result, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

print("\nSaved:", result_path)

if (
    interrupted
    or result["successful_requests"] != len(protocol["schedule"])
    or sampling_errors
):
    raise AssertionError("Benchmark incomplete or failed; inspect saved results.")

print("\nGroup                 p50 ms     p95 ms     p99 ms")
for cohort in ("general", "d40_diagnostic"):
    selected = [row for row in rows if row["cohort"] == cohort]
    values = [row["latency_ms"] for row in selected]
    p50, p95, p99 = np.percentile(values, [50, 95, 99])
    print(f"{cohort:<20} {p50:>10.2f} {p95:>10.2f} {p99:>10.2f}")

print("\nMean stage times across all measured requests:")
for name in sorted(stage_names):
    mean = np.mean([row["timing"]["stages"][name] for row in rows])
    print(f"{name:<26} {mean:>10.3f} ms")

print("\nSuccessful requests / loop second:", round(len(rows) / wall_seconds, 3))
print(
    "Server sampled peak RSS MiB:",
    round(result["server_sampled_peak_rss_bytes"] / 1024**2, 2),
)
print("PASS: Updated HTTP and stage measurements completed.")
print("No performance acceptance decision made.")

## 50. Summarize Updated HTTP and Stage Measurements

Report HTTP latency separately for each cohort. Distinguish stage averages
across all requests from averages conditional on segmentation being invoked.

M5 timing is accumulated across all pothole crops in a request.
It is not necessarily the latency of one segmentation call.

These measurements characterize the updated service under the recorded
conditions. Differences from earlier runs do not establish causation.

In [ ]:
from pathlib import Path
import json
import numpy as np

measurement_path = Path(V3_BENCHMARK_DIR) / "measurements.json"
measurements = json.loads(
    measurement_path.read_text(encoding="utf-8")
)

rows = measurements["requests"]

if (
    measurements["interrupted"]
    or len(rows) != measurements["expected_requests"]
    or not all(row["success"] for row in rows)
    or measurements["sampling_errors"]
):
    raise RuntimeError("Complete, successful measurements are required.")

with_m5 = [row for row in rows if row["segmentation_calls"] > 0]
without_m5 = [row for row in rows if row["segmentation_calls"] == 0]


def latency_summary(selected):
    if not selected:
        return {"requests": 0}

    values = np.asarray(
        [row["latency_ms"] for row in selected], dtype=float
    )
    p50, p95, p99 = np.percentile(values, [50, 95, 99])

    return {
        "requests": len(selected),
        "unique_images": len({row["image"] for row in selected}),
        "mean_ms": float(values.mean()),
        "p50_ms": float(p50),
        "p95_ms": float(p95),
        "p99_ms": float(p99),
    }


groups = {
    "general": [row for row in rows if row["cohort"] == "general"],
    "d40_diagnostic": [
        row for row in rows if row["cohort"] == "d40_diagnostic"
    ],
    "M5_skipped": without_m5,
    "M5_invoked": with_m5,
}

stage_names = list(rows[0]["timing"]["stages"])
stage_summary = {}

for name in stage_names:
    stage_summary[name] = {
        "mean_all_requests_ms": float(np.mean([
            row["timing"]["stages"][name] for row in rows
        ])),
        "mean_M5_invoked_requests_ms": (
            float(np.mean([
                row["timing"]["stages"][name] for row in with_m5
            ]))
            if with_m5 else None
        ),
    }

total_m5_calls = sum(row["segmentation_calls"] for row in rows)
total_m5_ms = sum(
    row["timing"]["stages"]["m5_predict"] for row in rows
)

summary = {
    "purpose": "updated_HTTP_and_stage_benchmark_summary",
    "backend": "PyTorch",
    "report_schema_version": "1.2",
    "artifact_hashes": measurements["artifact_hashes"],
    "successful_requests": len(rows),
    "failed_requests": 0,
    "latency_groups": {
        name: latency_summary(selected)
        for name, selected in groups.items()
    },
    "stage_times": stage_summary,
    "M5_invoked_requests": len(with_m5),
    "M5_skipped_requests": len(without_m5),
    "total_M5_calls": total_m5_calls,
    "aggregate_M5_time_per_call_ms": (
        total_m5_ms / total_m5_calls if total_m5_calls else None
    ),
    "successful_requests_per_loop_second": (
        len(rows) / measurements["measurement_loop_wall_seconds"]
    ),
    "server_baseline_rss_mib": (
        measurements["server_baseline_rss_bytes"] / 1024**2
    ),
    "server_sampled_peak_rss_mib": (
        measurements["server_sampled_peak_rss_bytes"] / 1024**2
    ),
    "performance_acceptance_assessed": False,
    "notes": [
        "HTTP latency includes receipt of the full response body.",
        "Loop throughput includes client-side response validation.",
        "M5 stage time accumulates all segmentation calls per request.",
        "Per-call M5 mean is aggregate duration divided by call count.",
        "No per-call latency percentiles were recorded.",
        "Memory is sampled server-process RSS.",
        "Three repeats per image are not independent image samples.",
    ],
}

summary_path = Path(V3_BENCHMARK_DIR) / "summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Requests with M5:", len(with_m5))
print("Requests without M5:", len(without_m5))
print("Total M5 calls:", total_m5_calls)

if with_m5:
    print(
        "Mean accumulated M5 time when invoked:",
        f"{stage_summary['m5_predict']['mean_M5_invoked_requests_ms']:.3f} ms",
    )
    print(
        "Aggregate M5 time per call:",
        f"{summary['aggregate_M5_time_per_call_ms']:.3f} ms",
    )

print("\nGroup                 p50 ms     p95 ms     p99 ms")
for name, result in summary["latency_groups"].items():
    if result["requests"]:
        print(
            f"{name:<20}"
            f" {result['p50_ms']:>10.2f}"
            f" {result['p95_ms']:>10.2f}"
            f" {result['p99_ms']:>10.2f}"
        )

print("\nSaved:", summary_path)
print("PASS: Updated benchmark summary saved.")

## 51. Benchmark Backends in Separate CPU Processes

Run each backend in a separate fresh Python process using the same images,
warm-up cases, schedule, and explicit rect=False settings.

Measure complete pipeline-call latency, sequential loop throughput, per-stage
timing, and sampled process RSS. RSS includes loaded models, libraries, and
preloaded images; it is not model-only memory.

Use the installed runtime thread defaults and record that they are not
normalized. This comparison uses Ultralytics wrappers for both backends.
It is not an HTTP or concurrent-load benchmark.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import subprocess
import sys
import time

run_root = Path(RUN_DIR).resolve()
active_bundle = Path(ACTIVE_SERVICE_BUNDLE).resolve()
onnx_root = Path(ONNX_DIR).resolve()

# Reuse the fixed workload and the currently verified service settings.
workload = json.loads(
    (run_root / "http_benchmark" / "protocol.json").read_text(
        encoding="utf-8"
    )
)
config = json.loads(
    (active_bundle / "config.json").read_text(encoding="utf-8")
)

model_files = {
    "PyTorch": {
        "m4": str((Path(PROJECT_ROOT) / config["m4"]["checkpoint"]).resolve()),
        "m5": str((Path(PROJECT_ROOT) / config["m5"]["checkpoint"]).resolve()),
    },
    "ONNX": {
        "m4": str(onnx_root / "m4" / "m4.onnx"),
        "m5": str(onnx_root / "m5" / "m5.onnx"),
    },
}

model_hashes = {}
for backend, models in model_files.items():
    model_hashes[backend] = {}
    for name, filename in models.items():
        with Path(filename).open("rb") as stream:
            model_hashes[backend][name] = hashlib.file_digest(
                stream, "sha256"
            ).hexdigest()

for name in ("m4", "m5"):
    if model_hashes["PyTorch"][name] != config[name]["sha256"]:
        raise ValueError(f"PyTorch checkpoint fingerprint mismatch: {name}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
ISOLATED_BENCHMARK_DIR = run_root / f"isolated_backend_benchmark_{stamp}"
ISOLATED_BENCHMARK_DIR.mkdir(exist_ok=False)

isolated_protocol = {
    "purpose": "isolated_CPU_backend_benchmark",
    "backend_order": ["PyTorch", "ONNX"],
    "bundle": str(active_bundle),
    "dataset_root": str(Path(M4_DIR).resolve()),
    "models": model_files,
    "model_sha256": model_hashes,
    "pipeline_sha256": hashlib.sha256(
        (active_bundle / "pipeline.py").read_bytes()
    ).hexdigest(),
    "m4_settings": {**config["m4"]["settings"], "rect": False},
    "m5_settings": {**config["m5"]["settings"], "rect": False},
    "cases": workload["cases"],
    "warmup_case_indices": workload["warmup_case_indices"],
    "schedule": workload["schedule"],
    "rss_sampling_interval_seconds": 0.05,
    "thread_policy": "Installed runtime defaults; not normalized",
    "memory_scope": "Process RSS including models, libraries and preloaded inputs",
    "latency_scope": "Complete pipeline function call",
    "throughput_scope": "Successful calls divided by measurement-loop wall time",
    "acceptance_targets": None,
}

protocol_path = ISOLATED_BENCHMARK_DIR / "protocol.json"
protocol_path.write_text(
    json.dumps(isolated_protocol, indent=2) + "\n",
    encoding="utf-8",
)

worker_source = r'''
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import platform
import sys
import threading
import time

import cv2
import numpy as np
import psutil
from ultralytics import YOLO

protocol_path = Path(sys.argv[1])
backend = sys.argv[2]
output_path = Path(sys.argv[3])
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

bundle = Path(protocol["bundle"])
pipeline_file = bundle / "pipeline.py"
if hashlib.sha256(pipeline_file.read_bytes()).hexdigest() != protocol["pipeline_sha256"]:
    raise ValueError("Pipeline fingerprint mismatch.")

sys.path.insert(0, str(bundle))
from pipeline import run_integrated_inference

models = {}
for name, task in (("m4", "detect"), ("m5", "segment")):
    path = Path(protocol["models"][backend][name])
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    if digest != protocol["model_sha256"][backend][name]:
        raise ValueError(f"Model fingerprint mismatch: {name}")
    models[name] = YOLO(str(path), task=task)

images = []
for case in protocol["cases"]:
    raw = (Path(protocol["dataset_root"]) / case["image"]).read_bytes()
    if hashlib.sha256(raw).hexdigest() != case["image_sha256"]:
        raise ValueError("Input image fingerprint mismatch.")
    image = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("Unreadable input.")
    images.append(image)

def infer(index):
    return run_integrated_inference(
        images[index], models["m4"], models["m5"],
        protocol["m4_settings"], protocol["m5_settings"],
        device="cpu",
    )

for index in protocol["warmup_case_indices"]:
    output = infer(index)
    del output

process = psutil.Process()
baseline_rss = int(process.memory_info().rss)
samples = []
sampling_errors = []
stop = threading.Event()
origin = time.perf_counter()

def sample_rss():
    while not stop.is_set():
        try:
            samples.append({
                "elapsed_seconds": time.perf_counter() - origin,
                "rss_bytes": int(process.memory_info().rss),
            })
        except Exception as exc:
            sampling_errors.append(str(exc))
            return
        stop.wait(protocol["rss_sampling_interval_seconds"])

sampler = threading.Thread(target=sample_rss, daemon=True)
rows = []
sampler.start()
loop_start = time.perf_counter()

try:
    for number, item in enumerate(protocol["schedule"], 1):
        index = item["case_index"]
        case = protocol["cases"][index]
        row = {
            "repeat": item["repeat"],
            "image": case["image"],
            "cohort": case["cohort"],
            "success": False,
            "latency_ms": None,
            "error": None,
        }
        started = time.perf_counter()
        try:
            output = infer(index)
            row["latency_ms"] = (time.perf_counter() - started) * 1000
            report, overlay, mask = output
            row.update({
                "success": True,
                "timing": report["timing"],
                "detection_count": report["detection_count"],
                "segmentation_calls": report["segmentation_calls"],
                "union_mask_pixels": report["union_mask_pixels"],
            })
            del output, report, overlay, mask
        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"
        rows.append(row)

        if number % 30 == 0:
            print(f"{backend}: {number}/{len(protocol['schedule'])}", flush=True)

finally:
    wall_seconds = time.perf_counter() - loop_start
    stop.set()
    sampler.join(timeout=5)

    result = {
        "backend": backend,
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": {
            name: metadata.version(name)
            for name in ("ultralytics", "torch", "onnxruntime", "numpy")
        },
        "protocol_sha256": hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
        "pipeline_sha256": protocol["pipeline_sha256"],
        "model_sha256": protocol["model_sha256"][backend],
        "expected_calls": len(protocol["schedule"]),
        "successful_calls": sum(row["success"] for row in rows),
        "measurement_loop_wall_seconds": wall_seconds,
        "baseline_rss_bytes": baseline_rss,
        "sampled_peak_rss_bytes": max(
            [baseline_rss] + [sample["rss_bytes"] for sample in samples]
        ),
        "sampling_errors": sampling_errors,
        "rss_samples": samples,
        "requests": rows,
    }
    output_path.write_text(
        json.dumps(result, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

if (
    result["successful_calls"] != result["expected_calls"]
    or sampling_errors
):
    raise RuntimeError("Incomplete or failed backend measurements.")

print(f"PASS: {backend} measurements saved.", flush=True)
'''

worker_path = ISOLATED_BENCHMARK_DIR / "benchmark_worker.py"
compile(worker_source, str(worker_path), "exec")
worker_path.write_text(worker_source, encoding="utf-8")

# Stop only the updated server process owned by this notebook.
if service_v3_process.poll() is None:
    command = service_v3_process.args
    if (
        not isinstance(command, (list, tuple))
        or "uvicorn" not in command
        or "8766" not in command
    ):
        raise RuntimeError("Could not verify updated server process identity.")

    service_v3_process.terminate()
    try:
        service_v3_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        service_v3_process.kill()
        service_v3_process.wait(timeout=10)

print("Updated HTTP service stopped for isolated measurements.")

for backend in isolated_protocol["backend_order"]:
    result_path = ISOLATED_BENCHMARK_DIR / f"{backend.lower()}_measurements.json"
    log_path = ISOLATED_BENCHMARK_DIR / f"{backend.lower()}.log"

    print(f"\nStarting fresh {backend} process...", flush=True)

    with log_path.open("wb") as log:
        worker = subprocess.Popen(
            [
                sys.executable, str(worker_path), str(protocol_path),
                backend, str(result_path),
            ],
            cwd=str(ISOLATED_BENCHMARK_DIR),
            stdout=log,
            stderr=subprocess.STDOUT,
            creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
        )

        started = time.monotonic()
        try:
            while True:
                try:
                    return_code = worker.wait(timeout=20)
                    break
                except subprocess.TimeoutExpired:
                    elapsed = time.monotonic() - started
                    print(
                        f"{backend}: running, {elapsed:.0f} seconds elapsed",
                        flush=True,
                    )
                    if elapsed > 600:
                        raise TimeoutError(f"{backend} exceeded 600 seconds.")
        except BaseException:
            if worker.poll() is None:
                worker.terminate()
                try:
                    worker.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    worker.kill()
                    worker.wait()
            raise

    if return_code != 0:
        print(log_path.read_text(encoding="utf-8", errors="replace")[-4000:])
        raise RuntimeError(f"{backend} benchmark failed. See {log_path}")

    print(f"PASS: {backend} process completed.")

# Summarize the two independently measured processes.
import numpy as np

summaries = {}

print(
    "\nBackend     Calls    p50 ms    p95 ms    p99 ms"
    "   Calls/s   Peak MiB"
)

for backend in isolated_protocol["backend_order"]:
    result = json.loads(
        (ISOLATED_BENCHMARK_DIR / f"{backend.lower()}_measurements.json")
        .read_text(encoding="utf-8")
    )
    rows = result["requests"]
    values = np.asarray([row["latency_ms"] for row in rows], dtype=float)
    p50, p95, p99 = np.percentile(values, [50, 95, 99])

    summary = {
        "calls": len(rows),
        "mean_ms": float(values.mean()),
        "p50_ms": float(p50),
        "p95_ms": float(p95),
        "p99_ms": float(p99),
        "successful_calls_per_loop_second": (
            result["successful_calls"] / result["measurement_loop_wall_seconds"]
        ),
        "baseline_rss_mib": result["baseline_rss_bytes"] / 1024**2,
        "sampled_peak_rss_mib": result["sampled_peak_rss_bytes"] / 1024**2,
        "cohorts": {},
    }

    for cohort in ("general", "d40_diagnostic"):
        selected = [
            row["latency_ms"] for row in rows if row["cohort"] == cohort
        ]
        percentiles = np.percentile(selected, [50, 95, 99])
        summary["cohorts"][cohort] = {
            "calls": len(selected),
            "mean_ms": float(np.mean(selected)),
            "p50_ms": float(percentiles[0]),
            "p95_ms": float(percentiles[1]),
            "p99_ms": float(percentiles[2]),
        }

    summaries[backend] = summary

    print(
        f"{backend:<10} {len(rows):>6}"
        f" {p50:>9.2f} {p95:>9.2f} {p99:>9.2f}"
        f" {summary['successful_calls_per_loop_second']:>9.3f}"
        f" {summary['sampled_peak_rss_mib']:>10.2f}"
    )

(ISOLATED_BENCHMARK_DIR / "summary.json").write_text(
    json.dumps({
        "purpose": "isolated_backend_benchmark_summary",
        "results": summaries,
        "mixed_workload_is_deployment_representative": False,
        "performance_acceptance_assessed": False,
        "memory_scope": isolated_protocol["memory_scope"],
        "thread_policy": isolated_protocol["thread_policy"],
    }, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("\nSaved:", ISOLATED_BENCHMARK_DIR)
print("PASS: Separate-process backend benchmarks completed.")
print("The HTTP service remains stopped.")
print("Results use an artificial 50/50 cohort mixture; cohort details are saved.")

## 52. Isolated Backend Findings and Deployment Decision

Both backends completed 540 pipeline calls in separate fresh CPU processes.
Each process used the same 180 development images, ten warm-up calls,
and three measured repeats per image.

| Backend | Calls | p50 (ms) | p95 (ms) | p99 (ms) | Calls/loop second | Sampled peak RSS (MiB) |
|---|---:|---:|---:|---:|---:|---:|
| PyTorch | 540 | 64.11 | 235.90 | 354.67 | 10.548 | 638.95 |
| ONNX Runtime | 540 | 119.33 | 616.84 | 750.14 | 5.483 | 787.49 |

### Interpretation

PyTorch had lower latency percentiles, higher measured throughput,
and lower sampled peak process memory in this experiment.

The earlier same-process comparison showed lower ONNX mean latency.
The experiments used different execution conditions; their results
must remain separate rather than being combined into a universal
backend speed claim.

These measurements cover direct pipeline calls, not HTTP requests.
Process RSS includes libraries, models, and preloaded input images.
Memory sampling may miss short-lived peaks.

The workload contains an artificial equal mixture of general and
D40-enriched development images. It does not represent deployment
prevalence. Runtime thread defaults were not normalized, and backend
execution order was fixed. Results apply to the recorded hardware,
software, inputs, and configuration.

### Deployment Decision

Retain PyTorch for the local demonstration service. This choice is
supported by the recorded HTTP functional tests and these isolated
backend measurements.

Retain ONNX as a verified export alternative with the previously
documented parity coverage. Successful benchmark execution does not
extend that parity coverage or establish prediction accuracy.

This is a backend selection decision, not production acceptance.
No predetermined quantitative performance targets were available.
Low D40 recall and unverified integrated segmentation and severity-proxy
accuracy remain limitations.

The HTTP service was stopped before this experiment and remains stopped.

---------------
## 53. Create a Development Failure Gallery

Select the first available D40 false-negative and false-positive example
per source, ordered by cohort and image path.

Match predictions in descending confidence order to unmatched reference
boxes of the same class at IoU >= 0.50. Use the recorded operating point:
confidence 0.10 and NMS IoU 0.50.

Green boxes show reference annotations. Red boxes show false positives,
orange boxes show missed references, and blue boxes show matched predictions.

These examples illustrate development failures, not their deployment
frequency. Detection boxes cannot validate segmentation or severity accuracy.
Gallery images require visual privacy review before external sharing.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

from PIL import Image, ImageDraw
from IPython.display import display

run_root = Path(RUN_DIR)
dataset_root = Path(M4_DIR)
sweep_root = Path(SWEEP_DIR)

def gallery_read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def gallery_sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

general_manifest_path = run_root / "development_manifest.json"
diagnostic_manifest_path = run_root / "d40_diagnostic_manifest.json"
prediction_path = sweep_root / "predictions_nms_0.50.json"

general_manifest = gallery_read(general_manifest_path)
diagnostic_manifest = gallery_read(diagnostic_manifest_path)
general_gt = gallery_read(run_root / "development_ground_truth.json")
cached = gallery_read(prediction_path)

assert cached["general_manifest_sha256"] == gallery_sha(general_manifest_path)
assert cached["d40_manifest_sha256"] == gallery_sha(diagnostic_manifest_path)
assert cached["settings"]["iou"] == 0.50
assert cached["settings"]["conf"] <= 0.10

CONFIDENCE = 0.10
MATCH_IOU = 0.50

def gallery_iou(a, b):
    width = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    height = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = width * height
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

evaluated = []

for cohort, records in [
    ("general", general_manifest["records"]),
    ("d40_diagnostic", diagnostic_manifest["records"]),
]:
    for record in sorted(records, key=lambda row: row["image"]):
        image_key = record["image"]
        truth = (
            general_gt[image_key]
            if cohort == "general"
            else record
        )
        gt_boxes = truth["boxes"]

        predictions = sorted(
            [
                dict(prediction)
                for prediction in cached["predictions"][cohort][image_key]
                if prediction["confidence"] >= CONFIDENCE
            ],
            key=lambda prediction: prediction["confidence"],
            reverse=True,
        )

        unmatched = set(range(len(gt_boxes)))
        matches = []

        for prediction in predictions:
            eligible = sorted(
                index for index in unmatched
                if gt_boxes[index]["class_id"] == prediction["class_id"]
            )
            best_index = None
            best_iou = 0.0

            for index in eligible:
                overlap = gallery_iou(
                    prediction["bbox_xyxy"],
                    gt_boxes[index]["bbox_xyxy"],
                )
                if overlap > best_iou:
                    best_index = index
                    best_iou = overlap

            matched = best_index is not None and best_iou >= MATCH_IOU
            if matched:
                unmatched.remove(best_index)

            matches.append({
                "prediction": prediction,
                "matched_gt_index": best_index if matched else None,
                "matched_iou": best_iou if matched else None,
                "outcome": "TP" if matched else "FP",
            })

        evaluated.append({
            "cohort": cohort,
            "source": record["source"],
            "image": image_key,
            "image_sha256": record["image_sha256"],
            "width": truth["width"],
            "height": truth["height"],
            "ground_truth": gt_boxes,
            "predictions": matches,
            "missed_gt_indices": sorted(unmatched),
            "d40_fp": sum(
                item["outcome"] == "FP"
                and item["prediction"]["class_id"] == 3
                for item in matches
            ),
            "d40_fn": sum(
                gt_boxes[index]["class_id"] == 3
                for index in unmatched
            ),
        })

# Confirm that matching reproduces the recorded operating-point counts.
for cohort, expected in [
    ("general", (3, 8, 2)),
    ("d40_diagnostic", (26, 21, 96)),
]:
    rows = [row for row in evaluated if row["cohort"] == cohort]
    tp = sum(
        item["outcome"] == "TP"
        and item["prediction"]["class_id"] == 3
        for row in rows
        for item in row["predictions"]
    )
    actual = (
        tp,
        sum(row["d40_fp"] for row in rows),
        sum(row["d40_fn"] for row in rows),
    )
    assert actual == expected, (
        f"{cohort}: matching differs from recorded results: {actual}"
    )
    print(f"PASS: {cohort} D40 TP/FP/FN = {actual}")

selected = {}
selection_log = []

for source in ["China_Drone", "China_MotorBike", "Czech"]:
    for category, field in [
        ("missed_pothole", "d40_fn"),
        ("false_positive_pothole", "d40_fp"),
    ]:
        candidates = sorted(
            [
                row for row in evaluated
                if row["source"] == source and row[field] > 0
            ],
            key=lambda row: (row["cohort"], row["image"]),
        )
        if not candidates:
            selection_log.append({
                "source": source,
                "category": category,
                "available": False,
            })
            continue

        chosen = candidates[0]
        key = (chosen["cohort"], chosen["image"])
        if key not in selected:
            selected[key] = dict(chosen, selection_reasons=[])
        selected[key]["selection_reasons"].append(category)

        selection_log.append({
            "source": source,
            "category": category,
            "available": True,
            "cohort": chosen["cohort"],
            "image": chosen["image"],
        })

# Verify selected images before creating the gallery.
for row in selected.values():
    image_path = dataset_root / row["image"]
    assert gallery_sha(image_path) == row["image_sha256"], image_path

FAILURE_GALLERY_DIR = run_root / (
    "failure_gallery_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
FAILURE_GALLERY_DIR.mkdir(parents=True, exist_ok=False)

gallery_records = []
index_lines = [
    "# Development Failure Gallery",
    "",
    "Confidence: 0.10; NMS IoU: 0.50; matching IoU: 0.50.",
    "",
    "Selected examples are illustrative, not prevalence estimates.",
    "Privacy review is pending before external sharing.",
    "",
]

for number, row in enumerate(selected.values(), start=1):
    with Image.open(dataset_root / row["image"]) as opened:
        canvas = opened.convert("RGB")

    assert canvas.size == (row["width"], row["height"])
    draw = ImageDraw.Draw(canvas)
    missed = set(row["missed_gt_indices"])

    def draw_box(box, label, color):
        x1, y1, x2, y2 = box
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        draw.text(
            (max(0, x1), max(0, y1 - 13)),
            label,
            fill=color,
            stroke_width=1,
            stroke_fill="black",
        )

    for index, box in enumerate(row["ground_truth"]):
        is_missed = index in missed
        draw_box(
            box["bbox_xyxy"],
            f"{'FN' if is_missed else 'GT'} {box['damage_code']}",
            "orange" if is_missed else "lime",
        )

    for item in row["predictions"]:
        prediction = item["prediction"]
        draw_box(
            prediction["bbox_xyxy"],
            f"{item['outcome']} {prediction['damage_code']} "
            f"{prediction['confidence']:.3f}",
            "red" if item["outcome"] == "FP" else "deepskyblue",
        )

    filename = f"{number:02d}_{Path(row['image']).stem}.png"
    output_path = FAILURE_GALLERY_DIR / filename
    canvas.save(output_path)

    saved_record = {
        **row,
        "gallery_image": filename,
        "gallery_image_sha256": gallery_sha(output_path),
        "privacy_review_status": "pending",
    }
    gallery_records.append(saved_record)

    title = (
        f"{number}. {row['source']} / {row['cohort']} — "
        f"D40 FP={row['d40_fp']}, FN={row['d40_fn']}"
    )
    print("\n" + title)
    print("Selection:", ", ".join(row["selection_reasons"]))
    display(canvas)

    index_lines.extend([
        f"## {title}",
        "",
        f"Source image: `{row['image']}`",
        "",
        f"Selection: {', '.join(row['selection_reasons'])}",
        "",
        f"![Annotated failure example]({filename})",
        "",
    ])

gallery_manifest = {
    "purpose": "development_failure_gallery",
    "confidence": CONFIDENCE,
    "nms_iou": 0.50,
    "matching_iou": MATCH_IOU,
    "selection_rule": (
        "First available D40 FN and FP example per source, "
        "ordered by cohort and image path; duplicate images merged."
    ),
    "prediction_file_sha256": gallery_sha(prediction_path),
    "general_manifest_sha256": gallery_sha(general_manifest_path),
    "diagnostic_manifest_sha256": gallery_sha(diagnostic_manifest_path),
    "privacy_review_status": "pending",
    "selection_log": selection_log,
    "examples": gallery_records,
}

(FAILURE_GALLERY_DIR / "manifest.json").write_text(
    json.dumps(gallery_manifest, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
(FAILURE_GALLERY_DIR / "README.md").write_text(
    "\n".join(index_lines) + "\n",
    encoding="utf-8",
)

print("\nSaved:", FAILURE_GALLERY_DIR)
print("Unique gallery images:", len(gallery_records))
print("PASS: Development failure gallery generated.")
print("No new inference performed. No test data accessed.")
print("Review images for visible faces or plates before external sharing.")

## 54. Prepare Failure Gallery Presentation Copies

Create presentation copies from the original images and saved matching
records. Numbered boxes refer to a separate legend.

For the selected China MotorBike image 000004, conceal the top 85 pixel
rows containing people and vehicles. This display-only redaction does
not modify inference inputs, annotations, predictions, or metrics.

Original evidence is retained. Presentation copies require a final
visual check before inclusion in the report.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

gallery_root = Path(FAILURE_GALLERY_DIR)
manifest_path = gallery_root / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

presentation_dir = gallery_root / (
    "presentation_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
presentation_dir.mkdir(parents=True, exist_ok=False)

font_path = Path(r"C:\Windows\Fonts\arial.ttf")
font = (
    ImageFont.truetype(str(font_path), 16)
    if font_path.is_file()
    else ImageFont.load_default()
)

colors = {
    "GT": "#15803d",
    "FN": "#c26700",
    "TP": "#007ca8",
    "FP": "#d00000",
}

prepared_records = []
readme = [
    "# Presentation Failure Gallery",
    "",
    "GT: matched reference; FN: missed reference;",
    "TP: matched prediction; FP: unmatched prediction.",
    "",
    "Matching uses same-class IoU >= 0.50.",
    "These selected development examples do not estimate deployment frequency.",
    "",
]

def rectangles_intersect(a, b):
    return (
        a[0] < b[2] and a[2] > b[0]
        and a[1] < b[3] and a[3] > b[1]
    )

for number, record in enumerate(manifest["examples"], start=1):
    source_path = Path(M4_DIR) / record["image"]
    source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
    assert source_hash == record["image_sha256"], source_path

    with Image.open(source_path) as opened:
        picture = opened.convert("RGB")

    width, height = picture.size
    drawing = ImageDraw.Draw(picture)
    redactions = []

    if source_path.name == "China_MotorBike_China_MotorBike_000004.jpg":
        evaluated_boxes = [
            box["bbox_xyxy"]
            for box in record["ground_truth"]
        ] + [
            item["prediction"]["bbox_xyxy"]
            for item in record["predictions"]
        ]

        assert all(box[1] >= 85 for box in evaluated_boxes), (
            "Redaction would cover an evaluated box."
        )

        drawing.rectangle(
            (0, 0, width - 1, 84),
            fill="#202020",
        )
        drawing.text(
            (10, 25),
            "People and vehicles concealed",
            font=font,
            fill="white",
        )
        redactions.append({
            "rectangle_xyxy_exclusive": [0, 0, width, 85],
            "method": "opaque_fill",
            "purpose": "conceal_people_and_vehicles",
        })

    missed = set(record["missed_gt_indices"])
    entries = []

    for index, box in enumerate(record["ground_truth"]):
        entries.append({
            "outcome": "FN" if index in missed else "GT",
            "class": box["damage_code"],
            "box": box["bbox_xyxy"],
            "confidence": None,
        })

    for item in record["predictions"]:
        prediction = item["prediction"]
        entries.append({
            "outcome": item["outcome"],
            "class": prediction["damage_code"],
            "box": prediction["bbox_xyxy"],
            "confidence": prediction["confidence"],
        })

    # Draw box outlines first.
    for entry in entries:
        drawing.rectangle(
            tuple(entry["box"]),
            outline=colors[entry["outcome"]],
            width=3,
        )

    placed_labels = []

    for label_number, entry in enumerate(entries, start=1):
        color = colors[entry["outcome"]]
        x1, y1, x2, y2 = entry["box"]
        text = str(label_number)

        bounds = drawing.textbbox((0, 0), text, font=font)
        label_width = bounds[2] - bounds[0] + 10
        label_height = 23

        anchor_x = max(
            0,
            min(int(x1), width - label_width),
        )
        # Place the number above the box, not over the defect.
        anchor_y = max(
            0,
            min(int(y1) - label_height - 3, height - label_height),
        )

        chosen = None

        for offset in range(0, height, label_height + 3):
            for direction in (-1, 1):
                candidate_y = max(
                    0,
                    min(
                        anchor_y + direction * offset,
                        height - label_height,
                    ),
                )
                candidate = (
                    anchor_x,
                    candidate_y,
                    anchor_x + label_width,
                    candidate_y + label_height,
                )

                # Keep labels clear of each other and their own box.
                overlaps_label = any(
                    rectangles_intersect(candidate, previous)
                    for previous in placed_labels
                )
                overlaps_own_box = rectangles_intersect(
                    candidate, (x1, y1, x2, y2)
                )

                if not overlaps_label and not overlaps_own_box:
                    chosen = candidate
                    break

            if chosen is not None:
                break

        if chosen is None:
            raise RuntimeError(
                f"No free label position found for example {number}, "
                f"box {label_number}."
            )

        placed_labels.append(chosen)

        drawing.line(
            (
                chosen[0] + label_width / 2,
                chosen[1] + label_height / 2,
                x1,
                y1,
            ),
            fill=color,
            width=1,
        )
        drawing.rectangle(chosen, fill=color)
        drawing.text(
            (
                chosen[0] + 5 - bounds[0],
                chosen[1] + (label_height - (bounds[3] - bounds[1])) / 2
                - bounds[1],
            ),
            text,
            font=font,
            fill="white",
        )
        entry["number"] = label_number

    panel_width = max(width, 700)
    header_height = 68
    legend_height = 32 + 26 * len(entries)

    panel = Image.new(
        "RGB",
        (panel_width, header_height + height + legend_height),
        "white",
    )
    panel.paste(picture, (0, header_height))
    panel_draw = ImageDraw.Draw(panel)

    panel_draw.text(
        (12, 8),
        f"Example {number}: {record['source']} / {record['cohort']}",
        font=font,
        fill="black",
    )
    panel_draw.text(
        (12, 34),
        f"D40 false positives: {record['d40_fp']} | "
        f"D40 missed references: {record['d40_fn']}",
        font=font,
        fill="black",
    )

    for index, entry in enumerate(entries):
        confidence_text = (
            ""
            if entry["confidence"] is None
            else f" | confidence={entry['confidence']:.3f}"
        )

        panel_draw.text(
            (12, header_height + height + 12 + 26 * index),
            f"{entry['number']}: {entry['outcome']} "
            f"{entry['class']}{confidence_text}",
            font=font,
            fill=colors[entry["outcome"]],
        )

    filename = f"{number:02d}_{source_path.stem}_presentation.png"
    output_path = presentation_dir / filename
    panel.save(output_path)

    prepared_records.append({
        "source_image": record["image"],
        "source_image_sha256": source_hash,
        "presentation_image": filename,
        "presentation_image_sha256": hashlib.sha256(
            output_path.read_bytes()
        ).hexdigest(),
        "redactions": redactions,
        "legend": entries,
        "final_visual_review": "pending",
    })

    readme.extend([
        f"## Example {number}: {record['source']}",
        "",
        f"![Failure example {number}]({filename})",
        "",
    ])

    display(panel)

(presentation_dir / "manifest.json").write_text(
    json.dumps(
        {
            "purpose": "failure_gallery_presentation",
            "source_manifest_sha256": hashlib.sha256(
                manifest_path.read_bytes()
            ).hexdigest(),
            "inference_inputs_modified": False,
            "metrics_modified": False,
            "examples": prepared_records,
        },
        indent=2,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

(presentation_dir / "README.md").write_text(
    "\n".join(readme) + "\n",
    encoding="utf-8",
)

print("Saved:", presentation_dir)
print("Presentation images:", len(prepared_records))
print("PASS: Separate presentation copies created.")
print("Original images, predictions, and metrics unchanged.")
print("Final visual review pending.")

## 55. Failure Gallery Findings

The six selected examples come from the D40 diagnostic cohort.
They illustrate failure types and are not representative estimates
of their frequency in deployment.

### Observed Failures

- **China Drone, image 000057:** two reference potholes were missed.
- **China Drone, image 000487:** one pothole was matched, while two
  additional D40 predictions were counted as false positives.
  Overlapping predictions illustrate the effect of one-to-one matching.
- **China MotorBike, image 000004:** a small reference pothole was missed.
  Its limited image footprint is a possible contributing factor,
  not an experimentally established cause.
- **China MotorBike, image 000163:** a prediction overlapped part of
  the reference pothole but did not satisfy the matching threshold.
  It therefore contributed both one false positive and one false negative.
  This illustrates a localization failure.
- **Czech, image 000270:** the reference pothole was missed while other
  damage predictions were present.
- **Czech, image 001251:** one D40 prediction matched a reference pothole,
  while another D40 prediction was unmatched.

### Integration Consequences

M5 runs only on M4-predicted D40 crops. Missed D40 detections may therefore
prevent segmentation of actual potholes. False-positive D40 detections
may trigger unnecessary segmentation.

A nonempty M5 mask does not independently confirm that a predicted region
is a pothole. Detection reference boxes cannot establish mask accuracy
or severity-proxy accuracy.

### Presentation Review

The six presentation copies were visually inspected.
The selected MotorBike image 000004 has its top 85 pixel rows concealed
to hide people and vehicles. The reference pothole remains visible.
Numbered labels and separate legends improve readability.

These are display-only changes. Original images, inference inputs,
predictions, reference annotations, and evaluation metrics are unchanged.
This review applies only to these six presentation copies; the service
does not automatically anonymize uploaded images.

------------------
## 56. Record the Integrated System Model Card

Document the intended use, models, development evaluation, performance,
output interpretation, and remaining limitations of the schema 1.2
PyTorch prototype.

This card describes the integrated system. Backend agreement and
functional checks do not establish prediction accuracy.

In [ ]:
from pathlib import Path
import hashlib
import json
from IPython.display import display, Markdown

active_bundle = Path(ACTIVE_SERVICE_BUNDLE).resolve()
config_path = active_bundle / "config.json"
pipeline_path = active_bundle / "pipeline.py"

assert config_path.is_file(), config_path
assert pipeline_path.is_file(), pipeline_path

model_card = """# M8 Integrated Road Damage System — Model Card

## System and Intended Use

Contributor: Ghassan Saad AlGhaidani — 225495.

This educational prototype combines an M4 YOLO11n road-damage detector
with an M5 YOLO11n segmentation model. The selected local HTTP backend
is PyTorch on CPU, with report schema version 1.2.

The intended use is a supervised demonstration of road-damage detection,
pothole segmentation, image-area reporting, and inference timing.
Outputs require human interpretation.

The system is not approved for autonomous maintenance prioritization,
road-safety certification, or engineering severity assessment.

## Models and Processing

M4 detects D00, D10, D20, and D40. Only predicted D40 regions trigger M5,
whose class 0 represents potholes. Other damage detections are retained.

M4 uses image size 640, confidence 0.10, NMS IoU 0.50, and max_det 300.
M5 uses image size 640, confidence 0.05, NMS IoU 0.70, and max_det 100.

M5 crops are resized to a 640-by-640 square using INTER_AREA.
Binary masks are restored to crop dimensions using nearest-neighbor
interpolation and placed in original-image coordinates.
Overlapping masks are unioned without double-counting pixels.

M8 integrates existing checkpoints; it does not retrain the models.
Training provenance and original model evaluations belong to the
respective M4 and M5 handoffs.

## Outputs and Severity Interpretation

The HTTP response includes a prediction report, model fingerprints,
an annotated PNG, and a binary union-mask PNG.

Reports include detection classes, confidence scores, bounding boxes,
segmentation status, predicted mask areas, and CPU stage timings.

For a D40 detection with a nonempty predicted mask, the severity proxy is:

100 × predicted crop-mask pixels / full-image pixels.

This is an image-area proxy. It is not physical area, depth, or an
engineering severity category. It depends on camera position,
perspective, framing, and segmentation quality.

Missing masks produce unavailable severity rather than zero severity.
Crack severity is explicitly unsupported. Per-detection mask areas can
overlap; the overall union area is calculated separately.

## Development Evaluation

The general subset contains 90 validation images: 30 each from
China Drone, China MotorBike, and Czech. It contains 21 verified
background images and 132 reference boxes, including five D40 boxes.

A separate D40-enriched diagnostic cohort contains 90 additional images
and 122 D40 reference boxes. It is not representative of deployment
class prevalence.

At the selected operating point and same-class matching IoU >= 0.50:

| Cohort | Micro-F1 | D40 TP | D40 FP | D40 FN | D40 precision | D40 recall |
|---|---:|---:|---:|---:|---:|---:|
| General | 0.350 | 3 | 8 | 2 | 0.273 | 0.600 |
| D40 diagnostic | 0.258 | 26 | 21 | 96 | 0.553 | 0.213 |

These data informed operating-point selection. Results are development
measurements, not unbiased final-test performance or AP/mAP.
No Norway final-test data was used in the documented M8 experiments.

No reference segmentation masks or severity labels were used to measure
accuracy of the integrated pipeline on these cohorts.

## Functional Verification and ONNX

The updated pipeline passed 27 controlled edge-case checks.
The updated HTTP service passed its recorded smoke checks and six input
validation checks.

Both models were exported to ONNX. Earlier checks passed 18 raw-output
comparisons on six images, pipeline comparisons on seven images, and
union-mask comparisons on 47 fixed crops.

Those checks have their recorded version and sample scope. They do not
prove universal backend equivalence, model accuracy, or complete parity
of every field introduced by later reporting changes.

## Performance

The updated local PyTorch HTTP service completed 540 measured requests
with concurrency 1 after ten warm-up requests.

| Cohort | p50 ms | p95 ms | p99 ms |
|---|---:|---:|---:|
| General | 95.63 | 193.88 | 312.61 |
| D40 diagnostic | 120.06 | 308.97 | 412.41 |

Successful requests per measurement-loop second: 7.054.
Sampled peak server RSS: 518.46 MiB.

A separate fresh-process direct-pipeline comparison measured:

| Backend | Calls | p50 ms | p95 ms | p99 ms | Calls/loop second | Peak RSS MiB |
|---|---:|---:|---:|---:|---:|---:|
| PyTorch | 540 | 64.11 | 235.90 | 354.67 | 10.548 | 638.95 |
| ONNX Runtime | 540 | 119.33 | 616.84 | 750.14 | 5.483 | 787.49 |

HTTP and direct-pipeline measurements have different scopes.
The backend comparison used an artificial 50/50 cohort mixture,
fixed backend order, and unnormalized runtime thread defaults.
Worker RSS includes libraries, models, and preloaded images.
Sampling may miss short-lived memory peaks.

Results apply to the recorded Intel Core Ultra 7 155H CPU environment.
Concurrent-load capacity and production performance were not established.

## Limitations and Failure Modes

- Low D40 recall can prevent M5 from seeing actual potholes.
- False-positive D40 detections can trigger segmentation of unrelated regions.
- Partial or duplicate detections can fail one-to-one box matching.
- A nonempty predicted mask does not confirm a true pothole.
- Image-area proxies are not calibrated across camera viewpoints.
- Clean-environment dependency installation remains unverified.
- The local HTTP prototype has not been validated for public deployment.
- Uploaded images are not automatically anonymized.

The failure gallery documents selected development examples.
Its presentation redaction applies only to those prepared images.

## Acceptance Status

PyTorch is retained for the local supervised demonstration.
This is not production approval.

Predetermined quantitative acceptance targets were not available for the
recorded benchmarks; retrospective measurements must not be presented
as passing criteria defined in advance.

Operational deployment remains NO-GO pending an agreed prospective
evaluation protocol, independent accuracy assessment, appropriate
severity validation, and deployment-specific reliability requirements.
"""

card_path = Path(RUN_DIR) / "model_card_v3.md"
card_path.write_text(model_card.strip() + "\n", encoding="utf-8")

provenance = {
    "purpose": "integrated_system_model_card",
    "report_schema_version": "1.2",
    "backend": "PyTorch",
    "active_bundle": str(active_bundle),
    "pipeline_sha256": hashlib.sha256(
        pipeline_path.read_bytes()
    ).hexdigest(),
    "config_sha256": hashlib.sha256(
        config_path.read_bytes()
    ).hexdigest(),
    "model_card_sha256": hashlib.sha256(
        card_path.read_bytes()
    ).hexdigest(),
    "metrics_source": "Recorded development experiment summaries",
    "production_approved": False,
}

provenance_path = Path(RUN_DIR) / "model_card_v3_provenance.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

display(Markdown(model_card))
print("Saved:", card_path)
print("Saved:", provenance_path)
print("PASS: Model card and artifact fingerprints recorded.")
print("No inference performed. Service files unchanged.")

## 57. Document the Active Service Bundle

Copy the recorded dependency files and model card into the active
schema 1.2 service bundle. Write execution instructions matching its
actual HTTP interface and configuration.

Verify that service code and configuration remain unchanged.
Dependency records describe the observed environment; they do not
establish successful installation in a clean environment.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil

run_root = Path(RUN_DIR).resolve()
active_bundle = Path(ACTIVE_SERVICE_BUNDLE).resolve()
original_bundle = run_root / "service_bundle"

assert active_bundle.is_relative_to(run_root)
assert active_bundle != original_bundle


def documentation_sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


protected_names = [
    "pipeline.py",
    "app.py",
    "config.json",
    "launch.local.json",
]

for name in protected_names:
    assert (active_bundle / name).is_file(), name

before = {
    name: documentation_sha(active_bundle / name)
    for name in protected_names
}

config = json.loads(
    (active_bundle / "config.json").read_text(encoding="utf-8")
)

assert config["schema_version"] == "1.2"
assert config["device"] == "cpu"

copy_sources = {
    "requirements-service.txt":
        original_bundle / "requirements-service.txt",
    "requirements-verification.txt":
        original_bundle / "requirements-verification.txt",
    "environment_inventory.json":
        original_bundle / "environment_inventory.json",
    "MODEL_CARD.md":
        run_root / "model_card_v3.md",
    "model_card_provenance.json":
        run_root / "model_card_v3_provenance.json",
}

for source in copy_sources.values():
    assert source.is_file(), f"Missing file: {source}"

for destination_name, source in copy_sources.items():
    destination = active_bundle / destination_name
    shutil.copy2(source, destination)

    assert documentation_sha(source) == documentation_sha(destination)
    print("Copied:", destination_name)

readme = r"""
# M8 Road Damage HTTP Prototype

## Version and Scope

This bundle contains the PyTorch CPU pipeline with report schema 1.2,
including pothole image-area proxies and per-stage timings.

The FastAPI application version remains 0.1.0. Application version and
prediction-report schema version are separate identifiers.

Read MODEL_CARD.md for evaluation results, intended use, and limitations.
This bundle is intended for a supervised local demonstration.
Operational deployment is not approved.

## Required External Models

Models and datasets are not included in the service-only package.

Set M8_PROJECT_ROOT to the directory containing Drive_assests.
The checkpoint paths below are relative to that root.
Startup verifies their SHA-256 fingerprints.

__CHECKPOINTS__

Preserve this directory structure when moving the bundle.
Set M8_PROJECT_ROOT explicitly after relocation. The launch.local.json
fallback describes the original local layout and may not remain valid.

## Dependencies

requirements-service.txt records observed direct service dependencies.
environment_inventory.json records the original inspected environment.

These records are not a complete dependency lock. Installation in a
clean environment has not been verified. PyTorch wheel availability
depends on the Python version, operating system, and package index.

The following is a proposed clean-environment setup, not a recorded
successful installation. Run it from the extracted bundle directory:

```powershell
py -3.14 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements-service.txt
```

ONNX export and verification tools are optional for the PyTorch service:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-verification.txt
```

## Start the Service

From the bundle directory, set the project root and use the Python
interpreter containing the required dependencies:

```powershell
$env:M8_PROJECT_ROOT = "C:\path\to\phase2"
python -m uvicorn app:app --host 127.0.0.1 --port 8766 --workers 1
```

For a successfully prepared virtual environment, replace `python` with
`.\.venv\Scripts\python.exe`.

Interactive API: http://127.0.0.1:8766/docs

Stop a foreground server with Ctrl+C. Do not start a second server on
the same port. Keep the service bound to localhost for this prototype.

## Health Check

```powershell
curl.exe --fail-with-body http://127.0.0.1:8766/health
```

GET /health returns readiness, CPU device, model fingerprints,
and upload limits. Successful startup verifies model tasks and classes.

## Predict

```powershell
curl.exe --fail-with-body -X POST `
  "http://127.0.0.1:8766/predict" `
  -F "file=@C:\path\to\example.png" `
  -o response.json
```

The upload field must be named `file`.

Supported input: readable single-frame JPEG or PNG.
Maximum encoded upload: 10 MiB.
Maximum decoded image size: 12,000,000 pixels.

Expected validation responses:

| Condition | HTTP status |
|---|---:|
| Missing file field | 422 |
| Empty or invalid image | 400 |
| Unsupported format | 415 |
| Byte or pixel limit exceeded | 413 |

## Response Structure

The JSON response contains:

- `report`: detection, segmentation, area-proxy, and timing results.
- `model_fingerprints`: SHA-256 fingerprints for M4 and M5.
- `annotated_png_base64`: base64-encoded annotated PNG.
- `mask_png_base64`: base64-encoded binary PNG, using values 0 and 255.

Bounding boxes use original-image pixel coordinates.

M4 retains all four damage classes. M5 runs only on predicted D40 crops.
No D40 detection means segmentation is skipped; it does not prove that
the road contains no potholes.

For a nonempty D40 mask, the severity proxy is predicted mask pixels
divided by full-image pixels, multiplied by 100. It is not physical
area, depth, or an engineering severity category.

Missing masks produce unavailable severity, not zero severity.
Crack severity is unsupported. Global union area avoids counting
overlapping mask pixels more than once.

Stage timings are CPU wall-clock measurements within the pipeline.
They exclude model loading, upload/file decoding, PNG encoding,
and HTTP transport. They are not end-to-end HTTP latency.

## Decode the Returned Images

Run this Python snippet beside response.json:

```python
from pathlib import Path
import base64
import json

response = json.loads(
    Path("response.json").read_text(encoding="utf-8")
)

for key, filename in [
    ("annotated_png_base64", "annotated.png"),
    ("mask_png_base64", "mask.png"),
]:
    Path(filename).write_bytes(
        base64.b64decode(response[key], validate=True)
    )
```

## Evidence and Limitations

Development reports, benchmark measurements, ONNX artifacts, and the
failure gallery are maintained separately under the experiment directory.

Previous-version results retain their original scope. Consult the model
card and evidence manifests before attributing a result to this version.

Low D40 recall limits detector-gated segmentation. Integrated mask and
severity-proxy accuracy remain unverified on the development cohorts.
No independent final-test evaluation or clean-install verification is
claimed.

The service does not automatically hide faces or license plates.
Prepared gallery redactions apply only to those presentation images.
Review and anonymize images before external sharing.

The prototype has not been validated for public hosting or concurrent
production workloads.
"""

checkpoint_lines = []

for name in ("m4", "m5"):
    checkpoint_lines.extend([
        f"### {name.upper()}",
        "",
        f"Path: `{config[name]['checkpoint']}`",
        "",
        f"SHA-256: `{config[name]['sha256']}`",
        "",
    ])

readme = readme.replace(
    "__CHECKPOINTS__",
    "\n".join(checkpoint_lines),
)

readme_path = active_bundle / "README.md"
readme_path.write_text(
    readme.strip() + "\n",
    encoding="utf-8",
)

after = {
    name: documentation_sha(active_bundle / name)
    for name in protected_names
}

assert before == after, "Service code or configuration changed."

document_names = [
    *copy_sources.keys(),
    "README.md",
]

documentation_record = {
    "purpose": "active_service_documentation",
    "report_schema_version": "1.2",
    "backend": "PyTorch",
    "protected_artifacts_unchanged": before == after,
    "protected_artifact_sha256": after,
    "documentation_sha256": {
        name: documentation_sha(active_bundle / name)
        for name in document_names
    },
    "clean_install_verified": False,
    "server_started_by_this_cell": False,
}

record_path = active_bundle / "documentation_manifest.json"
record_path.write_text(
    json.dumps(
        documentation_record,
        indent=2,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

print("\nSaved:", readme_path)
print("Saved:", record_path)
print("PASS: Active bundle documentation prepared.")
print("PASS: Service code and configuration unchanged.")
print("No packages installed. No server started.")

## 58. Verify the Updated Pipeline in a Fresh Python Process

Run the active schema 1.2 pipeline in a separate Python process using
the current installed environment and the saved configuration.

Compare its prediction report with the recorded HTTP smoke-test report,
excluding variable timing measurements. Allow only the previously
documented small numerical tolerances for boxes and confidence scores.

Compare the mask and annotated image exactly with the notebook reference.
Validate timing values separately.

This verifies the selected example in a fresh process, not installation
in a clean environment or prediction accuracy.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

import cv2
import numpy as np

required_variables = [
    "RUN_DIR",
    "ACTIVE_SERVICE_BUNDLE",
    "PROJECT_ROOT",
    "integration_bgr",
    "timed_mask",
    "timed_overlay",
]
missing = [name for name in required_variables if name not in globals()]
assert not missing, f"Missing notebook variables: {missing}"

bundle = Path(ACTIVE_SERVICE_BUNDLE).resolve()
root = Path(PROJECT_ROOT).resolve()

verification_dir = Path(RUN_DIR) / (
    "fresh_process_v3_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
verification_dir.mkdir(parents=True, exist_ok=False)

reference_report_path = bundle / "http_smoke_report.json"
assert reference_report_path.is_file()

reference_report = json.loads(
    reference_report_path.read_text(encoding="utf-8")
)
assert reference_report["schema_version"] == "1.2"

assert integration_bgr.dtype == np.uint8
assert integration_bgr.shape[:2] == timed_mask.shape
assert integration_bgr.shape == timed_overlay.shape

for filename, array in [
    ("input.png", integration_bgr),
    ("reference_mask.png", timed_mask.astype(np.uint8) * 255),
    ("reference_overlay.png", timed_overlay),
]:
    success, encoded = cv2.imencode(".png", array)
    assert success, filename
    (verification_dir / filename).write_bytes(encoded.tobytes())

(verification_dir / "reference_report.json").write_text(
    json.dumps(reference_report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

def fresh_sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

job = {
    "bundle": str(bundle),
    "project_root": str(root),
    "pipeline_sha256": fresh_sha(bundle / "pipeline.py"),
    "config_sha256": fresh_sha(bundle / "config.json"),
    "reference_sha256": {
        name: fresh_sha(verification_dir / name)
        for name in [
            "input.png",
            "reference_mask.png",
            "reference_overlay.png",
            "reference_report.json",
        ]
    },
}
job_path = verification_dir / "job.json"
job_path.write_text(json.dumps(job, indent=2), encoding="utf-8")

worker_source = r'''
from pathlib import Path
import hashlib
import json
import math
import sys

import cv2
import numpy as np
from ultralytics import YOLO

work = Path(sys.argv[1]).resolve()
job = json.loads((work / "job.json").read_text(encoding="utf-8"))
bundle = Path(job["bundle"])
root = Path(job["project_root"])

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha(bundle / "pipeline.py") == job["pipeline_sha256"]
assert sha(bundle / "config.json") == job["config_sha256"]

for name, expected in job["reference_sha256"].items():
    assert sha(work / name) == expected, name

sys.path.insert(0, str(bundle))
from pipeline import run_integrated_inference

config = json.loads(
    (bundle / "config.json").read_text(encoding="utf-8")
)
assert config["schema_version"] == "1.2"
assert config["device"] == "cpu"

models = {}
for name in ("m4", "m5"):
    path = (root / config[name]["checkpoint"]).resolve()
    assert path.is_relative_to(root)
    assert path.is_file(), path
    assert sha(path) == config[name]["sha256"], name
    models[name] = YOLO(str(path))
    models[name].to("cpu")

def read_image(name, flags):
    data = np.frombuffer((work / name).read_bytes(), dtype=np.uint8)
    result = cv2.imdecode(data, flags)
    assert result is not None, name
    return result

image = read_image("input.png", cv2.IMREAD_COLOR)
unchanged_input = image.copy()

report, overlay, mask = run_integrated_inference(
    image_bgr=image,
    detector=models["m4"],
    segmenter=models["m5"],
    detector_settings=config["m4"]["settings"],
    segmenter_settings=config["m5"]["settings"],
    device="cpu",
)

reference = json.loads(
    (work / "reference_report.json").read_text(encoding="utf-8")
)
differences = []

def compare(old, new, path="report"):
    if isinstance(old, dict):
        if not isinstance(new, dict) or set(old) != set(new):
            differences.append(path + ": dictionary keys differ")
            return
        for key in old:
            compare(old[key], new[key], path + "." + key)
    elif isinstance(old, list):
        if not isinstance(new, list) or len(old) != len(new):
            differences.append(path + ": list length differs")
            return
        for index, (a, b) in enumerate(zip(old, new)):
            compare(a, b, f"{path}[{index}]")
    elif isinstance(old, (int, float)) and not isinstance(old, bool):
        if not isinstance(new, (int, float)) or isinstance(new, bool):
            differences.append(path + ": numeric type differs")
            return
        tolerance = (
            0.0001 if ".bbox_xyxy[" in path
            else 0.000001 if path.endswith(".confidence")
            else 0.0
        )
        if not math.isclose(old, new, rel_tol=0.0, abs_tol=tolerance):
            differences.append(f"{path}: {old!r} != {new!r}")
    elif old != new:
        differences.append(f"{path}: {old!r} != {new!r}")

compare(
    {key: value for key, value in reference.items() if key != "timing"},
    {key: value for key, value in report.items() if key != "timing"},
)

timing = report["timing"]
stages = timing["stages"]
timing_values = [*stages.values(), timing["total_ms"]]

timing_valid = (
    set(stages) == set(reference["timing"]["stages"])
    and timing["device"] == "cpu"
    and timing["units"] == "milliseconds"
    and all(math.isfinite(value) and value >= 0 for value in timing_values)
    and math.isclose(
        sum(stages.values()),
        timing["total_ms"],
        rel_tol=1e-9,
        abs_tol=1e-6,
    )
)

reference_mask = read_image("reference_mask.png", cv2.IMREAD_GRAYSCALE)
reference_overlay = read_image("reference_overlay.png", cv2.IMREAD_COLOR)

checks = {
    "schema_version_1_2": report["schema_version"] == "1.2",
    "prediction_report_within_declared_tolerances": not differences,
    "mask_exact_match": np.array_equal(
        mask.astype(np.uint8) * 255, reference_mask
    ),
    "overlay_exact_match": np.array_equal(overlay, reference_overlay),
    "timing_valid": timing_valid,
    "input_preserved": np.array_equal(image, unchanged_input),
}

(work / "prediction_report.json").write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

result = {
    "purpose": "fresh_process_v3_verification",
    "checks": {key: bool(value) for key, value in checks.items()},
    "differences": differences,
    "pipeline_sha256": job["pipeline_sha256"],
    "config_sha256": job["config_sha256"],
    "checkpoint_sha256": {
        name: config[name]["sha256"] for name in ("m4", "m5")
    },
    "reference_sha256": job["reference_sha256"],
    "python_executable": sys.executable,
    "python_version": sys.version,
    "bbox_absolute_tolerance_pixels": 0.0001,
    "confidence_absolute_tolerance": 0.000001,
    "timing_values_compared_for_equality": False,
    "clean_install_verified": False,
}

(work / "verification.json").write_text(
    json.dumps(result, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}", flush=True)

if differences:
    print(json.dumps(differences[:20], indent=2), flush=True)

if not all(checks.values()):
    raise AssertionError("Fresh-process verification failed.")

print("PASS: Updated pipeline reproduced in a fresh Python process.")
'''

worker_path = verification_dir / "verify_worker.py"
compile(worker_source, str(worker_path), "exec")
worker_path.write_text(worker_source, encoding="utf-8")

log_path = verification_dir / "process.log"

print("Starting fresh Python process...", flush=True)

with log_path.open("w", encoding="utf-8") as log:
    completed = subprocess.run(
        [sys.executable, str(worker_path), str(verification_dir)],
        cwd=str(verification_dir),
        stdout=log,
        stderr=subprocess.STDOUT,
        timeout=180,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )

print(log_path.read_text(encoding="utf-8", errors="replace"))

assert completed.returncode == 0, (
    f"Verification failed. Inspect: {log_path}"
)

result = json.loads(
    (verification_dir / "verification.json").read_text(encoding="utf-8")
)
assert all(result["checks"].values())

print("Saved:", verification_dir / "verification.json")
print("Log:", log_path)
print("Clean-environment installation remains unverified.")
print("HTTP service was not started.")

## 59. Package and Verify the Updated Service

Package the active schema 1.2 service, dependency records, README,
model card, and fresh-process verification evidence.

Verify service fingerprints, checkpoint fingerprints, archive contents,
and every archived file hash. Models, datasets, ONNX exports, and the
full evaluation evidence remain external.

Packaging does not establish clean-install portability or production
acceptance.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile

run_root = Path(RUN_DIR).resolve()
bundle = Path(ACTIVE_SERVICE_BUNDLE).resolve()
fresh_dir = Path(verification_dir).resolve()
project_root = Path(PROJECT_ROOT).resolve()

assert bundle.is_relative_to(run_root)
assert fresh_dir.is_relative_to(run_root)

def package_sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

config = json.loads(
    (bundle / "config.json").read_text(encoding="utf-8")
)
verification = json.loads(
    (fresh_dir / "verification.json").read_text(encoding="utf-8")
)
documentation = json.loads(
    (bundle / "documentation_manifest.json").read_text(encoding="utf-8")
)

assert config["schema_version"] == "1.2"
assert verification["purpose"] == "fresh_process_v3_verification"
assert verification["checks"]
assert all(value is True for value in verification["checks"].values())

assert package_sha(bundle / "pipeline.py") == verification["pipeline_sha256"]
assert package_sha(bundle / "config.json") == verification["config_sha256"]

for name, expected in documentation["protected_artifact_sha256"].items():
    assert package_sha(bundle / name) == expected, name

for name, expected in documentation["documentation_sha256"].items():
    assert package_sha(bundle / name) == expected, name

for name in ("pipeline.py", "app.py"):
    source = (bundle / name).read_text(encoding="utf-8")
    compile(source, str(bundle / name), "exec")
    print("PASS: Python syntax:", name)

for model_name in ("m4", "m5"):
    checkpoint = (
        project_root / config[model_name]["checkpoint"]
    ).resolve()

    assert checkpoint.is_relative_to(project_root)
    assert checkpoint.is_file(), checkpoint
    assert package_sha(checkpoint) == config[model_name]["sha256"]
    print("PASS: Checkpoint verified:", model_name.upper())

service_names = [
    "pipeline.py",
    "app.py",
    "config.json",
    "launch.local.json",
    "requirements-service.txt",
    "requirements-verification.txt",
    "environment_inventory.json",
    "README.md",
    "MODEL_CARD.md",
    "model_card_provenance.json",
    "documentation_manifest.json",
]

files_to_package = {
    name: bundle / name
    for name in service_names
}

files_to_package.update({
    "verification/fresh_process_verification.json":
        fresh_dir / "verification.json",
    "verification/fresh_process_prediction_report.json":
        fresh_dir / "prediction_report.json",
    "verification/fresh_process_worker.py":
        fresh_dir / "verify_worker.py",
})

for path in files_to_package.values():
    assert path.is_file(), path

package_dir = run_root / (
    "service_package_v3_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
package_dir.mkdir(parents=True, exist_ok=False)

archive_path = package_dir / "M8_service_bundle_v3.zip"

file_records = {
    archive_name: {
        "sha256": package_sha(source_path),
        "bytes": source_path.stat().st_size,
    }
    for archive_name, source_path in files_to_package.items()
}

bundle_manifest = {
    "purpose": "verified_service_delivery_package",
    "report_schema_version": "1.2",
    "backend": "PyTorch",
    "device": "cpu",
    "source_bundle_relative_to_run": bundle.relative_to(run_root).as_posix(),
    "fresh_verification_relative_to_run":
        fresh_dir.relative_to(run_root).as_posix(),
    "files": file_records,
    "external_checkpoints": {
        name: {
            "path_relative_to_project_root": config[name]["checkpoint"],
            "sha256": config[name]["sha256"],
        }
        for name in ("m4", "m5")
    },
    "external_artifacts": [
        "Model checkpoints",
        "Datasets",
        "ONNX exports",
        "Full evaluation and benchmark evidence",
        "Failure gallery",
        "Notebook",
    ],
    "fresh_process_checks_passed": True,
    "clean_install_verified": False,
    "production_approved": False,
    "relocation_instruction": "Set M8_PROJECT_ROOT explicitly.",
    "verification_note": (
        "Archived verification files record the original local run. "
        "Its complete input/reference files remain in the external "
        "fresh-process evidence directory."
    ),
}

manifest_bytes = (
    json.dumps(bundle_manifest, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

with zipfile.ZipFile(
    archive_path,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for archive_name, source_path in files_to_package.items():
        archive.write(source_path, arcname=archive_name)
    archive.writestr("bundle_manifest.json", manifest_bytes)

with zipfile.ZipFile(archive_path, "r") as archive:
    expected_names = set(files_to_package) | {"bundle_manifest.json"}

    assert len(archive.namelist()) == len(expected_names)
    assert set(archive.namelist()) == expected_names
    assert archive.testzip() is None

    for archive_name, record in file_records.items():
        content = archive.read(archive_name)
        assert len(content) == record["bytes"], archive_name
        assert (
            hashlib.sha256(content).hexdigest() == record["sha256"]
        ), archive_name

    assert archive.read("bundle_manifest.json") == manifest_bytes

manifest_path = package_dir / "bundle_manifest.json"
manifest_path.write_bytes(manifest_bytes)

archive_hash = package_sha(archive_path)
checksum_path = package_dir / "SHA256SUMS.txt"
checksum_path.write_text(
    f"{archive_hash}  {archive_path.name}\n",
    encoding="utf-8",
)

package_verification = {
    "archive": archive_path.name,
    "archive_sha256": archive_hash,
    "archived_files_including_manifest": len(files_to_package) + 1,
    "zip_integrity_passed": True,
    "all_file_hashes_passed": True,
    "checkpoint_hashes_passed": True,
    "clean_install_verified": False,
    "production_approved": False,
}

(package_dir / "package_verification.json").write_text(
    json.dumps(package_verification, indent=2) + "\n",
    encoding="utf-8",
)

FINAL_SERVICE_PACKAGE = archive_path

print("\nPASS: Updated archive contents and fingerprints verified.")
print("Packaged service/documentation files:", len(service_names))
print("Packaged verification files:", 3)
print("ZIP:", archive_path)
print("Manifest:", manifest_path)
print("SHA-256:", archive_hash)
print("Models and full evaluation evidence remain external.")
print("When relocating, set M8_PROJECT_ROOT explicitly.")
print("No server started. No production acceptance claimed.")

## 60. Delivery Checklist and Demonstration Plan

### Service Deliverable

Use the verified `M8_service_bundle_v3.zip` from:

`service_package_v3_20261107T113714_763360Z`

Archive SHA-256:

`3268c7219ff9bb57bbf7965a1cf02da00fd4a56e1f6d132b2b6eb543bcdd767b`

The archive contains service code, configuration, dependency records,
README, model card, manifests, and selected fresh-process verification
evidence. It supersedes the earlier service-only archive for delivery.

Checkpoints, datasets, ONNX exports, the notebook, failure gallery,
and full evaluation evidence remain external.

### Supporting Deliverables

Retain and submit or link, according to the course submission format:

- The saved M8 notebook.
- The verified service archive and its checksum.
- Both ONNX exports and their export manifest.
- Development evaluation and threshold-selection evidence.
- ONNX numerical, pipeline, and crop-mask parity reports.
- Updated HTTP and isolated-backend benchmark reports.
- The reviewed failure-gallery presentation copies.
- The model card and documented limitations.
- Instructions for obtaining and locating the external checkpoints.

Do not include unredacted gallery images in presentation materials.

### Live Demonstration

1. Extract the service archive into a separate demonstration directory.
2. Set `M8_PROJECT_ROOT` explicitly.
3. Start the service using the README instructions and the verified
   Python environment.
4. Open `/docs` and demonstrate `GET /health`.
5. Upload the recorded pothole example to `POST /predict`.
6. Show damage classes, bounding boxes, segmentation status,
   image-area proxy, stage timings, and returned images.
7. Demonstrate an invalid upload and its expected error response.
8. Explain one missed detection and one false positive from the gallery.
9. Stop the foreground server with Ctrl+C.

Restarting the service in the existing environment is not evidence of
successful installation in a clean environment.

### Recorded Fallback

Prepare a short screen recording of the demonstrated workflow.
Identify prerecorded results explicitly if used during presentation.

Fallback recording status: recorded and saved in the M8 directory.  
Extracted-archive live HTTP demonstration: completed successfully.    
The extracted service returned matching masks, annotated images,
detection counts, and severity fields; timing checks passed.
Evidence: delivery_rehearsal_20261107T113958_987260Z/verification.json.

Clean-environment installation remains unverified.
### Final Interpretation

The prototype passed its recorded functional and reproducibility checks.
PyTorch is the selected local demonstration backend.

Operational deployment remains NO-GO. Low D40 recall, unverified
integrated mask and severity-proxy accuracy, and missing prospective
quantitative acceptance criteria prevent a production-readiness claim.

No independent final-test evaluation or clean-environment installation
is claimed.

## 61. Rehearse the Extracted Service Archive

Extract the verified delivery archive into a new directory.
Verify every packaged file and start the extracted service in a separate
Python process with M8_PROJECT_ROOT explicitly configured.

Check HTTP health and model fingerprints, then submit the recorded
smoke-test input. Compare returned image artifacts with the saved
reference and check report schema and timing consistency.

This rehearses the delivered archive in the existing installed
environment. It does not verify a clean dependency installation.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import json
import math
import os
import socket
import subprocess
import sys
import time
import zipfile

import cv2
import numpy as np
import requests

run_root = Path(RUN_DIR).resolve()
archive_path = Path(FINAL_SERVICE_PACKAGE).resolve()
assert archive_path.is_file(), archive_path

# Verify the ZIP against the recorded packaging result.
package_record = json.loads(
    (archive_path.parent / "package_verification.json").read_text(
        encoding="utf-8"
    )
)
archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
assert archive_hash == package_record["archive_sha256"]

DEMO_DIR = run_root / (
    "delivery_rehearsal_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
extracted_bundle = DEMO_DIR / "service"
extracted_bundle.mkdir(parents=True, exist_ok=False)

with zipfile.ZipFile(archive_path, "r") as archive:
    assert archive.testzip() is None
    assert len(archive.namelist()) == len(set(archive.namelist()))

    manifest = json.loads(archive.read("bundle_manifest.json"))
    expected_names = set(manifest["files"]) | {"bundle_manifest.json"}
    assert set(archive.namelist()) == expected_names

    # Validate all paths and hashes before extraction.
    for name in archive.namelist():
        target = (extracted_bundle / name).resolve()
        assert target.is_relative_to(extracted_bundle.resolve()), name

        if name in manifest["files"]:
            content = archive.read(name)
            expected = manifest["files"][name]
            assert len(content) == expected["bytes"], name
            assert (
                hashlib.sha256(content).hexdigest()
                == expected["sha256"]
            ), name

    archive.extractall(extracted_bundle)

for name, expected in manifest["files"].items():
    actual = hashlib.sha256(
        (extracted_bundle / name).read_bytes()
    ).hexdigest()
    assert actual == expected["sha256"], name

print("PASS: Extracted archive files match their fingerprints.")

config = json.loads(
    (extracted_bundle / "config.json").read_text(encoding="utf-8")
)

# Locate the previously verified input and image references.
fresh_reference_dir = (
    run_root / manifest["fresh_verification_relative_to_run"]
).resolve()
assert fresh_reference_dir.is_relative_to(run_root)

fresh_verification = json.loads(
    (fresh_reference_dir / "verification.json").read_text(
        encoding="utf-8"
    )
)
assert all(fresh_verification["checks"].values())

for name, expected in fresh_verification["reference_sha256"].items():
    actual = hashlib.sha256(
        (fresh_reference_dir / name).read_bytes()
    ).hexdigest()
    assert actual == expected, name

assert (
    manifest["files"]["pipeline.py"]["sha256"]
    == fresh_verification["pipeline_sha256"]
)
assert (
    manifest["files"]["config.json"]["sha256"]
    == fresh_verification["config_sha256"]
)

DEMO_PORT = 8767
DEMO_URL = f"http://127.0.0.1:{DEMO_PORT}"

with socket.socket() as probe:
    assert probe.connect_ex(("127.0.0.1", DEMO_PORT)) != 0, (
        f"Port {DEMO_PORT} is already in use. "
        "Do not rerun this cell while its service is running."
    )

environment = os.environ.copy()
environment["M8_PROJECT_ROOT"] = str(Path(PROJECT_ROOT).resolve())
environment["PYTHONUNBUFFERED"] = "1"

demo_log_path = DEMO_DIR / "http_service.log"
log_stream = demo_log_path.open("w", encoding="utf-8")

try:
    demo_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "app:app",
            "--host",
            "127.0.0.1",
            "--port",
            str(DEMO_PORT),
            "--workers",
            "1",
        ],
        cwd=str(extracted_bundle),
        env=environment,
        stdout=log_stream,
        stderr=subprocess.STDOUT,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
finally:
    log_stream.close()

try:
    print("Starting extracted service...", flush=True)
    deadline = time.monotonic() + 90
    health = None

    while time.monotonic() < deadline:
        if demo_process.poll() is not None:
            raise RuntimeError("Extracted service exited during startup.")

        try:
            response = requests.get(
                DEMO_URL + "/health",
                timeout=2,
            )
            if response.status_code == 200:
                health = response.json()
                break
        except requests.RequestException:
            pass

        time.sleep(0.5)

    assert health is not None, "Service did not become ready."
    assert health["status"] == "ready"
    assert health["device"] == "cpu"
    assert health["m4_sha256"] == config["m4"]["sha256"]
    assert health["m5_sha256"] == config["m5"]["sha256"]
    print("PASS: Extracted service health and model fingerprints.")

    input_bytes = (fresh_reference_dir / "input.png").read_bytes()

    response = requests.post(
        DEMO_URL + "/predict",
        files={"file": ("input.png", input_bytes, "image/png")},
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    report = payload["report"]

    def decode_png_bytes(data, flags):
        result = cv2.imdecode(
            np.frombuffer(data, dtype=np.uint8),
            flags,
        )
        assert result is not None, "PNG decoding failed."
        return result

    mask_bytes = base64.b64decode(
        payload["mask_png_base64"], validate=True
    )
    overlay_bytes = base64.b64decode(
        payload["annotated_png_base64"], validate=True
    )

    mask = decode_png_bytes(mask_bytes, cv2.IMREAD_GRAYSCALE)
    overlay = decode_png_bytes(overlay_bytes, cv2.IMREAD_COLOR)

    reference_mask = decode_png_bytes(
        (fresh_reference_dir / "reference_mask.png").read_bytes(),
        cv2.IMREAD_GRAYSCALE,
    )
    reference_overlay = decode_png_bytes(
        (fresh_reference_dir / "reference_overlay.png").read_bytes(),
        cv2.IMREAD_COLOR,
    )

    reference_report = json.loads(
        (fresh_reference_dir / "reference_report.json").read_text(
            encoding="utf-8"
        )
    )

    timing = report["timing"]
    timing_values = [
        *timing["stages"].values(),
        timing["total_ms"],
    ]

    checks = {
        "schema_version_1_2": report["schema_version"] == "1.2",
        "model_fingerprints_match": payload["model_fingerprints"] == {
            "m4": config["m4"]["sha256"],
            "m5": config["m5"]["sha256"],
        },
        "mask_exact_match": np.array_equal(mask, reference_mask),
        "overlay_exact_match": np.array_equal(
            overlay, reference_overlay
        ),
        "detection_count_matches": (
            len(report["detections"])
            == len(reference_report["detections"])
        ),
        "severity_fields_match": [
            (item["severity"], item["severity_status"])
            for item in report["detections"]
        ] == [
            (item["severity"], item["severity_status"])
            for item in reference_report["detections"]
        ],
        "timing_valid": (
            timing["device"] == "cpu"
            and timing["units"] == "milliseconds"
            and set(timing["stages"])
            == set(reference_report["timing"]["stages"])
            and all(
                math.isfinite(value) and value >= 0
                for value in timing_values
            )
            and math.isclose(
                sum(timing["stages"].values()),
                timing["total_ms"],
                rel_tol=1e-9,
                abs_tol=1e-6,
            )
        ),
    }

    for name, passed in checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {name}")

    (DEMO_DIR / "prediction_report.json").write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    (DEMO_DIR / "mask.png").write_bytes(mask_bytes)
    (DEMO_DIR / "annotated.png").write_bytes(overlay_bytes)

    verification = {
        "purpose": "extracted_archive_HTTP_rehearsal",
        "archive": str(archive_path),
        "archive_sha256": archive_hash,
        "extracted_bundle": str(extracted_bundle),
        "checks": {
            name: bool(passed)
            for name, passed in checks.items()
        },
        "full_report_numerical_comparison_performed": False,
        "clean_install_verified": False,
        "service_url": DEMO_URL,
        "service_pid": demo_process.pid,
    }

    (DEMO_DIR / "verification.json").write_text(
        json.dumps(verification, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

    assert all(checks.values()), "Extracted service verification failed."

except BaseException:
    if demo_process.poll() is None:
        demo_process.terminate()
        try:
            demo_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            demo_process.kill()
            demo_process.wait()

    print(demo_log_path.read_text(
        encoding="utf-8", errors="replace"
    )[-5000:])
    raise

print("\nPASS: Extracted delivery archive HTTP rehearsal completed.")
print("Evidence:", DEMO_DIR / "verification.json")
print("Interactive API:", DEMO_URL + "/docs")
print("Annotated image:", DEMO_DIR / "annotated.png")
print("Log:", demo_log_path)
print("Service remains running locally for the live demonstration.")
print("Clean-environment installation remains unverified.")

## 62. Create the Current-Run Submission Evidence Index

Build the submission index from the artifact paths retained by the
current notebook session, without hard-coded timestamped directories.

Verify that all experiment artifacts belong to RUN_DIR. Check that
the packaged service, fresh-process evidence, and extracted-archive
rehearsal refer to the same pipeline and configuration.

Record file fingerprints. The notebook remains editable and is indexed
without a fixed hash. Video presence does not verify playback or content.

In [ ]:
from pathlib import Path
import hashlib
import json
import zipfile
from urllib.parse import quote

required_variables = [
    "M8_DIR",
    "RUN_DIR",
    "SWEEP_DIR",
    "ONNX_DIR",
    "ACTIVE_SERVICE_BUNDLE",
    "V3_BENCHMARK_DIR",
    "ISOLATED_BENCHMARK_DIR",
    "FAILURE_GALLERY_DIR",
    "presentation_dir",
    "FINAL_SERVICE_PACKAGE",
    "DEMO_DIR",
]

missing = [
    name for name in required_variables
    if name not in globals()
]
if missing:
    raise RuntimeError(
        "Missing current-run artifact variables: "
        + ", ".join(missing)
        + ". Run the relevant preceding sections or explicitly resume "
        "the intended experiment. Do not guess artifact directories."
    )

index_m8_root = Path(M8_DIR).resolve()
index_run_root = Path(RUN_DIR).resolve()

assert index_run_root.is_relative_to(index_m8_root / "results")

artifact_paths = {
    "sweep": Path(SWEEP_DIR).resolve(),
    "onnx": Path(ONNX_DIR).resolve(),
    "service": Path(ACTIVE_SERVICE_BUNDLE).resolve(),
    "http_benchmark": Path(V3_BENCHMARK_DIR).resolve(),
    "backend_benchmark": Path(ISOLATED_BENCHMARK_DIR).resolve(),
    "gallery": Path(FAILURE_GALLERY_DIR).resolve(),
    "presentation": Path(presentation_dir).resolve(),
    "package": Path(FINAL_SERVICE_PACKAGE).resolve(),
    "rehearsal": Path(DEMO_DIR).resolve(),
}

for name, path in artifact_paths.items():
    assert path.exists(), f"Missing {name}: {path}"
    assert path.is_relative_to(index_run_root), (
        f"{name} belongs to a different run: {path}"
    )

assert artifact_paths["presentation"].is_relative_to(
    artifact_paths["gallery"]
)


def index_sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def index_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


# Resolve the fresh-process evidence from the selected package manifest.
selected_zip = artifact_paths["package"]
package_check = index_json(
    selected_zip.parent / "package_verification.json"
)
selected_zip_hash = index_sha(selected_zip)

assert selected_zip_hash == package_check["archive_sha256"]

with zipfile.ZipFile(selected_zip, "r") as archive:
    assert archive.testzip() is None
    packaged_manifest = json.loads(
        archive.read("bundle_manifest.json")
    )

    expected_members = set(packaged_manifest["files"]) | {
        "bundle_manifest.json"
    }
    assert len(archive.namelist()) == len(expected_members)
    assert set(archive.namelist()) == expected_members

    for name, record in packaged_manifest["files"].items():
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == record["sha256"]

fresh_evidence_dir = (
    index_run_root
    / packaged_manifest["fresh_verification_relative_to_run"]
).resolve()

assert fresh_evidence_dir.is_relative_to(index_run_root)

fresh_check = index_json(fresh_evidence_dir / "verification.json")
rehearsal_check = index_json(
    artifact_paths["rehearsal"] / "verification.json"
)

assert fresh_check["checks"]
assert all(value is True for value in fresh_check["checks"].values())
assert rehearsal_check["checks"]
assert all(
    value is True
    for value in rehearsal_check["checks"].values()
)
assert rehearsal_check["archive_sha256"] == selected_zip_hash

for filename, verification_key in [
    ("pipeline.py", "pipeline_sha256"),
    ("config.json", "config_sha256"),
]:
    current_hash = index_sha(
        artifact_paths["service"] / filename
    )
    assert (
        current_hash
        == packaged_manifest["files"][filename]["sha256"]
        == fresh_check[verification_key]
    ), f"Version mismatch: {filename}"

# Choose the recording without depending on its timestamped filename.
# If more than one video exists, explicitly set DEMO_VIDEO_PATH first.
if "DEMO_VIDEO_PATH" in globals():
    selected_video = Path(DEMO_VIDEO_PATH).resolve()
else:
    video_candidates = sorted(
        path.resolve()
        for path in index_m8_root.iterdir()
        if path.is_file()
        and path.suffix.lower() in {".mp4", ".mov", ".mkv", ".webm"}
    )

    if len(video_candidates) != 1:
        raise RuntimeError(
            f"Found {len(video_candidates)} videos in M8. "
            "Set DEMO_VIDEO_PATH to the intended recording, then rerun."
        )

    selected_video = video_candidates[0]

assert selected_video.is_file()
assert selected_video.is_relative_to(index_m8_root)
assert selected_video.stat().st_size > 0

index_entries = [
    (
        "Notebook",
        index_m8_root / "M8.ipynb",
        "Main notebook; save after running this cell.",
    ),
    (
        "Demo recording",
        selected_video,
        "Recorded fallback. This cell does not assess video content.",
    ),
    (
        "Updated service ZIP",
        selected_zip,
        "Selected schema 1.2 service delivery archive.",
    ),
    (
        "Package verification",
        selected_zip.parent / "package_verification.json",
        "Archive integrity and fingerprint checks.",
    ),
    (
        "Model card",
        index_run_root / "model_card_v3.md",
        "Intended use, recorded results, and limitations.",
    ),
    (
        "Operating point",
        index_run_root / "provisional_operating_point.json",
        "Development-selected detector settings.",
    ),
    (
        "Threshold comparison",
        artifact_paths["sweep"] / "operating_point_comparison.csv",
        "Detection metrics across development operating points.",
    ),
    (
        "Updated pipeline edge cases",
        index_run_root / "pipeline_v3_edge_case_tests.json",
        "Pipeline logic, severity-proxy, and timing checks.",
    ),
    (
        "Updated HTTP validation",
        artifact_paths["service"] / "http_input_validation.json",
        "Updated service input validation evidence.",
    ),
    (
        "Updated HTTP benchmark",
        artifact_paths["http_benchmark"] / "summary.json",
        "PyTorch HTTP latency, throughput, and memory.",
    ),
    (
        "Isolated backend benchmark",
        artifact_paths["backend_benchmark"] / "summary.json",
        "Fresh-process direct-pipeline measurements.",
    ),
    (
        "Updated fresh-process verification",
        fresh_evidence_dir / "verification.json",
        "Reproduction in the existing installed environment.",
    ),
    (
        "Extracted archive rehearsal",
        artifact_paths["rehearsal"] / "verification.json",
        "HTTP rehearsal of the selected delivery archive.",
    ),
    (
        "Failure gallery presentation",
        artifact_paths["presentation"] / "README.md",
        "Presentation copies; visual review applies to the inspected version.",
    ),
    (
        "M4 ONNX",
        artifact_paths["onnx"] / "m4" / "m4.onnx",
        "Exported detector.",
    ),
    (
        "M5 ONNX",
        artifact_paths["onnx"] / "m5" / "m5.onnx",
        "Exported segmenter.",
    ),
    (
        "ONNX export manifest",
        artifact_paths["onnx"] / "export_manifest.json",
        "Export settings and artifact records.",
    ),
    (
        "Raw numerical parity",
        artifact_paths["onnx"] / "raw_numerical_parity.json",
        "Raw-output agreement on the recorded sample.",
    ),
    (
        "Pipeline parity",
        artifact_paths["onnx"] / "pipeline_parity.json",
        "Agreement within its recorded version and sample scope.",
    ),
    (
        "Crop-mask parity",
        artifact_paths["onnx"] / "crop_mask_parity.json",
        "Union-mask agreement on the recorded crops.",
    ),
]

# Validate every entry before replacing the index.
for title, path, description in index_entries:
    assert path.is_file(), f"Missing {title}: {path}"
    assert path.stat().st_size > 0, f"Empty file: {path}"
    assert path.resolve().is_relative_to(index_m8_root), path

index_records = []
index_markdown = [
    "# M8 Submission Evidence Index",
    "",
    "Contributor: Ghassan Saad AlGhaidani — 225495.",
    "",
    f"Experiment: `{index_run_root.name}`",
    "",
    "Use the service archive selected below.",
    "Keep relative paths intact when transferring the M8 directory.",
    "Detailed experiment evidence is retained beside the listed summaries.",
    "",
]

for title, path, description in index_entries:
    relative_path = path.relative_to(index_m8_root).as_posix()
    is_notebook = path.suffix.lower() == ".ipynb"

    index_records.append({
        "title": title,
        "path_relative_to_M8": relative_path,
        "description": description,
        "sha256": None if is_notebook else index_sha(path),
        "bytes": None if is_notebook else path.stat().st_size,
        "mutable_notebook": is_notebook,
    })

    index_markdown.extend([
        f"## {title}",
        "",
        f"[Open file]({quote(relative_path, safe='/')})",
        "",
        description,
        "",
    ])

    print("FOUND:", title)

index_markdown.extend([
    "## External Dependencies and Scope",
    "",
    "The service ZIP excludes checkpoints and datasets.",
    "Follow its README and set M8_PROJECT_ROOT explicitly.",
    "",
    "Functional service checks do not establish prediction accuracy.",
    "Clean-environment installation and full fresh-kernel notebook replay",
    "remain unverified. Integrated mask and severity-proxy accuracy have",
    "not been established on the development cohorts.",
    "",
    "No independent final-test performance or production approval is claimed.",
    "Quantitative acceptance criteria were not predetermined for the",
    "recorded benchmarks.",
    "",
    "The service does not automatically anonymize uploaded images.",
    "This index validates file integrity, not video content or image privacy.",
    "",
])

index_output = index_m8_root / "SUBMISSION_INDEX.md"
index_manifest_output = index_m8_root / "submission_files.json"

index_output.write_text(
    "\n".join(index_markdown) + "\n",
    encoding="utf-8",
)

index_manifest_output.write_text(
    json.dumps(
        {
            "purpose": "current_run_submission_evidence_index",
            "paths_relative_to": "M8",
            "run_relative_to_M8":
                index_run_root.relative_to(index_m8_root).as_posix(),
            "files": index_records,
            "package_and_rehearsal_consistent": True,
            "pipeline_and_config_versions_consistent": True,
            "notebook_hash_omitted_because_still_editable": True,
            "video_playback_verified_by_this_cell": False,
            "production_approved": False,
        },
        indent=2,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

print("\nSaved:", index_output)
print("Saved:", index_manifest_output)
print("Indexed files:", len(index_records))
print("PASS: Current-run artifact paths and fingerprints verified.")
print("PASS: Package, pipeline, and rehearsal versions are consistent.")
print("Save the notebook with Ctrl+S.")